In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
MODELS  = f"{BASE}/models"
RESULTS = f"{BASE}/results"
FIGS    = f"{BASE}/figures"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, numpy as np, pandas as pd

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
RAW  = f"{BASE}/data_raw"
PROC = f"{BASE}/data_processed"
ART  = f"{BASE}/artifacts"
for p in [RAW, PROC, ART]:
    os.makedirs(p, exist_ok=True)

cic_dir = f"{RAW}/CICIDS2017"
cic_csvs = sorted(glob.glob(cic_dir + "/**/*.csv", recursive=True))
print("Found CICIDS CSVs:", len(cic_csvs))

OUT_DIR = f"{PROC}/cicids_clean_shards_full"
os.makedirs(OUT_DIR, exist_ok=True)

def map_5class(label):
    if label == "BENIGN":
        return 0
    elif label in ["FTP-Patator", "SSH-Patator"]:
        return 1
    elif label in ["DoS Hulk", "DoS GoldenEye", "DoS slowloris", "DoS Slowhttptest", "DDoS"]:
        return 2
    elif label == "PortScan":
        return 3
    else:
        return 4

drop_cols = {"Flow ID", "Src IP", "Dst IP", "Timestamp", "__source_file__"}

CHUNKSIZE = 200_000
written = 0

for file_i, path in enumerate(cic_csvs):
    print("\nProcessing:", os.path.basename(path))
    try:
        it = pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False, encoding_errors="ignore")
    except TypeError:
        it = pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=False, encoding="latin1")

    for chunk_j, df in enumerate(it):
        df.columns = [c.strip() for c in df.columns]
        if "Label" not in df.columns:
            continue

        # drop leakage cols if present
        cols_to_drop = [c for c in df.columns if c in drop_cols]
        if cols_to_drop:
            df.drop(columns=cols_to_drop, inplace=True, errors="ignore")

        # replace inf and drop NaNs
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.dropna(inplace=True)

        # labels
        df["label_binary"] = (df["Label"] != "BENIGN").astype(np.int8)
        df["label_5class"] = df["Label"].map(map_5class).astype(np.int8)

        out_path = f"{OUT_DIR}/cic_clean_{file_i:02d}_{chunk_j:03d}.csv"
        df.to_csv(out_path, index=False)
        written += 1
        if written % 10 == 0:
            print("  shards written:", written)

print("\nTotal shards written:", written)
print("Shard dir:", OUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found CICIDS CSVs: 8

Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv

Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv

Processing: Monday-WorkingHours.pcap_ISCX.csv

Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  shards written: 10

Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv

Processing: Tuesday-WorkingHours.pcap_ISCX.csv

Processing: Wednesday-workingHours.pcap_ISCX.csv

Total shards written: 18
Shard dir: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/cicids_clean_shards_full


In [ ]:
import pandas as pd, glob, json

shards = sorted(glob.glob(OUT_DIR + "/*.csv"))
assert shards, "No shards created."
tmp = pd.read_csv(shards[0], nrows=5)
tmp.columns = [c.strip() for c in tmp.columns]

exclude = {"Label", "label_binary", "label_5class"}
feature_list = [c for c in tmp.columns if c not in exclude]

with open(f"{ART}/feature_list.json", "w") as f:
    json.dump(feature_list, f, indent=2)

print("Features:", len(feature_list))
print("Saved:", f"{ART}/feature_list.json")


Features: 78
Saved: /content/drive/MyDrive/A3IDPS_COLAB/artifacts/feature_list.json


In [ ]:
import glob, random
shards = sorted(glob.glob(OUT_DIR + "/*.csv"))
random.seed(42)
random.shuffle(shards)

n = len(shards)
train_files = shards[: int(0.7*n)]
val_files   = shards[int(0.7*n) : int(0.85*n)]
test_files  = shards[int(0.85*n) :]

print("Shards:", n)
print("Train/Val/Test:", len(train_files), len(val_files), len(test_files))


Shards: 18
Train/Val/Test: 12 3 3


In [ ]:
import tensorflow as tf, json

with open(f"{ART}/feature_list.json") as f:
    feature_list = json.load(f)

LABEL_COL = "label_5class"
CSV_COLUMNS = feature_list + [LABEL_COL]

BATCH = 1024   # ↓ was 4096 (too heavy)
SHUFFLE_BUF = 20_000  # ↓ was 200_000 (too RAM heavy)

# Force schema: float32 everywhere
column_defaults = (
    [tf.constant(0.0, dtype=tf.float32)] * len(feature_list)
    + [tf.constant(0.0, dtype=tf.float32)]
)

def make_ds(files, shuffle=True):
    ds = tf.data.experimental.make_csv_dataset(
        files,
        batch_size=BATCH,
        select_columns=CSV_COLUMNS,
        label_name=LABEL_COL,
        column_defaults=column_defaults,
        num_epochs=1,
        header=True,
        na_value="",
        shuffle=shuffle,
        shuffle_buffer_size=SHUFFLE_BUF if shuffle else 1,
        # Make pipeline more deterministic & stable
        sloppy=False
    )

    ds = ds.ignore_errors()

    def pack(features, label):
        x = tf.stack([features[c] for c in feature_list], axis=1)  # float32
        y = tf.cast(tf.round(label), tf.int32)
        return x, y

    # Reduce parallelism pressure
    ds = ds.map(pack, num_parallel_calls=1)

    # Prefetch a fixed small number (AUTOTUNE can grow queues)
    ds = ds.prefetch(2)
    return ds

train_ds = make_ds(train_files, shuffle=True)
val_ds   = make_ds(val_files, shuffle=False)
test_ds  = make_ds(test_files, shuffle=False)

print("Rebuilt datasets with BATCH=", BATCH, "SHUFFLE_BUF=", SHUFFLE_BUF)


Rebuilt datasets with BATCH= 1024 SHUFFLE_BUF= 20000


In [ ]:
xb, yb = next(iter(train_ds.take(1)))
print(xb.shape, yb.shape, xb.dtype, yb.dtype)
print("y min/max:", int(tf.reduce_min(yb)), int(tf.reduce_max(yb)))


(1024, 78) (1024,) <dtype: 'float32'> <dtype: 'int32'>
y min/max: 0 2


In [ ]:
norm = tf.keras.layers.Normalization(axis=-1)

# safest: partial adapt for speed
norm.adapt(train_ds.map(lambda x, y: x).take(300))

print("Normalization adapted ✓")


Normalization adapted ✓


In [ ]:
num_features = len(feature_list)
num_classes = 5

model = tf.keras.Sequential([
    tf.keras.Input(shape=(num_features,)),
    norm,
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5, min_lr=1e-5),
]

EPOCHS = 12
STEPS_PER_EPOCH = 300   # ↓ was 800
VAL_STEPS = 80

history = model.fit(
    train_ds.repeat(),
    validation_data=val_ds.repeat(),
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VAL_STEPS,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/12
300/300 ━━━━━━━━━━━━━━━━━━━━ 47s 136ms/step - accuracy: 0.9670 - loss: 0.1329 - val_accuracy: 0.9917 - val_loss: 0.0637 - learning_rate: 0.0010
Epoch 2/12
300/300 ━━━━━━━━━━━━━━━━━━━━ 40s 133ms/step - accuracy: 0.9874 - loss: 0.0521 - val_accuracy: 0.9883 - val_loss: 0.0920 - learning_rate: 0.0010
Epoch 3/12
300/300 ━━━━━━━━━━━━━━━━━━━━ 40s 132ms/step - accuracy: 0.9981 - loss: 0.0066 - val_accuracy: 0.9955 - val_loss: 0.0868 - learning_rate: 5.0000e-04


In [ ]:
print("✅ Fit returned. Epochs actually run:", len(history.history["loss"]))
print("Last epoch metrics:")
for k in history.history:
    print(f"  {k}: {history.history[k][-1]}")


✅ Fit returned. Epochs actually run: 3
Last epoch metrics:
  accuracy: 0.9993294477462769
  loss: 0.002682481426745653
  val_accuracy: 0.9954589605331421
  val_loss: 0.08679847419261932
  learning_rate: 0.0005000000237487257


In [ ]:
import json, os

# Save model
IDS_PATH = f"{MODELS}/ids_base_5class.keras"
model.save(IDS_PATH)

# Save history
with open(f"{RESULTS}/ids_base_5class_history.json", "w") as f:
    json.dump(history.history, f, indent=2)

# Save clean metrics
clean_metrics = {
    "epochs_run": len(history.history["loss"]),
    "final_train_accuracy": history.history["accuracy"][-1],
    "final_val_accuracy": history.history["val_accuracy"][-1],
    "final_val_loss": history.history["val_loss"][-1],
}
with open(f"{RESULTS}/ids_base_5class_clean_metrics.json", "w") as f:
    json.dump(clean_metrics, f, indent=2)

print("✅ Base IDS + metrics saved")
print("Model:", IDS_PATH)


✅ Base IDS + metrics saved
Model: /content/drive/MyDrive/A3IDPS_COLAB/models/ids_base_5class.keras


In [ ]:
import tensorflow as tf

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

@tf.function
def fgsm_batch(x, y, eps):
    with tf.GradientTape() as tape:
        tape.watch(x)
        preds = model(x, training=False)
        loss = loss_fn(y, preds)
    grad = tape.gradient(loss, x)
    x_adv = x + eps * tf.sign(grad)
    return x_adv


In [ ]:
import pandas as pd
import numpy as np
import os

FGSM_DIR = f"{PROC}/fgsm_test_shards"
os.makedirs(FGSM_DIR, exist_ok=True)

EPS = 0.01      # start with 0.01; we will also test 0.05 later
BATCHES_TO_SAVE_PER_FILE = 50  # controls shard size

feature_names = feature_list  # 78 features

file_id = 0
rows_written = 0

buffer_x = []
buffer_y = []

for step, (xb, yb) in enumerate(test_ds):
    x_adv = fgsm_batch(xb, yb, tf.constant(EPS, tf.float32))

    buffer_x.append(x_adv.numpy())
    buffer_y.append(yb.numpy())

    if (step + 1) % BATCHES_TO_SAVE_PER_FILE == 0:
        X_out = np.vstack(buffer_x)
        y_out = np.concatenate(buffer_y)

        df_out = pd.DataFrame(X_out, columns=feature_names)
        df_out["label_5class"] = y_out.astype(np.int32)

        out_path = f"{FGSM_DIR}/fgsm_eps{EPS}_part{file_id:04d}.csv"
        df_out.to_csv(out_path, index=False)

        rows_written += len(df_out)
        file_id += 1
        buffer_x, buffer_y = [], []

        if file_id % 5 == 0:
            print(f"Saved {file_id} shards, total rows: {rows_written}")

# save remaining
if buffer_x:
    X_out = np.vstack(buffer_x)
    y_out = np.concatenate(buffer_y)
    df_out = pd.DataFrame(X_out, columns=feature_names)
    df_out["label_5class"] = y_out.astype(np.int32)
    out_path = f"{FGSM_DIR}/fgsm_eps{EPS}_part{file_id:04d}.csv"
    df_out.to_csv(out_path, index=False)
    rows_written += len(df_out)

print("✅ FGSM generation complete.")
print("Saved in:", FGSM_DIR)
print("Total FGSM rows:", rows_written)


Saved 5 shards, total rows: 256000
✅ FGSM generation complete.
Saved in: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/fgsm_test_shards
Total FGSM rows: 486178


In [ ]:
import glob, json
import tensorflow as tf

fgsm_files = sorted(glob.glob(FGSM_DIR + f"/fgsm_eps{EPS}_part*.csv"))
print("FGSM shards:", len(fgsm_files))

def make_adv_ds(files, batch_size=512):
    ds = tf.data.experimental.make_csv_dataset(
        files,
        batch_size=batch_size,
        select_columns=feature_list + ["label_5class"],
        label_name="label_5class",
        column_defaults=[tf.constant(0.0, tf.float32)] * len(feature_list) + [tf.constant(0.0, tf.float32)],
        num_epochs=1,
        header=True,
        shuffle=False,
        sloppy=False,
        num_parallel_reads=1
    ).ignore_errors()

    def pack(features, label):
        x = tf.stack([features[c] for c in feature_list], axis=1)
        y = tf.cast(tf.round(label), tf.int32)
        return x, y

    ds = ds.map(pack, num_parallel_calls=1).prefetch(1)
    return ds

fgsm_ds = make_adv_ds(fgsm_files, batch_size=512)


FGSM shards: 10


In [ ]:
fgsm_loss, fgsm_acc = model.evaluate(fgsm_ds, verbose=1)
print(f"✅ FGSM Test Accuracy @ eps={EPS}:", fgsm_acc)


950/950 ━━━━━━━━━━━━━━━━━━━━ 34s 36ms/step - accuracy: 0.4823 - loss: 297624.1875
✅ FGSM Test Accuracy @ eps=0.01: 0.6347160935401917


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [ ]:
import os, json, glob

# Make sure paths exist (in case of restart)
BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
RESULTS = f"{BASE}/results"
PROC = f"{BASE}/data_processed"
os.makedirs(RESULTS, exist_ok=True)

EPS = 0.01
FGSM_DIR = f"{PROC}/fgsm_test_shards"

fgsm_files = sorted(glob.glob(FGSM_DIR + f"/fgsm_eps{EPS}_part*.csv"))
fgsm_rows = 0
for fpath in fgsm_files:
    # quick row count without loading full file
    with open(fpath, "rb") as f:
        fgsm_rows += sum(1 for _ in f) - 1  # minus header

fgsm_metrics = {
    "epsilon": EPS,
    "fgsm_accuracy": float(fgsm_acc),
    "fgsm_loss": float(fgsm_loss),
    "num_fgsm_shards": len(fgsm_files),
    "num_fgsm_rows": int(fgsm_rows),
    "model_path": f"{BASE}/models/ids_base_5class.keras",
    "fgsm_shard_dir": FGSM_DIR
}

out_path = f"{RESULTS}/fgsm_eps{EPS}_metrics.json"
with open(out_path, "w") as f:
    json.dump(fgsm_metrics, f, indent=2)

print("✅ Saved:", out_path)
print("FGSM shards:", len(fgsm_files), "rows:", fgsm_rows)


✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/fgsm_eps0.01_metrics.json
FGSM shards: 10 rows: 486178


In [ ]:
import os, numpy as np, pandas as pd, tensorflow as tf

EPS = 0.05
FGSM_DIR = f"{PROC}/fgsm_test_shards"
os.makedirs(FGSM_DIR, exist_ok=True)

BATCHES_TO_SAVE_PER_FILE = 50
feature_names = feature_list

file_id = 0
rows_written = 0
buffer_x, buffer_y = [], []

for step, (xb, yb) in enumerate(test_ds):
    x_adv = fgsm_batch(xb, yb, tf.constant(EPS, tf.float32))
    buffer_x.append(x_adv.numpy())
    buffer_y.append(yb.numpy())

    if (step + 1) % BATCHES_TO_SAVE_PER_FILE == 0:
        X_out = np.vstack(buffer_x)
        y_out = np.concatenate(buffer_y)

        df_out = pd.DataFrame(X_out, columns=feature_names)
        df_out["label_5class"] = y_out.astype(np.int32)

        out_path = f"{FGSM_DIR}/fgsm_eps{EPS}_part{file_id:04d}.csv"
        df_out.to_csv(out_path, index=False)

        rows_written += len(df_out)
        file_id += 1
        buffer_x, buffer_y = [], []

        if file_id % 5 == 0:
            print(f"Saved {file_id} shards, total rows: {rows_written}")

if buffer_x:
    X_out = np.vstack(buffer_x)
    y_out = np.concatenate(buffer_y)
    df_out = pd.DataFrame(X_out, columns=feature_names)
    df_out["label_5class"] = y_out.astype(np.int32)
    out_path = f"{FGSM_DIR}/fgsm_eps{EPS}_part{file_id:04d}.csv"
    df_out.to_csv(out_path, index=False)
    rows_written += len(df_out)

print("✅ FGSM eps=0.05 generation complete. Rows:", rows_written)


Saved 5 shards, total rows: 256000
✅ FGSM eps=0.05 generation complete. Rows: 486178


In [ ]:
import glob, tensorflow as tf

EPS = 0.05
fgsm_files = sorted(glob.glob(FGSM_DIR + f"/fgsm_eps{EPS}_part*.csv"))
fgsm_ds = make_adv_ds(fgsm_files, batch_size=512)

fgsm_loss_005, fgsm_acc_005 = model.evaluate(fgsm_ds, verbose=1)
print(f"✅ FGSM Test Accuracy @ eps={EPS}:", fgsm_acc_005)


950/950 ━━━━━━━━━━━━━━━━━━━━ 35s 36ms/step - accuracy: 0.4823 - loss: 1488110.2500
✅ FGSM Test Accuracy @ eps=0.05: 0.6347160935401917


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [ ]:
import json, os

fgsm_metrics_005 = {
    "epsilon": 0.05,
    "fgsm_accuracy": float(fgsm_acc_005),
    "fgsm_loss": float(fgsm_loss_005),
    "num_fgsm_shards": len(fgsm_files),
    "num_fgsm_rows": int(rows_written),
    "model_path": f"{BASE}/models/ids_base_5class.keras",
    "fgsm_shard_dir": FGSM_DIR
}

out_path = f"{RESULTS}/fgsm_eps0.05_metrics.json"
with open(out_path, "w") as f:
    json.dump(fgsm_metrics_005, f, indent=2)

print("✅ Saved:", out_path)


✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/fgsm_eps0.05_metrics.json


In [ ]:
import glob, os

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
PROC = f"{BASE}/data_processed"
FGSM_DIR = f"{PROC}/fgsm_test_shards"

files_001 = sorted(glob.glob(FGSM_DIR + "/fgsm_eps0.01_part*.csv"))
files_005 = sorted(glob.glob(FGSM_DIR + "/fgsm_eps0.05_part*.csv"))

print("eps=0.01 shards:", len(files_001))
print("eps=0.05 shards:", len(files_005))

print("Example eps=0.01:", os.path.basename(files_001[0]) if files_001 else None)
print("Example eps=0.05:", os.path.basename(files_005[0]) if files_005 else None)


eps=0.01 shards: 10
eps=0.05 shards: 10
Example eps=0.01: fgsm_eps0.01_part0000.csv
Example eps=0.05: fgsm_eps0.05_part0000.csv


In [ ]:
import pandas as pd
import numpy as np

assert len(files_001) > 0 and len(files_005) > 0, "Missing one epsilon file set."

df01 = pd.read_csv(files_001[0], nrows=200)
df05 = pd.read_csv(files_005[0], nrows=200)

# Compare feature columns only (exclude label)
feat_cols = [c for c in df01.columns if c != "label_5class"]

diff = np.abs(df05[feat_cols].values - df01[feat_cols].values)

print("Mean |diff|:", float(diff.mean()))
print("Max  |diff|:", float(diff.max()))


Mean |diff|: 0.039436087643450114
Max  |diff|: 0.09999999997671694


In [ ]:
import os, json, glob

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
RESULTS = f"{BASE}/results"
PROC = f"{BASE}/data_processed"
FGSM_DIR = f"{PROC}/fgsm_test_shards"
os.makedirs(RESULTS, exist_ok=True)

def count_rows(csv_path):
    # fast line count (header excluded)
    with open(csv_path, "rb") as f:
        return sum(1 for _ in f) - 1

def save_fgsm_metrics(eps, acc, loss):
    files = sorted(glob.glob(FGSM_DIR + f"/fgsm_eps{eps}_part*.csv"))
    rows = sum(count_rows(p) for p in files)
    out = {
        "epsilon": float(eps),
        "fgsm_accuracy": float(acc),
        "fgsm_loss": float(loss),
        "num_shards": len(files),
        "num_rows": int(rows),
        "shard_dir": FGSM_DIR,
        "model_path": f"{BASE}/models/ids_base_5class.keras",
    }
    out_path = f"{RESULTS}/fgsm_eps{eps}_metrics.json"
    with open(out_path, "w") as f:
        json.dump(out, f, indent=2)
    print("✅ Saved:", out_path, "| rows:", rows, "| shards:", len(files))

# Use your variables if they exist; otherwise set them manually:
# eps=0.01 metrics:
save_fgsm_metrics("0.01", fgsm_acc, fgsm_loss)

# eps=0.05 metrics: (use your variables fgsm_acc_005, fgsm_loss_005 if you stored them)
# If you didn't store them, re-evaluate eps=0.05 once and then call save.
try:
    save_fgsm_metrics("0.05", fgsm_acc_005, fgsm_loss_005)
except NameError:
    print("⚠️ fgsm_acc_005 / fgsm_loss_005 not found in memory. Re-evaluate eps=0.05 once, then re-run save_fgsm_metrics.")


✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/fgsm_eps0.01_metrics.json | rows: 486178 | shards: 10
✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/fgsm_eps0.05_metrics.json | rows: 486178 | shards: 10


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import glob

target_names = ["Benign", "BruteForce", "DoS/DDoS", "PortScan", "Web/Other"]
LABELS = [0,1,2,3,4]

def eval_confusion_on_fgsm(eps):
    files = sorted(glob.glob(FGSM_DIR + f"/fgsm_eps{eps}_part*.csv"))
    ds = make_adv_ds(files, batch_size=512)

    y_true, y_pred = [], []
    for xb, yb in ds:
        probs = model.predict(xb, verbose=0)
        pred = np.argmax(probs, axis=1)
        y_true.append(yb.numpy())
        y_pred.append(pred)

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    cm = confusion_matrix(y_true, y_pred, labels=LABELS)

    print(f"\n=== FGSM eps={eps} ===")
    print("Confusion Matrix:\n", cm)

    report = classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        target_names=target_names,
        digits=4,
        zero_division=0
    )
    print("\nClassification Report:\n", report)

    # Save artifacts (VERY IMPORTANT)
    np.save(f"{RESULTS}/fgsm_eps{eps}_confusion.npy", cm)
    with open(f"{RESULTS}/fgsm_eps{eps}_classification_report.txt", "w") as f:
        f.write(report)

    print("✅ Saved confusion matrix + report for eps =", eps)

# Run both
eval_confusion_on_fgsm("0.01")
eval_confusion_on_fgsm("0.05")



=== FGSM eps=0.01 ===
Confusion Matrix:
 [[308585      0      0      0      0]
 [     0      0      0      0      0]
 [128025      0      0      0      0]
 [ 49536      0      0      0      0]
 [    32      0      0      0      0]]

Classification Report:
               precision    recall  f1-score   support

      Benign     0.6347    1.0000    0.7765    308585
  BruteForce     0.0000    0.0000    0.0000         0
    DoS/DDoS     0.0000    0.0000    0.0000    128025
    PortScan     0.0000    0.0000    0.0000     49536
   Web/Other     0.0000    0.0000    0.0000        32

    accuracy                         0.6347    486178
   macro avg     0.1269    0.2000    0.1553    486178
weighted avg     0.4029    0.6347    0.4929    486178

✅ Saved confusion matrix + report for eps = 0.01

=== FGSM eps=0.05 ===
Confusion Matrix:
 [[308585      0      0      0      0]
 [     0      0      0      0      0]
 [128025      0      0      0      0]
 [ 49536      0      0      0      0]
 [    32  

In [ ]:
import numpy as np, os

cm01 = np.load(f"{RESULTS}/fgsm_eps0.01_confusion.npy")
cm05 = np.load(f"{RESULTS}/fgsm_eps0.05_confusion.npy")

print("Same confusion matrix?", np.array_equal(cm01, cm05))
print("L1 diff sum:", int(np.abs(cm01 - cm05).sum()))


Same confusion matrix? True
L1 diff sum: 0


In [ ]:
import glob, os

files01 = sorted(glob.glob(FGSM_DIR + "/fgsm_eps0.01_part*.csv"))
files05 = sorted(glob.glob(FGSM_DIR + "/fgsm_eps0.05_part*.csv"))

print("eps0.01 first/last:", os.path.basename(files01[0]), os.path.basename(files01[-1]))
print("eps0.05 first/last:", os.path.basename(files05[0]), os.path.basename(files05[-1]))
print("counts:", len(files01), len(files05))


eps0.01 first/last: fgsm_eps0.01_part0000.csv fgsm_eps0.01_part0009.csv
eps0.05 first/last: fgsm_eps0.05_part0000.csv fgsm_eps0.05_part0009.csv
counts: 10 10


In [ ]:
#from google.colab import drive
#drive.mount("/content/drive")

import os, glob, json, numpy as np, tensorflow as tf

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
RAW     = f"{BASE}/data_raw"
PROC    = f"{BASE}/data_processed"
ART     = f"{BASE}/artifacts"
MODELS  = f"{BASE}/models"
RESULTS = f"{BASE}/results"
FIGS    = f"{BASE}/figures"
for p in [RAW, PROC, ART, MODELS, RESULTS, FIGS]:
    os.makedirs(p, exist_ok=True)

# cleaned CICIDS shards (full)
CLEAN_DIR = f"{PROC}/cicids_clean_shards_full"
shards = sorted(glob.glob(CLEAN_DIR + "/*.csv"))
print("Clean shards:", len(shards))
assert shards, "No cleaned shards found. Check CLEAN_DIR."

with open(f"{ART}/feature_list.json") as f:
    feature_list = json.load(f)

print("Num features:", len(feature_list))


Clean shards: 18
Num features: 78


# old code starts, gpu friendly

In [ ]:
LABEL_COL = "label_5class"
CSV_COLUMNS = feature_list + [LABEL_COL]

BATCH = 512
SHUFFLE_BUF = 5000

column_defaults = [tf.constant(0.0, tf.float32)] * len(feature_list) + [tf.constant(0.0, tf.float32)]

def make_attack_only_ds(files, target_class, shuffle=True):
    ds = tf.data.experimental.make_csv_dataset(
        files,
        batch_size=BATCH,
        select_columns=CSV_COLUMNS,
        label_name=LABEL_COL,
        column_defaults=column_defaults,
        num_epochs=1,
        header=True,
        shuffle=shuffle,
        shuffle_buffer_size=SHUFFLE_BUF if shuffle else 1,
        sloppy=False,
        num_parallel_reads=1
    ).ignore_errors()

    def pack(features, label):
        x = tf.stack([features[c] for c in feature_list], axis=1)
        y = tf.cast(tf.round(label), tf.int32)
        return x, y

    ds = ds.map(pack, num_parallel_calls=1)

    # filter per-row
    ds = ds.unbatch()
    ds = ds.filter(lambda x, y: tf.equal(y, target_class))
    ds = ds.map(lambda x, y: x, num_parallel_calls=1)  # keep only x
    ds = ds.batch(BATCH, drop_remainder=True).prefetch(1)
    return ds

bf_ds = make_attack_only_ds(shards, target_class=1, shuffle=True)

xb = next(iter(bf_ds.take(1)))
print("BruteForce batch:", xb.shape, xb.dtype)


BruteForce batch: (512, 78) <dtype: 'float32'>


In [ ]:
BOUNDS_STEPS = 400  # increase to 1000 if you want tighter bounds (slower)

fmin, fmax = None, None
for x in bf_ds.take(BOUNDS_STEPS):
    x = x.numpy()
    if fmin is None:
        fmin = x.min(axis=0); fmax = x.max(axis=0)
    else:
        fmin = np.minimum(fmin, x.min(axis=0))
        fmax = np.maximum(fmax, x.max(axis=0))

np.savez(f"{ART}/ganA_class1_bounds.npz", fmin=fmin, fmax=fmax)
print("✅ Saved bounds:", f"{ART}/ganA_class1_bounds.npz")


✅ Saved bounds: /content/drive/MyDrive/A3IDPS_COLAB/artifacts/ganA_class1_bounds.npz


In [ ]:
LATENT = 64
D = len(feature_list)

fmin_tf = tf.constant(fmin, dtype=tf.float32)
fmax_tf = tf.constant(fmax, dtype=tf.float32)

def scale_to_bounds(tanh_out):
    z01 = (tanh_out + 1.0) / 2.0
    return fmin_tf + z01 * (fmax_tf - fmin_tf)

def make_generator():
    z = tf.keras.Input(shape=(LATENT,))
    x = tf.keras.layers.Dense(128, activation="relu")(z)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dense(D, activation="tanh")(x)
    out = tf.keras.layers.Lambda(scale_to_bounds)(x)
    return tf.keras.Model(z, out, name="ganA_gen_class1")

def make_critic():
    x = tf.keras.Input(shape=(D,))
    h = tf.keras.layers.Dense(256)(x); h = tf.keras.layers.LeakyReLU(0.2)(h)
    h = tf.keras.layers.Dense(256)(h); h = tf.keras.layers.LeakyReLU(0.2)(h)
    h = tf.keras.layers.Dense(128)(h); h = tf.keras.layers.LeakyReLU(0.2)(h)
    out = tf.keras.layers.Dense(1)(h)
    return tf.keras.Model(x, out, name="ganA_critic_class1")

G = make_generator()
C = make_critic()

G.summary()
C.summary()


Model: "ganA_gen_class1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 78)             │        20,046 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 78)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 127,182 (496.80 KB)

 Trainable params: 127,182 (496.80 KB)

 Non-trainable params: 0 (0.00 B)

Model: "ganA_critic_class1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 78)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 256)            │        20,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 119,041 (465.00 KB)

 Trainable params: 119,041 (465.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
G_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)
C_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)

LAMBDA_GP = 10.0
CRITIC_STEPS = 5

@tf.function
def gradient_penalty(real, fake):
    alpha = tf.random.uniform([tf.shape(real)[0], 1], 0.0, 1.0)
    interp = real + alpha * (fake - real)
    with tf.GradientTape() as tape:
        tape.watch(interp)
        pred = C(interp, training=True)
    grads = tape.gradient(pred, interp)
    slopes = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    return tf.reduce_mean((slopes - 1.0) ** 2)

@tf.function
def train_critic(real):
    z = tf.random.normal([tf.shape(real)[0], LATENT])
    with tf.GradientTape() as tape:
        fake = G(z, training=True)
        c_real = C(real, training=True)
        c_fake = C(fake, training=True)
        gp = gradient_penalty(real, fake)
        loss_c = tf.reduce_mean(c_fake) - tf.reduce_mean(c_real) + LAMBDA_GP * gp
    grads = tape.gradient(loss_c, C.trainable_variables)
    C_opt.apply_gradients(zip(grads, C.trainable_variables))
    return loss_c, gp

@tf.function
def train_gen(batch_size):
    z = tf.random.normal([batch_size, LATENT])
    with tf.GradientTape() as tape:
        fake = G(z, training=True)
        c_fake = C(fake, training=True)
        loss_g = -tf.reduce_mean(c_fake)
    grads = tape.gradient(loss_g, G.trainable_variables)
    G_opt.apply_gradients(zip(grads, G.trainable_variables))
    return loss_g


# old code ends, here, new code starts

In [ ]:
# ===== CPU QUICK GAN SETTINGS =====
TOTAL_STEPS = 600        # was 5000
LOG_EVERY = 50
CRITIC_STEPS = 1         # was 5  (BIG speedup)

BATCH = 256              # was 512 (helps CPU)
SHUFFLE_BUF = 5000
# ================================
print("CPU quick settings set ✓")


CPU quick settings set ✓


In [ ]:
LABEL_COL = "label_5class"
CSV_COLUMNS = feature_list + [LABEL_COL]
column_defaults = [tf.constant(0.0, tf.float32)] * len(feature_list) + [tf.constant(0.0, tf.float32)]

def make_attack_only_ds(files, target_class, shuffle=True):
    ds = tf.data.experimental.make_csv_dataset(
        files,
        batch_size=BATCH,
        select_columns=CSV_COLUMNS,
        label_name=LABEL_COL,
        column_defaults=column_defaults,
        num_epochs=1,
        header=True,
        shuffle=shuffle,
        shuffle_buffer_size=SHUFFLE_BUF if shuffle else 1,
        sloppy=False,
        num_parallel_reads=1
    ).ignore_errors()

    def pack(features, label):
        x = tf.stack([features[c] for c in feature_list], axis=1)
        y = tf.cast(tf.round(label), tf.int32)
        return x, y

    ds = ds.map(pack, num_parallel_calls=1)
    ds = ds.unbatch()
    ds = ds.filter(lambda x, y: tf.equal(y, target_class))
    ds = ds.map(lambda x, y: x, num_parallel_calls=1)
    ds = ds.batch(BATCH, drop_remainder=True).prefetch(1)
    return ds

bf_ds = make_attack_only_ds(shards, target_class=1, shuffle=True)

xb = next(iter(bf_ds.take(1)))
print("BruteForce batch:", xb.shape)


BruteForce batch: (256, 78)


In [ ]:
BOUNDS_STEPS = 200  # was 400; CPU faster

fmin, fmax = None, None
for x in bf_ds.take(BOUNDS_STEPS):
    x = x.numpy()
    if fmin is None:
        fmin = x.min(axis=0); fmax = x.max(axis=0)
    else:
        fmin = np.minimum(fmin, x.min(axis=0))
        fmax = np.maximum(fmax, x.max(axis=0))

np.savez(f"{ART}/ganA_class1_bounds.npz", fmin=fmin, fmax=fmax)
print("✅ Saved bounds:", f"{ART}/ganA_class1_bounds.npz")


✅ Saved bounds: /content/drive/MyDrive/A3IDPS_COLAB/artifacts/ganA_class1_bounds.npz


In [ ]:
LATENT = 64
D = len(feature_list)

fmin_tf = tf.constant(fmin, dtype=tf.float32)
fmax_tf = tf.constant(fmax, dtype=tf.float32)

def scale_to_bounds(tanh_out):
    z01 = (tanh_out + 1.0) / 2.0
    return fmin_tf + z01 * (fmax_tf - fmin_tf)

def make_generator_light():
    z = tf.keras.Input(shape=(LATENT,))
    x = tf.keras.layers.Dense(128, activation="relu")(z)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dense(D, activation="tanh")(x)
    out = tf.keras.layers.Lambda(scale_to_bounds)(x)
    return tf.keras.Model(z, out, name="ganA_gen_class1_light")

def make_critic_light():
    x = tf.keras.Input(shape=(D,))
    h = tf.keras.layers.Dense(128)(x); h = tf.keras.layers.LeakyReLU(0.2)(h)
    h = tf.keras.layers.Dense(128)(h); h = tf.keras.layers.LeakyReLU(0.2)(h)
    out = tf.keras.layers.Dense(1)(h)
    return tf.keras.Model(x, out, name="ganA_critic_class1_light")

G = make_generator_light()
C = make_critic_light()

print("✅ Light G/C built")


✅ Light G/C built


In [ ]:
G_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)
C_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)

LAMBDA_GP = 10.0

@tf.function
def gradient_penalty(real, fake):
    alpha = tf.random.uniform([tf.shape(real)[0], 1], 0.0, 1.0)
    interp = real + alpha * (fake - real)
    with tf.GradientTape() as tape:
        tape.watch(interp)
        pred = C(interp, training=True)
    grads = tape.gradient(pred, interp)
    slopes = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    return tf.reduce_mean((slopes - 1.0) ** 2)

@tf.function
def train_critic(real):
    z = tf.random.normal([tf.shape(real)[0], LATENT])
    with tf.GradientTape() as tape:
        fake = G(z, training=True)
        c_real = C(real, training=True)
        c_fake = C(fake, training=True)
        gp = gradient_penalty(real, fake)
        loss_c = tf.reduce_mean(c_fake) - tf.reduce_mean(c_real) + LAMBDA_GP * gp
    grads = tape.gradient(loss_c, C.trainable_variables)
    C_opt.apply_gradients(zip(grads, C.trainable_variables))
    return loss_c, gp

@tf.function
def train_gen(batch_size):
    z = tf.random.normal([batch_size, LATENT])
    with tf.GradientTape() as tape:
        fake = G(z, training=True)
        c_fake = C(fake, training=True)
        loss_g = -tf.reduce_mean(c_fake)
    grads = tape.gradient(loss_g, G.trainable_variables)
    G_opt.apply_gradients(zip(grads, G.trainable_variables))
    return loss_g


In [ ]:
import os, time, json

ckpt_dir = f"{MODELS}/ganA_class1_ckpt"
os.makedirs(ckpt_dir, exist_ok=True)

ckpt = tf.train.Checkpoint(G=G, C=C, G_opt=G_opt, C_opt=C_opt)
manager = tf.train.CheckpointManager(ckpt, ckpt_dir, max_to_keep=3)

ckpt.restore(manager.latest_checkpoint)
print("Restored checkpoint:", manager.latest_checkpoint)


Restored checkpoint: /content/drive/MyDrive/A3IDPS_COLAB/models/ganA_class1_ckpt/ckpt-6


In [ ]:
c_log, g_log, gp_log = [], [], []
it = iter(bf_ds.repeat())

t0 = time.time()
for step in range(1, TOTAL_STEPS + 1):

    # critic updates (CRITIC_STEPS=1)
    real = next(it)
    lc, gp = train_critic(real)

    # generator update
    lg = train_gen(tf.shape(real)[0])

    if step % LOG_EVERY == 0:
        c_log.append(float(lc.numpy()))
        g_log.append(float(lg.numpy()))
        gp_log.append(float(gp.numpy()))
        print(f"[{step}/{TOTAL_STEPS}] C={c_log[-1]:.4f}  G={g_log[-1]:.4f}  GP={gp_log[-1]:.4f}  dt={time.time()-t0:.1f}s")
        t0 = time.time()

    if step % 100 == 0:
        manager.save()
        print("✅ checkpoint saved")

with open(f"{RESULTS}/ganA_class1_curve_cpu.json", "w") as f:
    json.dump({"c_loss": c_log, "g_loss": g_log, "gp": gp_log, "log_every": LOG_EVERY, "steps": TOTAL_STEPS}, f, indent=2)

print("✅ CPU quick GAN-A class1 finished and saved curve.")


[50/600] C=-826676.9375  G=-471640.3750  GP=1.4003  dt=307.5s
[100/600] C=-410369.5938  G=-41819.6445  GP=1.8013  dt=259.6s
✅ checkpoint saved
[150/600] C=-1586755.5000  G=608245.6250  GP=2.6449  dt=281.7s
[200/600] C=-956745.1250  G=-792169.6875  GP=2.4294  dt=179.5s
✅ checkpoint saved
[250/600] C=-915917.9375  G=774797.7500  GP=3.3329  dt=542.6s
[300/600] C=-2529476.5000  G=70258.5234  GP=3.0141  dt=68.8s
✅ checkpoint saved
[350/600] C=-2416145.7500  G=777482.2500  GP=3.7770  dt=414.7s
[400/600] C=-1399817.3750  G=529062.0000  GP=4.1897  dt=454.5s
✅ checkpoint saved
[450/600] C=-1052340.8750  G=-750063.0000  GP=2.9287  dt=360.2s
[500/600] C=-1460672.7500  G=-508315.6875  GP=2.8755  dt=198.0s
✅ checkpoint saved
[550/600] C=-1787259.7500  G=-89455.8594  GP=2.9929  dt=322.1s
[600/600] C=-1885551.7500  G=124325.8906  GP=3.6928  dt=183.2s
✅ checkpoint saved
✅ CPU quick GAN-A class1 finished and saved curve.


In [ ]:
gen_path = f"{MODELS}/ganA_class1_generator.keras"
G.save(gen_path)
print("✅ Saved generator:", gen_path)


✅ Saved generator: /content/drive/MyDrive/A3IDPS_COLAB/models/ganA_class1_generator.keras


In [ ]:
import pandas as pd

SYN_DIR = f"{PROC}/ganA_synth_shards"
os.makedirs(SYN_DIR, exist_ok=True)

TOTAL_SYN = 50_000     # CPU friendly now
PER_SHARD = 25_000

made = 0
sid = 0
while made < TOTAL_SYN:
    n = min(PER_SHARD, TOTAL_SYN - made)
    z = tf.random.normal([n, LATENT])
    x_syn = G(z, training=False).numpy()

    df = pd.DataFrame(x_syn, columns=feature_list)
    df["label_5class"] = 1
    out = f"{SYN_DIR}/ganA_class1_synth_{sid:03d}.csv"
    df.to_csv(out, index=False)

    made += n
    sid += 1
    print("Saved:", out, "| total:", made)

print("✅ Synthetic BruteForce shards saved in:", SYN_DIR)


Saved: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/ganA_synth_shards/ganA_class1_synth_000.csv | total: 25000
Saved: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/ganA_synth_shards/ganA_class1_synth_001.csv | total: 50000
✅ Synthetic BruteForce shards saved in: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/ganA_synth_shards


In [ ]:
base_ids_path = f"{MODELS}/ids_base_5class.keras"
ids = tf.keras.models.load_model(base_ids_path)
print("✅ Loaded base IDS:", base_ids_path)


✅ Loaded base IDS: /content/drive/MyDrive/A3IDPS_COLAB/models/ids_base_5class.keras


In [ ]:
import glob

def make_synth_ds(glob_pattern, batch_size=512, shuffle=True):
    files = sorted(glob.glob(glob_pattern))
    assert files, f"No synth files found: {glob_pattern}"

    col_defaults = [tf.constant(0.0, tf.float32)]*len(feature_list) + [tf.constant(0.0, tf.float32)]
    ds = tf.data.experimental.make_csv_dataset(
        files,
        batch_size=batch_size,
        select_columns=feature_list + ["label_5class"],
        label_name="label_5class",
        column_defaults=col_defaults,
        num_epochs=1,
        header=True,
        shuffle=shuffle,
        shuffle_buffer_size=5000 if shuffle else 1,
        sloppy=False,
        num_parallel_reads=1
    ).ignore_errors()

    def pack(features, label):
        x = tf.stack([features[c] for c in feature_list], axis=1)
        y = tf.cast(tf.round(label), tf.int32)
        return x, y

    return ds.map(pack, num_parallel_calls=1).prefetch(1)

# synthetic BruteForce stream
bf_synth_ds = make_synth_ds(f"{SYN_DIR}/ganA_class1_synth_*.csv", batch_size=512, shuffle=True)

# Augment: 1 synthetic batch for every ~2 clean batches
aug_train = tf.data.Dataset.sample_from_datasets(
    [train_ds.repeat(), bf_synth_ds.repeat()],
    weights=[0.67, 0.33]
).prefetch(1)

print("✅ Augmented training stream ready")


✅ Augmented training stream ready


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5, min_lr=1e-5),
]

hist = ids.fit(
    aug_train,
    validation_data=val_ds.repeat(),
    epochs=8,
    steps_per_epoch=150,
    validation_steps=40,
    callbacks=callbacks,
    verbose=1
)

import json
with open(f"{RESULTS}/ids_after_ganA_class1_history.json", "w") as f:
    json.dump(hist.history, f, indent=2)

ids_path = f"{MODELS}/ids_after_ganA_class1.keras"
ids.save(ids_path)
print("✅ Saved retrained IDS:", ids_path)


Epoch 1/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 24s 124ms/step - accuracy: 0.9506 - loss: 0.2785 - val_accuracy: 0.9862 - val_loss: 0.0712 - learning_rate: 2.5000e-04
Epoch 2/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 18s 119ms/step - accuracy: 0.9834 - loss: 0.0676 - val_accuracy: 0.9964 - val_loss: 0.0378 - learning_rate: 2.5000e-04
Epoch 3/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 15s 99ms/step - accuracy: 0.9982 - loss: 0.0109 - val_accuracy: 0.9972 - val_loss: 0.0384 - learning_rate: 2.5000e-04
Epoch 4/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 16s 107ms/step - accuracy: 0.9988 - loss: 0.0247 - val_accuracy: 0.9972 - val_loss: 0.0357 - learning_rate: 1.2500e-04
Epoch 5/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 15s 99ms/step - accuracy: 0.9846 - loss: 0.0913 - val_accuracy: 0.9932 - val_loss: 0.0551 - learning_rate: 1.2500e-04
Epoch 6/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 14s 94ms/step - accuracy: 0.9988 - loss: 0.0133 - val_accuracy: 0.9949 - val_loss: 0.0466 - learning_rate: 6.2500e-05
✅ Saved retrained IDS: /content/drive/MyDrive/A3IDPS_CO

In [ ]:
clean_loss, clean_acc = ids.evaluate(test_ds, verbose=1)
print("✅ Clean acc after GAN-A class1:", clean_acc)


475/475 ━━━━━━━━━━━━━━━━━━━━ 39s 82ms/step - accuracy: 0.4821 - loss: 3.8144
✅ Clean acc after GAN-A class1: 0.6344507336616516


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [ ]:
def make_adv_ds_from_eps(eps, batch_size=512):
    files = sorted(glob.glob(FGSM_DIR + f"/fgsm_eps{eps}_part*.csv"))
    col_defaults = [tf.constant(0.0, tf.float32)]*len(feature_list) + [tf.constant(0.0, tf.float32)]
    ds = tf.data.experimental.make_csv_dataset(
        files,
        batch_size=batch_size,
        select_columns=feature_list + ["label_5class"],
        label_name="label_5class",
        column_defaults=col_defaults,
        num_epochs=1,
        header=True,
        shuffle=False,
        sloppy=False,
        num_parallel_reads=1
    ).ignore_errors()

    def pack(features, label):
        x = tf.stack([features[c] for c in feature_list], axis=1)
        y = tf.cast(tf.round(label), tf.int32)
        return x, y

    return ds.map(pack, num_parallel_calls=1).prefetch(1)

fgsm01_ds = make_adv_ds_from_eps("0.01")
fgsm05_ds = make_adv_ds_from_eps("0.05")

fgsm01_loss, fgsm01_acc = ids.evaluate(fgsm01_ds, verbose=1)
fgsm05_loss, fgsm05_acc = ids.evaluate(fgsm05_ds, verbose=1)

print("✅ FGSM acc eps=0.01 after GAN-A class1:", fgsm01_acc)
print("✅ FGSM acc eps=0.05 after GAN-A class1:", fgsm05_acc)


950/950 ━━━━━━━━━━━━━━━━━━━━ 38s 40ms/step - accuracy: 0.4823 - loss: 143557.9844
950/950 ━━━━━━━━━━━━━━━━━━━━ 34s 36ms/step - accuracy: 0.4823 - loss: 717785.6250
✅ FGSM acc eps=0.01 after GAN-A class1: 0.6347160935401917
✅ FGSM acc eps=0.05 after GAN-A class1: 0.6347160935401917


In [ ]:
import json, os

out = {
    "clean_acc": float(clean_acc),
    "fgsm_eps0.01_acc": float(fgsm01_acc),
    "fgsm_eps0.05_acc": float(fgsm05_acc),
}
with open(f"{RESULTS}/ids_after_ganA_class1_metrics.json", "w") as f:
    json.dump(out, f, indent=2)

print("✅ Saved:", f"{RESULTS}/ids_after_ganA_class1_metrics.json")


✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/ids_after_ganA_class1_metrics.json


In [ ]:
import json

# Evaluate base IDS (saved) on the same test_ds right now
base_ids = tf.keras.models.load_model(f"{MODELS}/ids_base_5class.keras")
base_loss_now, base_acc_now = base_ids.evaluate(test_ds, verbose=1)

print("Base IDS clean acc on CURRENT test_ds:", base_acc_now)

with open(f"{RESULTS}/sanity_base_clean_on_current_testds.json", "w") as f:
    json.dump({"base_clean_acc_on_current_test_ds": float(base_acc_now)}, f, indent=2)


475/475 ━━━━━━━━━━━━━━━━━━━━ 41s 83ms/step - accuracy: 0.4811 - loss: 9.4607
Base IDS clean acc on CURRENT test_ds: 0.6330808997154236


In [ ]:
dos_ds = make_attack_only_ds(shards, target_class=2, shuffle=True)
xb = next(iter(dos_ds.take(1)))
print("DoS/DDoS batch:", xb.shape)


DoS/DDoS batch: (256, 78)


In [ ]:
BOUNDS_STEPS = 200

fmin, fmax = None, None
for x in dos_ds.take(BOUNDS_STEPS):
    x = x.numpy()
    if fmin is None:
        fmin = x.min(axis=0); fmax = x.max(axis=0)
    else:
        fmin = np.minimum(fmin, x.min(axis=0))
        fmax = np.maximum(fmax, x.max(axis=0))

np.savez(f"{ART}/ganA_class2_bounds.npz", fmin=fmin, fmax=fmax)
print("✅ Saved bounds:", f"{ART}/ganA_class2_bounds.npz")


✅ Saved bounds: /content/drive/MyDrive/A3IDPS_COLAB/artifacts/ganA_class2_bounds.npz


In [ ]:
fmin_tf = tf.constant(fmin, dtype=tf.float32)
fmax_tf = tf.constant(fmax, dtype=tf.float32)

def scale_to_bounds(tanh_out):
    z01 = (tanh_out + 1.0) / 2.0
    return fmin_tf + z01 * (fmax_tf - fmin_tf)

def make_generator_light(name):
    z = tf.keras.Input(shape=(LATENT,))
    x = tf.keras.layers.Dense(128, activation="relu")(z)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dense(D, activation="tanh")(x)
    out = tf.keras.layers.Lambda(scale_to_bounds)(x)
    return tf.keras.Model(z, out, name=name)

def make_critic_light(name):
    x = tf.keras.Input(shape=(D,))
    h = tf.keras.layers.Dense(128)(x); h = tf.keras.layers.LeakyReLU(0.2)(h)
    h = tf.keras.layers.Dense(128)(h); h = tf.keras.layers.LeakyReLU(0.2)(h)
    out = tf.keras.layers.Dense(1)(h)
    return tf.keras.Model(x, out, name=name)

G = make_generator_light("ganA_gen_class2_light")
C = make_critic_light("ganA_critic_class2_light")

# Reset opts
G_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)
C_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)

print("✅ Built class2 G/C")


✅ Built class2 G/C


In [ ]:
ckpt_dir = f"{MODELS}/ganA_class2_ckpt"
os.makedirs(ckpt_dir, exist_ok=True)

ckpt = tf.train.Checkpoint(G=G, C=C, G_opt=G_opt, C_opt=C_opt)
manager = tf.train.CheckpointManager(ckpt, ckpt_dir, max_to_keep=3)

ckpt.restore(manager.latest_checkpoint)
print("Restored checkpoint:", manager.latest_checkpoint)


Restored checkpoint: None


In [ ]:
import time, json

TOTAL_STEPS = 600
LOG_EVERY = 50

c_log, g_log, gp_log = [], [], []
it = iter(dos_ds.repeat())

t0 = time.time()
for step in range(1, TOTAL_STEPS + 1):
    real = next(it)
    lc, gp = train_critic(real)
    lg = train_gen(tf.shape(real)[0])

    if step % LOG_EVERY == 0:
        c_log.append(float(lc.numpy()))
        g_log.append(float(lg.numpy()))
        gp_log.append(float(gp.numpy()))
        print(f"[{step}/{TOTAL_STEPS}] C={c_log[-1]:.4f}  G={g_log[-1]:.4f}  GP={gp_log[-1]:.4f}  dt={time.time()-t0:.1f}s")
        t0 = time.time()

    if step % 100 == 0:
        manager.save()
        print("✅ checkpoint saved")

with open(f"{RESULTS}/ganA_class2_curve_cpu.json", "w") as f:
    json.dump({"c_loss": c_log, "g_loss": g_log, "gp": gp_log, "log_every": LOG_EVERY, "steps": TOTAL_STEPS}, f, indent=2)

print("✅ Class2 GAN training done.")


[50/600] C=-6756859.5000  G=-1870503.2500  GP=3.1111  dt=7.5s
[100/600] C=-40489932.0000  G=-16593476.0000  GP=6.2424  dt=3.4s
✅ checkpoint saved
[150/600] C=-29886364.0000  G=-21444832.0000  GP=9.6806  dt=2.4s
[200/600] C=-55114660.0000  G=-13124880.0000  GP=21.7287  dt=4.3s
✅ checkpoint saved
[250/600] C=-139241584.0000  G=-17480844.0000  GP=38.0718  dt=5.8s
[300/600] C=-180986432.0000  G=-22303276.0000  GP=64.4905  dt=8.0s
✅ checkpoint saved
[350/600] C=-183078960.0000  G=-23761534.0000  GP=110.5154  dt=5.9s
[400/600] C=-242634992.0000  G=-31251850.0000  GP=162.3180  dt=5.0s
✅ checkpoint saved
[450/600] C=-206494000.0000  G=-32151626.0000  GP=244.7547  dt=6.1s
[500/600] C=-52523080.0000  G=-22891748.0000  GP=306.5309  dt=6.6s
✅ checkpoint saved
[550/600] C=-1771635072.0000  G=-63900344.0000  GP=344.8303  dt=145.3s
[600/600] C=-2041447040.0000  G=-97618688.0000  GP=494.2727  dt=2.1s
✅ checkpoint saved
✅ Class2 GAN training done.


In [ ]:
gen_path = f"{MODELS}/ganA_class2_generator.keras"
G.save(gen_path)
print("✅ Saved:", gen_path)


✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/models/ganA_class2_generator.keras


In [ ]:
import pandas as pd

SYN_DIR = f"{PROC}/ganA_synth_shards"
os.makedirs(SYN_DIR, exist_ok=True)

TOTAL_SYN = 100_000
PER_SHARD = 25_000

made = 0
sid = 0
while made < TOTAL_SYN:
    n = min(PER_SHARD, TOTAL_SYN - made)
    z = tf.random.normal([n, LATENT])
    x_syn = G(z, training=False).numpy()

    df = pd.DataFrame(x_syn, columns=feature_list)
    df["label_5class"] = 2
    out = f"{SYN_DIR}/ganA_class2_synth_{sid:03d}.csv"
    df.to_csv(out, index=False)

    made += n
    sid += 1
    print("Saved:", out, "| total:", made)

print("✅ Class2 synthetic saved.")


Saved: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/ganA_synth_shards/ganA_class2_synth_000.csv | total: 25000
Saved: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/ganA_synth_shards/ganA_class2_synth_001.csv | total: 50000
Saved: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/ganA_synth_shards/ganA_class2_synth_002.csv | total: 75000
Saved: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/ganA_synth_shards/ganA_class2_synth_003.csv | total: 100000
✅ Class2 synthetic saved.


In [ ]:
ids = tf.keras.models.load_model(f"{MODELS}/ids_after_ganA_class1.keras")
print("✅ Loaded IDS after class1")

dos_synth_ds = make_synth_ds(f"{SYN_DIR}/ganA_class2_synth_*.csv", batch_size=512, shuffle=True)

# Stronger mix: 50% synthetic
aug_train = tf.data.Dataset.sample_from_datasets(
    [train_ds.repeat(), dos_synth_ds.repeat()],
    weights=[0.50, 0.50]
).prefetch(1)

hist2 = ids.fit(
    aug_train,
    validation_data=val_ds.repeat(),
    epochs=8,
    steps_per_epoch=150,
    validation_steps=40,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5, min_lr=1e-5),
    ],
    verbose=1
)

import json
with open(f"{RESULTS}/ids_after_ganA_class2_history.json", "w") as f:
    json.dump(hist2.history, f, indent=2)

out_path = f"{MODELS}/ids_after_ganA_class2.keras"
ids.save(out_path)
print("✅ Saved:", out_path)


✅ Loaded IDS after class1
Epoch 1/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 21s 113ms/step - accuracy: 0.4907 - loss: 2.5465 - val_accuracy: 0.9915 - val_loss: 0.1226 - learning_rate: 3.1250e-05
Epoch 2/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 15s 103ms/step - accuracy: 0.8797 - loss: 0.2529 - val_accuracy: 0.9608 - val_loss: 0.1873 - learning_rate: 3.1250e-05
Epoch 3/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 15s 99ms/step - accuracy: 0.9497 - loss: 0.1703 - val_accuracy: 0.9745 - val_loss: 0.1109 - learning_rate: 1.5625e-05
Epoch 4/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 20s 134ms/step - accuracy: 0.9863 - loss: 0.0829 - val_accuracy: 0.9947 - val_loss: 0.0831 - learning_rate: 1.5625e-05
Epoch 5/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 12s 81ms/step - accuracy: 0.9844 - loss: 0.0794 - val_accuracy: 0.9958 - val_loss: 0.0714 - learning_rate: 1.5625e-05
Epoch 6/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 13s 87ms/step - accuracy: 0.9901 - loss: 0.0553 - val_accuracy: 0.9961 - val_loss: 0.0608 - learning_rate: 1.5625e-05
Epoch 7/8
150/150 ━━━━━━━━━━━

In [ ]:
clean_loss2, clean_acc2 = ids.evaluate(test_ds, verbose=1)
fgsm01_loss2, fgsm01_acc2 = ids.evaluate(make_adv_ds_from_eps("0.01"), verbose=1)
fgsm05_loss2, fgsm05_acc2 = ids.evaluate(make_adv_ds_from_eps("0.05"), verbose=1)

print("✅ After GAN-A class2 | Clean:", clean_acc2, "FGSM0.01:", fgsm01_acc2, "FGSM0.05:", fgsm05_acc2)

import json
with open(f"{RESULTS}/ids_after_ganA_class2_metrics.json", "w") as f:
    json.dump({
        "clean_acc": float(clean_acc2),
        "fgsm_eps0.01_acc": float(fgsm01_acc2),
        "fgsm_eps0.05_acc": float(fgsm05_acc2)
    }, f, indent=2)

print("✅ Saved metrics json for class2.")


475/475 ━━━━━━━━━━━━━━━━━━━━ 35s 74ms/step - accuracy: 0.6778 - loss: 1.1787


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


950/950 ━━━━━━━━━━━━━━━━━━━━ 34s 36ms/step - accuracy: 0.4822 - loss: 76569.9844
950/950 ━━━━━━━━━━━━━━━━━━━━ 34s 35ms/step - accuracy: 0.4822 - loss: 382848.9062
✅ After GAN-A class2 | Clean: 0.7489129304885864 FGSM0.01: 0.6344383955001831 FGSM0.05: 0.6344383955001831
✅ Saved metrics json for class2.


In [ ]:
# Stability tweaks for CPU quick GAN
LAMBDA_GP = 5.0          # was 10 (reduce exploding GP)
TOTAL_STEPS = 600
LOG_EVERY = 50


In [ ]:
port_ds = make_attack_only_ds(shards, target_class=3, shuffle=True)
xb = next(iter(port_ds.take(1)))
print("PortScan batch:", xb.shape)


PortScan batch: (256, 78)


In [ ]:
BOUNDS_STEPS = 200
fmin, fmax = None, None

for x in port_ds.take(BOUNDS_STEPS):
    x = x.numpy()
    if fmin is None:
        fmin = x.min(axis=0); fmax = x.max(axis=0)
    else:
        fmin = np.minimum(fmin, x.min(axis=0))
        fmax = np.maximum(fmax, x.max(axis=0))

np.savez(f"{ART}/ganA_class3_bounds.npz", fmin=fmin, fmax=fmax)
print("✅ Saved bounds:", f"{ART}/ganA_class3_bounds.npz")


✅ Saved bounds: /content/drive/MyDrive/A3IDPS_COLAB/artifacts/ganA_class3_bounds.npz


In [ ]:
fmin_tf = tf.constant(fmin, dtype=tf.float32)
fmax_tf = tf.constant(fmax, dtype=tf.float32)

def scale_to_bounds(tanh_out):
    z01 = (tanh_out + 1.0) / 2.0
    return fmin_tf + z01 * (fmax_tf - fmin_tf)

G = make_generator_light("ganA_gen_class3_light")
C = make_critic_light("ganA_critic_class3_light")

# Reset optimizers (with optional critic clipnorm)
G_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)
C_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9, clipnorm=1.0)

print("✅ Built class3 G/C")


✅ Built class3 G/C


In [ ]:
ckpt_dir = f"{MODELS}/ganA_class3_ckpt"
os.makedirs(ckpt_dir, exist_ok=True)

ckpt = tf.train.Checkpoint(G=G, C=C, G_opt=G_opt, C_opt=C_opt)
manager = tf.train.CheckpointManager(ckpt, ckpt_dir, max_to_keep=3)

ckpt.restore(manager.latest_checkpoint)
print("Restored checkpoint:", manager.latest_checkpoint)


Restored checkpoint: None


In [ ]:
import time, json

c_log, g_log, gp_log = [], [], []
it = iter(port_ds.repeat())

t0 = time.time()
for step in range(1, TOTAL_STEPS + 1):
    real = next(it)
    lc, gp = train_critic(real)   # uses updated LAMBDA_GP
    lg = train_gen(tf.shape(real)[0])

    if step % LOG_EVERY == 0:
        c_log.append(float(lc.numpy()))
        g_log.append(float(lg.numpy()))
        gp_log.append(float(gp.numpy()))
        print(f"[{step}/{TOTAL_STEPS}] C={c_log[-1]:.4f}  G={g_log[-1]:.4f}  GP={gp_log[-1]:.4f}  dt={time.time()-t0:.1f}s")
        t0 = time.time()

    if step % 100 == 0:
        manager.save()
        print("✅ checkpoint saved")

with open(f"{RESULTS}/ganA_class3_curve_cpu.json", "w") as f:
    json.dump({"c_loss": c_log, "g_loss": g_log, "gp": gp_log, "log_every": LOG_EVERY, "steps": TOTAL_STEPS}, f, indent=2)

print("✅ Class3 GAN training done.")


[50/600] C=19635928.0000  G=-17145720.0000  GP=595.2097  dt=283.4s
[100/600] C=-1685349.3750  G=2163677.5000  GP=518.9580  dt=1.9s
✅ checkpoint saved
[150/600] C=-10483751.0000  G=11359088.0000  GP=423.9570  dt=2.0s
[200/600] C=-11664205.0000  G=12209888.0000  GP=412.4792  dt=1.9s
✅ checkpoint saved
[250/600] C=-20981678.0000  G=17628586.0000  GP=391.3749  dt=2.6s
[300/600] C=-29973762.0000  G=28266820.0000  GP=381.3757  dt=3.2s
✅ checkpoint saved
[350/600] C=-38637532.0000  G=38815840.0000  GP=366.6571  dt=2.0s
[400/600] C=-35859576.0000  G=34619672.0000  GP=276.3301  dt=1.9s
✅ checkpoint saved
[450/600] C=-41409980.0000  G=41170108.0000  GP=296.9859  dt=28.5s
[500/600] C=-49723736.0000  G=49386564.0000  GP=354.4185  dt=1.6s
✅ checkpoint saved
[550/600] C=-58387116.0000  G=57684088.0000  GP=447.7654  dt=3.4s
[600/600] C=-67095620.0000  G=66500084.0000  GP=589.9968  dt=4.4s
✅ checkpoint saved
✅ Class3 GAN training done.


In [ ]:
gen_path = f"{MODELS}/ganA_class3_generator.keras"
G.save(gen_path)
print("✅ Saved:", gen_path)


✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/models/ganA_class3_generator.keras


In [ ]:
import pandas as pd

SYN_DIR = f"{PROC}/ganA_synth_shards"
os.makedirs(SYN_DIR, exist_ok=True)

TOTAL_SYN = 100_000
PER_SHARD = 25_000

made = 0
sid = 0
while made < TOTAL_SYN:
    n = min(PER_SHARD, TOTAL_SYN - made)
    z = tf.random.normal([n, LATENT])
    x_syn = G(z, training=False).numpy()

    df = pd.DataFrame(x_syn, columns=feature_list)
    df["label_5class"] = 3
    out = f"{SYN_DIR}/ganA_class3_synth_{sid:03d}.csv"
    df.to_csv(out, index=False)

    made += n
    sid += 1
    print("Saved:", out, "| total:", made)

print("✅ Class3 synthetic saved.")


Saved: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/ganA_synth_shards/ganA_class3_synth_000.csv | total: 25000
Saved: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/ganA_synth_shards/ganA_class3_synth_001.csv | total: 50000
Saved: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/ganA_synth_shards/ganA_class3_synth_002.csv | total: 75000
Saved: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/ganA_synth_shards/ganA_class3_synth_003.csv | total: 100000
✅ Class3 synthetic saved.


In [ ]:
ids = tf.keras.models.load_model(f"{MODELS}/ids_after_ganA_class2.keras")
print("✅ Loaded IDS after class2")

port_synth_ds = make_synth_ds(f"{SYN_DIR}/ganA_class3_synth_*.csv", batch_size=512, shuffle=True)

# Keep strong mix
aug_train = tf.data.Dataset.sample_from_datasets(
    [train_ds.repeat(), port_synth_ds.repeat()],
    weights=[0.50, 0.50]
).prefetch(1)

hist3 = ids.fit(
    aug_train,
    validation_data=val_ds.repeat(),
    epochs=8,
    steps_per_epoch=150,
    validation_steps=40,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5, min_lr=1e-5),
    ],
    verbose=1
)

import json
with open(f"{RESULTS}/ids_after_ganA_class3_history.json", "w") as f:
    json.dump(hist3.history, f, indent=2)

out_path = f"{MODELS}/ids_after_ganA_class3.keras"
ids.save(out_path)
print("✅ Saved:", out_path)


✅ Loaded IDS after class2
Epoch 1/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 21s 116ms/step - accuracy: 0.6418 - loss: 4.3300 - val_accuracy: 0.9966 - val_loss: 0.0449 - learning_rate: 1.5625e-05
Epoch 2/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 15s 99ms/step - accuracy: 0.7500 - loss: 0.8418 - val_accuracy: 0.9966 - val_loss: 0.0413 - learning_rate: 1.5625e-05
Epoch 3/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 21s 141ms/step - accuracy: 0.9522 - loss: 0.1705 - val_accuracy: 0.9967 - val_loss: 0.0405 - learning_rate: 1.5625e-05
Epoch 4/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 13s 86ms/step - accuracy: 0.9876 - loss: 0.0766 - val_accuracy: 0.9959 - val_loss: 0.0398 - learning_rate: 1.5625e-05
Epoch 5/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 11s 74ms/step - accuracy: 0.9575 - loss: 0.2075 - val_accuracy: 0.9949 - val_loss: 0.0424 - learning_rate: 1.5625e-05
Epoch 6/8
150/150 ━━━━━━━━━━━━━━━━━━━━ 13s 88ms/step - accuracy: 0.9909 - loss: 0.0500 - val_accuracy: 0.9952 - val_loss: 0.0409 - learning_rate: 1.0000e-05
✅ Saved: /content/drive/MyDriv

In [ ]:
clean_loss3, clean_acc3 = ids.evaluate(test_ds, verbose=1)
fgsm01_loss3, fgsm01_acc3 = ids.evaluate(make_adv_ds_from_eps("0.01"), verbose=1)
fgsm05_loss3, fgsm05_acc3 = ids.evaluate(make_adv_ds_from_eps("0.05"), verbose=1)

print("✅ After GAN-A class3 | Clean:", clean_acc3, "FGSM0.01:", fgsm01_acc3, "FGSM0.05:", fgsm05_acc3)

import json
with open(f"{RESULTS}/ids_after_ganA_class3_metrics.json", "w") as f:
    json.dump({
        "clean_acc": float(clean_acc3),
        "fgsm_eps0.01_acc": float(fgsm01_acc3),
        "fgsm_eps0.05_acc": float(fgsm05_acc3)
    }, f, indent=2)

print("✅ Saved metrics json for class3.")


475/475 ━━━━━━━━━━━━━━━━━━━━ 40s 84ms/step - accuracy: 0.6776 - loss: 1.1941


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


950/950 ━━━━━━━━━━━━━━━━━━━━ 38s 39ms/step - accuracy: 0.4822 - loss: 78471.3438
950/950 ━━━━━━━━━━━━━━━━━━━━ 38s 40ms/step - accuracy: 0.4822 - loss: 392356.1875
✅ After GAN-A class3 | Clean: 0.7483411431312561 FGSM0.01: 0.6344383955001831 FGSM0.05: 0.6344383955001831
✅ Saved metrics json for class3.


In [ ]:
LAMBDA_GP = 2.0   # further reduce exploding GP on CPU quick
TOTAL_STEPS = 600
LOG_EVERY = 50
print("Updated LAMBDA_GP:", LAMBDA_GP)


Updated LAMBDA_GP: 2.0


In [ ]:
web_ds = make_attack_only_ds(shards, target_class=4, shuffle=True)
xb = next(iter(web_ds.take(1)))
print("Web/Other batch:", xb.shape)


Web/Other batch: (256, 78)


In [ ]:
BOUNDS_STEPS = 200
fmin, fmax = None, None

for x in web_ds.take(BOUNDS_STEPS):
    x = x.numpy()
    if fmin is None:
        fmin = x.min(axis=0); fmax = x.max(axis=0)
    else:
        fmin = np.minimum(fmin, x.min(axis=0))
        fmax = np.maximum(fmax, x.max(axis=0))

np.savez(f"{ART}/ganA_class4_bounds.npz", fmin=fmin, fmax=fmax)
print("✅ Saved bounds:", f"{ART}/ganA_class4_bounds.npz")


✅ Saved bounds: /content/drive/MyDrive/A3IDPS_COLAB/artifacts/ganA_class4_bounds.npz


In [ ]:
fmin_tf = tf.constant(fmin, dtype=tf.float32)
fmax_tf = tf.constant(fmax, dtype=tf.float32)

def scale_to_bounds(tanh_out):
    z01 = (tanh_out + 1.0) / 2.0
    return fmin_tf + z01 * (fmax_tf - fmin_tf)

G = make_generator_light("ganA_gen_class4_light")
C = make_critic_light("ganA_critic_class4_light")

G_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)
C_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9, clipnorm=1.0)

print("✅ Built class4 G/C")


✅ Built class4 G/C


In [ ]:
ckpt_dir = f"{MODELS}/ganA_class4_ckpt"
os.makedirs(ckpt_dir, exist_ok=True)

ckpt = tf.train.Checkpoint(G=G, C=C, G_opt=G_opt, C_opt=C_opt)
manager = tf.train.CheckpointManager(ckpt, ckpt_dir, max_to_keep=3)

ckpt.restore(manager.latest_checkpoint)
print("Restored checkpoint:", manager.latest_checkpoint)


Restored checkpoint: None


In [ ]:
import time, json

c_log, g_log, gp_log = [], [], []
it = iter(web_ds.repeat())

t0 = time.time()
for step in range(1, TOTAL_STEPS + 1):
    real = next(it)
    lc, gp = train_critic(real)
    lg = train_gen(tf.shape(real)[0])

    if step % LOG_EVERY == 0:
        c_log.append(float(lc.numpy()))
        g_log.append(float(lg.numpy()))
        gp_log.append(float(gp.numpy()))
        print(f"[{step}/{TOTAL_STEPS}] C={c_log[-1]:.4f}  G={g_log[-1]:.4f}  GP={gp_log[-1]:.4f}  dt={time.time()-t0:.1f}s")
        t0 = time.time()

    if step % 100 == 0:
        manager.save()
        print("✅ checkpoint saved")

with open(f"{RESULTS}/ganA_class4_curve_cpu.json", "w") as f:
    json.dump({"c_loss": c_log, "g_loss": g_log, "gp": gp_log, "log_every": LOG_EVERY, "steps": TOTAL_STEPS}, f, indent=2)

print("✅ Class4 GAN training done.")


[50/600] C=-64276004.0000  G=64379132.0000  GP=591.9241  dt=1082.5s
[100/600] C=89775040.0000  G=13421168.0000  GP=878.8394  dt=898.4s
✅ checkpoint saved
[150/600] C=-3769503.7500  G=3515563.2500  GP=745.4036  dt=1097.3s
[200/600] C=4348256.5000  G=-1486308.2500  GP=842.4943  dt=1126.4s
✅ checkpoint saved
[250/600] C=10746096.0000  G=-3875392.0000  GP=905.0886  dt=950.6s
[300/600] C=714868.2500  G=-4239772.0000  GP=493.5490  dt=971.3s
✅ checkpoint saved
[350/600] C=42416596.0000  G=-43578348.0000  GP=441.0786  dt=771.7s
[400/600] C=26235962.0000  G=-47265036.0000  GP=559.4630  dt=1099.3s
✅ checkpoint saved
[450/600] C=-70329392.0000  G=-43634184.0000  GP=945.1343  dt=1310.2s
[500/600] C=-82652416.0000  G=-54069652.0000  GP=930.5851  dt=772.4s
✅ checkpoint saved


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, glob, json, numpy as np, tensorflow as tf

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
PROC    = f"{BASE}/data_processed"
ART     = f"{BASE}/artifacts"
MODELS  = f"{BASE}/models"
RESULTS = f"{BASE}/results"

for p in [PROC, ART, MODELS, RESULTS]:
    os.makedirs(p, exist_ok=True)

with open(f"{ART}/feature_list.json") as f:
    feature_list = json.load(f)

print("✅ feature_list loaded:", len(feature_list))


Mounted at /content/drive
✅ feature_list loaded: 78


In [ ]:
BATCH = 256
SHUFFLE_BUF = 5000
target_class = 4

CLEAN_DIR = f"{PROC}/cicids_clean_shards_full"
shards = sorted(glob.glob(CLEAN_DIR + "/*.csv"))
assert shards, "No cleaned shards found at CLEAN_DIR."

LABEL_COL = "label_5class"
CSV_COLUMNS = feature_list + [LABEL_COL]
column_defaults = [tf.constant(0.0, tf.float32)] * len(feature_list) + [tf.constant(0.0, tf.float32)]

def make_attack_only_ds(files, target_class, shuffle=True):
    ds = tf.data.experimental.make_csv_dataset(
        files,
        batch_size=BATCH,
        select_columns=CSV_COLUMNS,
        label_name=LABEL_COL,
        column_defaults=column_defaults,
        num_epochs=1,
        header=True,
        shuffle=shuffle,
        shuffle_buffer_size=SHUFFLE_BUF if shuffle else 1,
        sloppy=False,
        num_parallel_reads=1
    ).ignore_errors()

    def pack(features, label):
        x = tf.stack([features[c] for c in feature_list], axis=1)
        y = tf.cast(tf.round(label), tf.int32)
        return x, y

    ds = ds.map(pack, num_parallel_calls=1)
    ds = ds.unbatch()
    ds = ds.filter(lambda x, y: tf.equal(y, target_class))
    ds = ds.map(lambda x, y: x, num_parallel_calls=1)
    ds = ds.batch(BATCH, drop_remainder=True).prefetch(1)
    return ds

web_ds = make_attack_only_ds(shards, target_class=4, shuffle=True)
print("✅ web_ds batch:", next(iter(web_ds.take(1))).shape)


✅ web_ds batch: (256, 78)


In [ ]:
bounds = np.load(f"{ART}/ganA_class4_bounds.npz")
fmin = bounds["fmin"]; fmax = bounds["fmax"]

LATENT = 64
D = len(feature_list)

fmin_tf = tf.constant(fmin, tf.float32)
fmax_tf = tf.constant(fmax, tf.float32)

def scale_to_bounds(tanh_out):
    z01 = (tanh_out + 1.0) / 2.0
    return fmin_tf + z01 * (fmax_tf - fmin_tf)

def make_generator_light(name):
    z = tf.keras.Input(shape=(LATENT,))
    x = tf.keras.layers.Dense(128, activation="relu")(z)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dense(D, activation="tanh")(x)
    out = tf.keras.layers.Lambda(scale_to_bounds)(x)
    return tf.keras.Model(z, out, name=name)

def make_critic_light(name):
    x = tf.keras.Input(shape=(D,))
    h = tf.keras.layers.Dense(128)(x); h = tf.keras.layers.LeakyReLU(0.2)(h)
    h = tf.keras.layers.Dense(128)(h); h = tf.keras.layers.LeakyReLU(0.2)(h)
    out = tf.keras.layers.Dense(1)(h)
    return tf.keras.Model(x, out, name=name)

G = make_generator_light("ganA_gen_class4_light")
C = make_critic_light("ganA_critic_class4_light")

# Use the same LAMBDA_GP we intended for class4
LAMBDA_GP = 2.0

G_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)
C_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9, clipnorm=1.0)

print("✅ Rebuilt class4 G/C + opts")


✅ Rebuilt class4 G/C + opts


In [ ]:
@tf.function
def gradient_penalty(real, fake):
    alpha = tf.random.uniform([tf.shape(real)[0], 1], 0.0, 1.0)
    interp = real + alpha * (fake - real)
    with tf.GradientTape() as tape:
        tape.watch(interp)
        pred = C(interp, training=True)
    grads = tape.gradient(pred, interp)
    slopes = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    return tf.reduce_mean((slopes - 1.0) ** 2)

@tf.function
def train_critic(real):
    z = tf.random.normal([tf.shape(real)[0], LATENT])
    with tf.GradientTape() as tape:
        fake = G(z, training=True)
        c_real = C(real, training=True)
        c_fake = C(fake, training=True)
        gp = gradient_penalty(real, fake)
        loss_c = tf.reduce_mean(c_fake) - tf.reduce_mean(c_real) + LAMBDA_GP * gp
    grads = tape.gradient(loss_c, C.trainable_variables)
    C_opt.apply_gradients(zip(grads, C.trainable_variables))
    return loss_c, gp

@tf.function
def train_gen(batch_size):
    z = tf.random.normal([batch_size, LATENT])
    with tf.GradientTape() as tape:
        fake = G(z, training=True)
        c_fake = C(fake, training=True)
        loss_g = -tf.reduce_mean(c_fake)
    grads = tape.gradient(loss_g, G.trainable_variables)
    G_opt.apply_gradients(zip(grads, G.trainable_variables))
    return loss_g

ckpt_dir = f"{MODELS}/ganA_class4_ckpt"
ckpt = tf.train.Checkpoint(G=G, C=C, G_opt=G_opt, C_opt=C_opt)
manager = tf.train.CheckpointManager(ckpt, ckpt_dir, max_to_keep=3)

ckpt.restore(manager.latest_checkpoint)
print("✅ Restored:", manager.latest_checkpoint)


✅ Restored: /content/drive/MyDrive/A3IDPS_COLAB/models/ganA_class4_ckpt/ckpt-5


In [ ]:
import time

TOTAL_STEPS = 600
START_STEP = 501
LOG_EVERY = 25

it = iter(web_ds.repeat())
t0 = time.time()

for step in range(START_STEP, TOTAL_STEPS + 1):
    real = next(it)
    lc, gp = train_critic(real)
    lg = train_gen(tf.shape(real)[0])

    if step % LOG_EVERY == 0 or step == TOTAL_STEPS:
        print(f"[{step}/{TOTAL_STEPS}] C={float(lc.numpy()):.4f}  G={float(lg.numpy()):.4f}  GP={float(gp.numpy()):.4f}  dt={time.time()-t0:.1f}s")
        t0 = time.time()

manager.save()
print("✅ Finished 600 + saved final checkpoint")


[525/600] C=-38122600.0000  G=46226392.0000  GP=0.0545  dt=482.2s
[550/600] C=-57061984.0000  G=63946740.0000  GP=0.0108  dt=501.6s
[575/600] C=-65111340.0000  G=74351752.0000  GP=0.0166  dt=391.8s
[600/600] C=-64928528.0000  G=74894136.0000  GP=0.1785  dt=663.1s
✅ Finished 600 + saved final checkpoint


In [ ]:
gen_path = f"{MODELS}/ganA_class4_generator.keras"
G.save(gen_path)
print("✅ Saved:", gen_path)


✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/models/ganA_class4_generator.keras


In [ ]:
import pandas as pd

SYN_DIR = f"{PROC}/ganA_synth_shards"
os.makedirs(SYN_DIR, exist_ok=True)

TOTAL_SYN = 50_000
PER_SHARD = 25_000

made = 0
sid = 0
while made < TOTAL_SYN:
    n = min(PER_SHARD, TOTAL_SYN - made)
    z = tf.random.normal([n, LATENT])
    x_syn = G(z, training=False).numpy()

    df = pd.DataFrame(x_syn, columns=feature_list)
    df["label_5class"] = 4
    out = f"{SYN_DIR}/ganA_class4_synth_{sid:03d}.csv"
    df.to_csv(out, index=False)

    made += n
    sid += 1
    print("Saved:", out, "| total:", made)

print("✅ Class4 synthetic saved.")


Saved: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/ganA_synth_shards/ganA_class4_synth_000.csv | total: 25000
Saved: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/ganA_synth_shards/ganA_class4_synth_001.csv | total: 50000
✅ Class4 synthetic saved.


In [ ]:
import tensorflow as tf, glob

LABEL_COL = "label_5class"

def make_synth_ds(glob_pattern, batch_size=512, shuffle=True):
    files = sorted(glob.glob(glob_pattern))
    assert files, f"No synthetic files found for pattern: {glob_pattern}"

    column_defaults = (
        [tf.constant(0.0, tf.float32)] * len(feature_list)
        + [tf.constant(0.0, tf.float32)]
    )

    ds = tf.data.experimental.make_csv_dataset(
        files,
        batch_size=batch_size,
        select_columns=feature_list + [LABEL_COL],
        label_name=LABEL_COL,
        column_defaults=column_defaults,
        num_epochs=1,
        header=True,
        shuffle=shuffle,
        shuffle_buffer_size=5000 if shuffle else 1,
        sloppy=False,
        num_parallel_reads=1
    ).ignore_errors()

    def pack(features, label):
        x = tf.stack([features[c] for c in feature_list], axis=1)
        y = tf.cast(tf.round(label), tf.int32)
        return x, y

    return ds.map(pack, num_parallel_calls=1).prefetch(1)

print("✅ make_synth_ds restored")


✅ make_synth_ds restored


In [ ]:
import glob, tensorflow as tf

SYN_DIR = f"{PROC}/ganA_synth_shards"

# Check shards exist
c1 = sorted(glob.glob(f"{SYN_DIR}/ganA_class1_synth_*.csv"))
c2 = sorted(glob.glob(f"{SYN_DIR}/ganA_class2_synth_*.csv"))
c3 = sorted(glob.glob(f"{SYN_DIR}/ganA_class3_synth_*.csv"))
c4 = sorted(glob.glob(f"{SYN_DIR}/ganA_class4_synth_*.csv"))

print("Shards:", len(c1), len(c2), len(c3), len(c4))
assert len(c1)>0 and len(c2)>0 and len(c3)>0 and len(c4)>0, "Missing some class synth shards."

# One dataset reading all synthetic shards
all_synth_ds = make_synth_ds(f"{SYN_DIR}/ganA_class*_synth_*.csv", batch_size=512, shuffle=True)
print("✅ all_synth_ds ready")


Shards: 2 4 4 2
✅ all_synth_ds ready


In [ ]:
import glob, os

CLEAN_DIR = f"{PROC}/cicids_clean_shards_full"
files = sorted(glob.glob(CLEAN_DIR + "/*.csv"))

print("CLEAN_DIR:", CLEAN_DIR)
print("Total CSV files:", len(files))
print("First 20 files:")
for f in files[:20]:
    print(" -", os.path.basename(f))


CLEAN_DIR: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/cicids_clean_shards_full
Total CSV files: 18
First 20 files:
 - cic_clean_00_000.csv
 - cic_clean_00_001.csv
 - cic_clean_01_000.csv
 - cic_clean_01_001.csv
 - cic_clean_02_000.csv
 - cic_clean_03_000.csv
 - cic_clean_03_001.csv
 - cic_clean_03_002.csv
 - cic_clean_04_000.csv
 - cic_clean_04_001.csv
 - cic_clean_05_000.csv
 - cic_clean_06_000.csv
 - cic_clean_06_001.csv
 - cic_clean_06_002.csv
 - cic_clean_07_000.csv
 - cic_clean_07_001.csv
 - cic_clean_07_002.csv
 - cic_clean_07_003.csv


In [ ]:
import glob, os, tensorflow as tf

LABEL_COL = "label_5class"
CLEAN_DIR = f"{PROC}/cicids_clean_shards_full"
all_files = sorted(glob.glob(CLEAN_DIR + "/*.csv"))
assert len(all_files) == 18, f"Expected 18, got {len(all_files)}"

n = len(all_files)
n_train = 14
n_val   = 2
n_test  = n - n_train - n_val

train_files = all_files[:n_train]
val_files   = all_files[n_train:n_train+n_val]
test_files  = all_files[n_train+n_val:]

print("Split:", len(train_files), len(val_files), len(test_files))
print("Train first/last:", os.path.basename(train_files[0]), os.path.basename(train_files[-1]))
print("Val first/last:", os.path.basename(val_files[0]), os.path.basename(val_files[-1]))
print("Test first/last:", os.path.basename(test_files[0]), os.path.basename(test_files[-1]))

column_defaults = [tf.constant(0.0, tf.float32)] * len(feature_list) + [tf.constant(0.0, tf.float32)]

def make_clean_ds(files, batch_size=512, shuffle=True):
    ds = tf.data.experimental.make_csv_dataset(
        files,
        batch_size=batch_size,
        select_columns=feature_list + [LABEL_COL],
        label_name=LABEL_COL,
        column_defaults=column_defaults,
        num_epochs=1,
        header=True,
        shuffle=shuffle,
        shuffle_buffer_size=5000 if shuffle else 1,
        sloppy=False,
        num_parallel_reads=1
    ).ignore_errors()

    def pack(features, label):
        x = tf.stack([features[c] for c in feature_list], axis=1)
        y = tf.cast(tf.round(label), tf.int32)
        return x, y

    return ds.map(pack, num_parallel_calls=1).prefetch(1)

train_ds = make_clean_ds(train_files, batch_size=512, shuffle=True)
val_ds   = make_clean_ds(val_files, batch_size=512, shuffle=False)
test_ds  = make_clean_ds(test_files, batch_size=512, shuffle=False)

print("✅ train_ds / val_ds / test_ds restored")


Split: 14 2 2
Train first/last: cic_clean_00_000.csv cic_clean_06_002.csv
Val first/last: cic_clean_07_000.csv cic_clean_07_001.csv
Test first/last: cic_clean_07_002.csv cic_clean_07_003.csv
✅ train_ds / val_ds / test_ds restored


In [ ]:
import json, tensorflow as tf

ids = tf.keras.models.load_model(f"{MODELS}/ids_after_ganA_class3.keras")
print("✅ Loaded IDS after class3")

aug_train = tf.data.Dataset.sample_from_datasets(
    [train_ds.repeat(), all_synth_ds.repeat()],
    weights=[0.50, 0.50]
).prefetch(1)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5, min_lr=1e-5),
]

hist_all = ids.fit(
    aug_train,
    validation_data=val_ds.repeat(),
    epochs=10,
    steps_per_epoch=200,
    validation_steps=60,
    callbacks=callbacks,
    verbose=1
)

with open(f"{RESULTS}/ids_after_ganA_all_history.json", "w") as f:
    json.dump(hist_all.history, f, indent=2)

out_path = f"{MODELS}/ids_after_ganA_all.keras"
ids.save(out_path)
print("✅ Saved:", out_path)


✅ Loaded IDS after class3
Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 17s 60ms/step - accuracy: 0.9936 - loss: 0.0424 - val_accuracy: 0.9820 - val_loss: 0.0778 - learning_rate: 1.0000e-05
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 21s 104ms/step - accuracy: 0.9284 - loss: 3.1660 - val_accuracy: 0.9730 - val_loss: 0.1302 - learning_rate: 1.0000e-05
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 12s 58ms/step - accuracy: 0.8458 - loss: 0.6247 - val_accuracy: 0.9650 - val_loss: 0.1791 - learning_rate: 1.0000e-05
✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/models/ids_after_ganA_all.keras


In [ ]:
import tensorflow as tf, glob

LABEL_COL = "label_5class"
FGSM_DIR = f"{PROC}/fgsm_shards"   # this is where your fgsm_eps*.csv live

def make_adv_ds_from_eps(eps, batch_size=512):
    files = sorted(glob.glob(f"{FGSM_DIR}/fgsm_eps{eps}_part*.csv"))
    assert files, f"No FGSM shards found for eps={eps}"

    column_defaults = (
        [tf.constant(0.0, tf.float32)] * len(feature_list)
        + [tf.constant(0.0, tf.float32)]
    )

    ds = tf.data.experimental.make_csv_dataset(
        files,
        batch_size=batch_size,
        select_columns=feature_list + [LABEL_COL],
        label_name=LABEL_COL,
        column_defaults=column_defaults,
        num_epochs=1,
        header=True,
        shuffle=False,
        sloppy=False,
        num_parallel_reads=1
    ).ignore_errors()

    def pack(features, label):
        x = tf.stack([features[c] for c in feature_list], axis=1)
        y = tf.cast(tf.round(label), tf.int32)
        return x, y

    return ds.map(pack, num_parallel_calls=1).prefetch(1)

print("✅ make_adv_ds_from_eps restored")


✅ make_adv_ds_from_eps restored


In [ ]:
import glob, os

# search common places under data_processed
candidates = [
    f"{PROC}/fgsm_test_shards",
    f"{PROC}/fgsm_shards",
    f"{PROC}/fgsm",
    f"{PROC}/adv",
    f"{PROC}"
]

found = []
for d in candidates:
    files = sorted(glob.glob(d + "/fgsm_eps0.01_part*.csv"))
    if files:
        found.append((d, len(files), os.path.basename(files[0]), os.path.basename(files[-1])))

print("Found FGSM locations:")
for item in found:
    print(item)

# fallback brute search (can take a few seconds)
deep = sorted(glob.glob(f"{PROC}/**/fgsm_eps0.01_part*.csv", recursive=True))
print("Deep search hits:", len(deep))
if deep:
    print("Example path:", deep[0])


Found FGSM locations:
('/content/drive/MyDrive/A3IDPS_COLAB/data_processed/fgsm_test_shards', 10, 'fgsm_eps0.01_part0000.csv', 'fgsm_eps0.01_part0009.csv')
Deep search hits: 10
Example path: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/fgsm_test_shards/fgsm_eps0.01_part0000.csv


In [ ]:
FGSM_DIR = f"{PROC}/fgsm_test_shards"
print("FGSM_DIR set to:", FGSM_DIR)
print("eps0.01 count:", len(glob.glob(FGSM_DIR + "/fgsm_eps0.01_part*.csv")))
print("eps0.05 count:", len(glob.glob(FGSM_DIR + "/fgsm_eps0.05_part*.csv")))


FGSM_DIR set to: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/fgsm_test_shards
eps0.01 count: 10
eps0.05 count: 10


In [ ]:
import json, os

clean_loss, clean_acc = ids.evaluate(test_ds, verbose=1)

fgsm01_loss, fgsm01_acc = ids.evaluate(make_adv_ds_from_eps("0.01"), verbose=1)
fgsm05_loss, fgsm05_acc = ids.evaluate(make_adv_ds_from_eps("0.05"), verbose=1)

print("✅ GAN-A ALL | Clean:", clean_acc, "| FGSM0.01:", fgsm01_acc, "| FGSM0.05:", fgsm05_acc)

with open(f"{RESULTS}/ids_after_ganA_all_metrics.json", "w") as f:
    json.dump({
        "clean_acc": float(clean_acc),
        "fgsm_eps0.01_acc": float(fgsm01_acc),
        "fgsm_eps0.05_acc": float(fgsm05_acc),
    }, f, indent=2)

print("✅ Saved:", f"{RESULTS}/ids_after_ganA_all_metrics.json")


572/572 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.9922 - loss: 0.0334
950/950 ━━━━━━━━━━━━━━━━━━━━ 37s 37ms/step - accuracy: 0.4822 - loss: 80574.6875
950/950 ━━━━━━━━━━━━━━━━━━━━ 35s 36ms/step - accuracy: 0.4822 - loss: 402873.0312
✅ GAN-A ALL | Clean: 0.9832375645637512 | FGSM0.01: 0.6344383955001831 | FGSM0.05: 0.6344383955001831
✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/ids_after_ganA_all_metrics.json


In [ ]:
import tensorflow as tf

ids = tf.keras.models.load_model(f"{MODELS}/ids_after_ganA_all.keras")
print("✅ Loaded:", f"{MODELS}/ids_after_ganA_all.keras")


✅ Loaded: /content/drive/MyDrive/A3IDPS_COLAB/models/ids_after_ganA_all.keras


In [ ]:
import tensorflow as tf

@tf.function
def predict_batch(x):
    p = ids(x, training=False)
    return tf.argmax(p, axis=1, output_type=tf.int32)

def misclassified_only(ds):
    # ds yields (x, y)
    ds = ds.unbatch()
    def keep_mis(x, y):
        ypred = predict_batch(tf.expand_dims(x, axis=0))[0]
        return tf.not_equal(ypred, y)
    return ds.filter(keep_mis).batch(512).prefetch(1)


In [ ]:
fgsm01_mis = misclassified_only(make_adv_ds_from_eps("0.01"))
fgsm05_mis = misclassified_only(make_adv_ds_from_eps("0.05"))

# quick sanity: count a few batches
for b in fgsm01_mis.take(1):
    print("FGSM0.01 mis batch:", b[0].shape, b[1].shape)

for b in fgsm05_mis.take(1):
    print("FGSM0.05 mis batch:", b[0].shape, b[1].shape)


FGSM0.01 mis batch: (512, 78) (512,)
FGSM0.05 mis batch: (512, 78) (512,)


In [ ]:
# Combine misclassified streams
hard_ds = tf.data.Dataset.sample_from_datasets(
    [fgsm01_mis.repeat(), fgsm05_mis.repeat()],
    weights=[0.5, 0.5]
).prefetch(1)

# Feedback training mix: 60% clean + 40% hard examples
fb_train = tf.data.Dataset.sample_from_datasets(
    [train_ds.repeat(), hard_ds.repeat()],
    weights=[0.60, 0.40]
).prefetch(1)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5, min_lr=1e-5),
]

hist_fb1 = ids.fit(
    fb_train,
    validation_data=val_ds.repeat(),
    epochs=8,
    steps_per_epoch=200,
    validation_steps=60,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/8
200/200 ━━━━━━━━━━━━━━━━━━━━ 70s 326ms/step - accuracy: 0.1262 - loss: 95547.1562 - val_accuracy: 0.9503 - val_loss: 0.2025 - learning_rate: 1.0000e-05
Epoch 2/8
200/200 ━━━━━━━━━━━━━━━━━━━━ 160s 805ms/step - accuracy: 0.5344 - loss: 27029.4785 - val_accuracy: 0.9284 - val_loss: 0.2781 - learning_rate: 1.0000e-05
Epoch 3/8
200/200 ━━━━━━━━━━━━━━━━━━━━ 56s 279ms/step - accuracy: 0.5849 - loss: 84276.2031 - val_accuracy: 0.9295 - val_loss: 0.3064 - learning_rate: 1.0000e-05


In [ ]:
import json

fb_path = f"{MODELS}/ids_after_feedback_round1.keras"
ids.save(fb_path)
print("✅ Saved:", fb_path)

clean_loss, clean_acc = ids.evaluate(test_ds, verbose=1)
fgsm01_loss, fgsm01_acc = ids.evaluate(make_adv_ds_from_eps("0.01"), verbose=1)
fgsm05_loss, fgsm05_acc = ids.evaluate(make_adv_ds_from_eps("0.05"), verbose=1)

print("✅ Feedback R1 | Clean:", clean_acc, "| FGSM0.01:", fgsm01_acc, "| FGSM0.05:", fgsm05_acc)

with open(f"{RESULTS}/ids_after_feedback_round1_metrics.json", "w") as f:
    json.dump({
        "clean_acc": float(clean_acc),
        "fgsm_eps0.01_acc": float(fgsm01_acc),
        "fgsm_eps0.05_acc": float(fgsm05_acc),
        "model_path": fb_path
    }, f, indent=2)

print("✅ Saved metrics json.")


✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/models/ids_after_feedback_round1.keras
572/572 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.9780 - loss: 0.1411
950/950 ━━━━━━━━━━━━━━━━━━━━ 38s 40ms/step - accuracy: 0.5739 - loss: 16314.8066
950/950 ━━━━━━━━━━━━━━━━━━━━ 34s 35ms/step - accuracy: 0.5739 - loss: 81574.9062
✅ Feedback R1 | Clean: 0.980273425579071 | FGSM0.01: 0.6700776219367981 | FGSM0.05: 0.6700776219367981
✅ Saved metrics json.


In [ ]:
import tensorflow as tf

@tf.function
def predict_one(x):
    p = ids(tf.expand_dims(x, axis=0), training=False)
    return tf.argmax(p, axis=1, output_type=tf.int32)[0]

def misclassified_only(ds):
    ds = ds.unbatch()
    def keep_mis(x, y):
        ypred = predict_one(x)
        return tf.not_equal(ypred, y)
    return ds.filter(keep_mis).batch(512).prefetch(1)

# Mine hard sets with updated ids
clean_mis = misclassified_only(test_ds)                 # clean hard cases
fgsm01_mis = misclassified_only(make_adv_ds_from_eps("0.01"))
fgsm05_mis = misclassified_only(make_adv_ds_from_eps("0.05"))

print("✅ Hard-mining datasets created")


✅ Hard-mining datasets created


In [ ]:
hard_ds = tf.data.Dataset.sample_from_datasets(
    [clean_mis.repeat(), fgsm01_mis.repeat(), fgsm05_mis.repeat()],
    weights=[0.34, 0.33, 0.33]
).prefetch(1)

fb2_train = tf.data.Dataset.sample_from_datasets(
    [train_ds.repeat(), hard_ds.repeat()],
    weights=[0.50, 0.50]
).prefetch(1)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5, min_lr=1e-5),
]

hist_fb2 = ids.fit(
    fb2_train,
    validation_data=val_ds.repeat(),
    epochs=6,
    steps_per_epoch=200,
    validation_steps=60,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 470s 2s/step - accuracy: 0.6312 - loss: 24726.8906 - val_accuracy: 0.9378 - val_loss: 0.2454 - learning_rate: 1.0000e-05
Epoch 2/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 395s 2s/step - accuracy: 0.5735 - loss: 62605.0703 - val_accuracy: 0.9354 - val_loss: 0.2597 - learning_rate: 1.0000e-05
Epoch 3/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 539s 3s/step - accuracy: 0.4790 - loss: 75220.5625 - val_accuracy: 0.9418 - val_loss: 0.2443 - learning_rate: 1.0000e-05
Epoch 4/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 477s 2s/step - accuracy: 0.6022 - loss: 105318.5703 - val_accuracy: 0.9480 - val_loss: 0.2331 - learning_rate: 1.0000e-05
Epoch 5/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 733s 4s/step - accuracy: 0.6101 - loss: 58638.5391 - val_accuracy: 0.9620 - val_loss: 0.1961 - learning_rate: 1.0000e-05
Epoch 6/6
200/200 ━━━━━━━━━━━━━━━━━━━━ 684s 3s/step - accuracy: 0.6459 - loss: 17485.3730 - val_accuracy: 0.9511 - val_loss: 0.2252 - learning_rate: 1.0000e-05


In [ ]:
import json

fb2_path = f"{MODELS}/ids_after_feedback_round2.keras"
ids.save(fb2_path)
print("✅ Saved:", fb2_path)

clean_loss, clean_acc = ids.evaluate(test_ds, verbose=1)
fgsm01_loss, fgsm01_acc = ids.evaluate(make_adv_ds_from_eps("0.01"), verbose=1)
fgsm05_loss, fgsm05_acc = ids.evaluate(make_adv_ds_from_eps("0.05"), verbose=1)

print("✅ Feedback R2 | Clean:", clean_acc, "| FGSM0.01:", fgsm01_acc, "| FGSM0.05:", fgsm05_acc)

with open(f"{RESULTS}/ids_after_feedback_round2_metrics.json", "w") as f:
    json.dump({
        "clean_acc": float(clean_acc),
        "fgsm_eps0.01_acc": float(fgsm01_acc),
        "fgsm_eps0.05_acc": float(fgsm05_acc),
        "model_path": fb2_path
    }, f, indent=2)

print("✅ Saved metrics json.")


✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/models/ids_after_feedback_round2.keras
572/572 ━━━━━━━━━━━━━━━━━━━━ 22s 38ms/step - accuracy: 0.9858 - loss: 0.1375
950/950 ━━━━━━━━━━━━━━━━━━━━ 32s 33ms/step - accuracy: 0.7490 - loss: 4530.2305
950/950 ━━━━━━━━━━━━━━━━━━━━ 31s 33ms/step - accuracy: 0.7490 - loss: 22655.0566
✅ Feedback R2 | Clean: 0.9858118891716003 | FGSM0.01: 0.8110280632972717 | FGSM0.05: 0.8110280632972717
✅ Saved metrics json.


In [ ]:
import json, shutil, os

FINAL_DIR = f"{BASE}/final_artifacts_cicids"
os.makedirs(FINAL_DIR, exist_ok=True)

# Copy model
final_model_path = f"{FINAL_DIR}/ids_final_cicids_feedbackR2.keras"
ids.save(final_model_path)

# Copy key metrics jsons (if they exist)
to_copy = [
    f"{RESULTS}/ids_after_ganA_all_metrics.json",
    f"{RESULTS}/ids_after_feedback_round1_metrics.json",
    f"{RESULTS}/ids_after_feedback_round2_metrics.json",
]
for p in to_copy:
    if os.path.exists(p):
        shutil.copy(p, FINAL_DIR)

# Save summary
summary = {
    "final_model": final_model_path,
    "final_clean_acc": 0.9858118891716003,
    "final_fgsm_eps0.01_acc": 0.8110280632972717,
    "final_fgsm_eps0.05_acc": 0.8110280632972717,
}
with open(f"{FINAL_DIR}/FINAL_SUMMARY.json", "w") as f:
    json.dump(summary, f, indent=2)

print("✅ Final CICIDS bundle saved at:", FINAL_DIR)
print("✅ Model:", final_model_path)


✅ Final CICIDS bundle saved at: /content/drive/MyDrive/A3IDPS_COLAB/final_artifacts_cicids
✅ Model: /content/drive/MyDrive/A3IDPS_COLAB/final_artifacts_cicids/ids_final_cicids_feedbackR2.keras


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, glob, pandas as pd

BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
RAW  = f"{BASE}/data_raw"
NSL_ROOT_CANDIDATES = [
    f"{RAW}/nslkdd",
    f"{RAW}/NSL-KDD",
    f"{RAW}/NSLKDD",
    f"{BASE}/nslkdd",
    f"{BASE}/data/nslkdd"
]

found = []
for d in NSL_ROOT_CANDIDATES:
    if os.path.isdir(d):
        trainp = os.path.join(d, "KDDTrain+.txt")
        testp  = os.path.join(d, "KDDTest+.txt")
        found.append((d, os.path.exists(trainp), os.path.exists(testp)))

print("Candidates checked:")
for item in found:
    print(item)

# brute search if needed
hits = glob.glob(f"{BASE}/**/KDDTrain+.txt", recursive=True)
print("Deep search train hits:", len(hits))
if hits:
    print("Example:", hits[0])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Candidates checked:
('/content/drive/MyDrive/A3IDPS_COLAB/data_raw/NSL-KDD', True, True)
Deep search train hits: 2
Example: /content/drive/MyDrive/A3IDPS_COLAB/data_raw/NSL-KDD/KDDTrain+.txt


In [ ]:
import pandas as pd, os

NSL_DIR = "/content/drive/MyDrive/A3IDPS_COLAB/data_raw/NSL-KDD"
train_path = os.path.join(NSL_DIR, "KDDTrain+.txt")
test_path  = os.path.join(NSL_DIR, "KDDTest+.txt")

# Standard NSL-KDD column names (41 features + label + difficulty)
nsl_cols = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land","wrong_fragment",
    "urgent","hot","num_failed_logins","logged_in","num_compromised","root_shell","su_attempted",
    "num_root","num_file_creations","num_shells","num_access_files","num_outbound_cmds",
    "is_host_login","is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate",
    "dst_host_count","dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

df_tr = pd.read_csv(train_path, header=None, names=nsl_cols)
df_te = pd.read_csv(test_path, header=None, names=nsl_cols)

print("Train shape:", df_tr.shape)
print("Test shape :", df_te.shape)
print("Label head counts (train):")
print(df_tr["label"].value_counts().head(15))
print("\nCategorical sample:", df_tr[["protocol_type","service","flag"]].head())


Train shape: (125973, 43)
Test shape : (22544, 43)
Label head counts (train):
label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
Name: count, dtype: int64

Categorical sample:   protocol_type   service flag
0           tcp  ftp_data   SF
1           udp     other   SF
2           tcp   private   S0
3           tcp      http   SF
4           tcp      http   SF


In [ ]:
# NSL-KDD attack group mapping
dos = {
    "back","land","neptune","pod","smurf","teardrop","apache2","udpstorm","processtable","worm"
}
probe = {
    "satan","ipsweep","nmap","portsweep","mscan","saint"
}
r2l = {
    "guess_passwd","ftp_write","imap","phf","multihop","warezmaster","warezclient","spy",
    "xlock","xsnoop","snmpguess","snmpgetattack","httptunnel","sendmail","named"
}
u2r = {
    "buffer_overflow","loadmodule","rootkit","perl","sqlattack","xterm","ps"
}

def to_5class(lbl):
    if lbl == "normal":
        return 0  # Normal
    if lbl in dos:
        return 1  # DoS
    if lbl in probe:
        return 2  # Probe
    if lbl in r2l:
        return 3  # R2L
    if lbl in u2r:
        return 4  # U2R
    return 4  # unknown -> safest bucket (rare), or change to 2 if you prefer

df_tr["label_5class"] = df_tr["label"].apply(to_5class).astype("int32")
df_te["label_5class"] = df_te["label"].apply(to_5class).astype("int32")

print("Train 5-class counts:")
print(df_tr["label_5class"].value_counts().sort_index())
print("\nTest 5-class counts:")
print(df_te["label_5class"].value_counts().sort_index())


Train 5-class counts:
label_5class
0    67343
1    45927
2    11656
3      995
4       52
Name: count, dtype: int64

Test 5-class counts:
label_5class
0    9711
1    7167
2    2421
3    2885
4     360
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np, json, os

# Split train into train/val (stratified)
train_df, val_df = train_test_split(
    df_tr,
    test_size=0.10,
    random_state=42,
    stratify=df_tr["label_5class"]
)

cat_cols = ["protocol_type","service","flag"]
drop_cols = ["label","difficulty"]  # keep label_5class

def encode_fit_transform(train_df, val_df, test_df):
    # one-hot based on train categories only (consistent columns)
    X_tr = pd.get_dummies(train_df.drop(columns=drop_cols), columns=cat_cols)
    X_va = pd.get_dummies(val_df.drop(columns=drop_cols), columns=cat_cols)
    X_te = pd.get_dummies(test_df.drop(columns=drop_cols), columns=cat_cols)

    # align columns
    X_va = X_va.reindex(columns=X_tr.columns, fill_value=0)
    X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

    return X_tr, X_va, X_te

X_tr, X_va, X_te = encode_fit_transform(train_df, val_df, df_te)

y_tr = train_df["label_5class"].values.astype("int32")
y_va = val_df["label_5class"].values.astype("int32")
y_te = df_te["label_5class"].values.astype("int32")

# Scale (fit on train only)
scaler = StandardScaler(with_mean=True, with_std=True)
X_tr_scaled = scaler.fit_transform(X_tr.values).astype("float32")
X_va_scaled = scaler.transform(X_va.values).astype("float32")
X_te_scaled = scaler.transform(X_te.values).astype("float32")

print("NSL feature dim:", X_tr_scaled.shape[1])
print("Train/Val/Test:", X_tr_scaled.shape, X_va_scaled.shape, X_te_scaled.shape)


NSL feature dim: 123
Train/Val/Test: (113375, 123) (12598, 123) (22544, 123)


In [ ]:
NSL_PROC = f"{BASE}/data_processed/nslkdd"
os.makedirs(NSL_PROC, exist_ok=True)

# Save feature list (one-hot columns)
nsl_feature_list = list(X_tr.columns)
with open(f"{NSL_PROC}/nsl_feature_list.json", "w") as f:
    json.dump(nsl_feature_list, f, indent=2)

# Save scaler
import joblib
joblib.dump(scaler, f"{NSL_PROC}/nsl_scaler.joblib")

# Save numpy arrays (fast, avoids csv RAM issues)
np.save(f"{NSL_PROC}/X_train.npy", X_tr_scaled)
np.save(f"{NSL_PROC}/y_train.npy", y_tr)
np.save(f"{NSL_PROC}/X_val.npy", X_va_scaled)
np.save(f"{NSL_PROC}/y_val.npy", y_va)
np.save(f"{NSL_PROC}/X_test.npy", X_te_scaled)
np.save(f"{NSL_PROC}/y_test.npy", y_te)

print("✅ Saved NSL artifacts to:", NSL_PROC)


✅ Saved NSL artifacts to: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/nslkdd


In [ ]:
# Approach A: Teacher-Student / Adapter (NSL-KDD)

In [ ]:
import numpy as np, tensorflow as tf, os

NSL_PROC = f"{BASE}/data_processed/nslkdd"
X_tr = np.load(f"{NSL_PROC}/X_train.npy").astype("float32")
y_tr = np.load(f"{NSL_PROC}/y_train.npy").astype("int32")
X_va = np.load(f"{NSL_PROC}/X_val.npy").astype("float32")
y_va = np.load(f"{NSL_PROC}/y_val.npy").astype("int32")
X_te = np.load(f"{NSL_PROC}/X_test.npy").astype("float32")
y_te = np.load(f"{NSL_PROC}/y_test.npy").astype("int32")

print("Loaded:", X_tr.shape, X_va.shape, X_te.shape)

BATCH = 512
train_ds = tf.data.Dataset.from_tensor_slices((X_tr, y_tr)).shuffle(50000).batch(BATCH).prefetch(1)
val_ds   = tf.data.Dataset.from_tensor_slices((X_va, y_va)).batch(BATCH).prefetch(1)
test_ds  = tf.data.Dataset.from_tensor_slices((X_te, y_te)).batch(BATCH).prefetch(1)


Loaded: (113375, 123) (12598, 123) (22544, 123)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([0,1,2,3,4])
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
class_weight = {int(c): float(w) for c, w in zip(classes, cw)}
print("Class weights:", class_weight)


Class weights: {0: 0.37412552798310456, 1: 0.5485798616151353, 2: 2.161582459485224, 3: 25.306919642857142, 4: 482.4468085106383}


In [ ]:
import tensorflow as tf

D = X_tr.shape[1]

def make_student():
    inp = tf.keras.Input(shape=(D,))
    x = tf.keras.layers.Dense(256, activation="relu")(inp)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    out = tf.keras.layers.Dense(5, activation="softmax")(x)
    model = tf.keras.Model(inp, out, name="nsl_student_baseline")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(5e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

student_base = make_student()
student_base.summary()

hist = student_base.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5, min_lr=1e-5),
    ],
    class_weight=class_weight,
    verbose=1
)

base_path = f"{MODELS}/nsl_student_baseline.keras"
student_base.save(base_path)
print("✅ Saved:", base_path)


Model: "nsl_student_baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 123)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │        31,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 98,821 (386.02 KB)

 Trainable params: 98,821 (386.02 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/12
222/222 ━━━━━━━━━━━━━━━━━━━━ 12s 24ms/step - accuracy: 0.8409 - loss: 0.8909 - val_accuracy: 0.9798 - val_loss: 0.0826 - learning_rate: 5.0000e-04
Epoch 2/12
222/222 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9800 - loss: 0.1857 - val_accuracy: 0.9831 - val_loss: 0.0529 - learning_rate: 5.0000e-04
Epoch 3/12
222/222 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9850 - loss: 0.1001 - val_accuracy: 0.9941 - val_loss: 0.0273 - learning_rate: 5.0000e-04
Epoch 4/12
222/222 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9911 - loss: 0.1192 - val_accuracy: 0.9968 - val_loss: 0.0180 - learning_rate: 5.0000e-04
Epoch 5/12
222/222 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9933 - loss: 0.0609 - val_accuracy: 0.9970 - val_loss: 0.0261 - learning_rate: 5.0000e-04
Epoch 6/12
222/222 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9956 - loss: 0.0487 - val_accuracy: 0.9974 - val_loss: 0.0198 - learning_rate: 2.5000e-04
✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/models/nsl_studen

In [ ]:
from sklearn.metrics import classification_report, f1_score
import numpy as np

# predict
y_pred = np.argmax(student_base.predict(test_ds, verbose=1), axis=1)

print("Macro-F1:", f1_score(y_te, y_pred, average="macro"))
print(classification_report(y_te, y_pred, digits=4))


45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
Macro-F1: 0.7308806170416435
              precision    recall  f1-score   support

           0     0.9294    0.9750    0.9517      9711
           1     0.8590    0.8973    0.8777      7167
           2     0.6592    0.7662    0.7087      2421
           3     0.8771    0.3785    0.5288      2885
           4     0.4242    0.9556    0.5875       360

    accuracy                         0.8512     22544
   macro avg     0.7498    0.7945    0.7309     22544
weighted avg     0.8632    0.8512    0.8421     22544



In [ ]:
import json
from sklearn.metrics import confusion_matrix

rep = classification_report(y_te, y_pred, digits=4, output_dict=True)
out = {
    "clean_acc": float(rep["accuracy"]),
    "macro_f1": float(rep["macro avg"]["f1-score"]),
    "report": rep,
    "confusion_matrix": confusion_matrix(y_te, y_pred).tolist()
}
with open(f"{RESULTS}/nsl_student_baseline_clean_report.json", "w") as f:
    json.dump(out, f, indent=2)
print("✅ Saved report json")


✅ Saved report json


In [ ]:
import tensorflow as tf
import numpy as np

student_ts = make_student()
student_ts._name = "nsl_student_teacher_inspired"

# hyperparams
sigma = 0.01        # noise std
lam_kl = 1.0        # consistency weight
lam_ent = 0.01      # confidence shaping (lower entropy)
epochs = 8

# prepare class weights tensor
cw_vec = np.array([class_weight[i] for i in range(5)], dtype=np.float32)
cw_tf = tf.constant(cw_vec)

opt = tf.keras.optimizers.Adam(5e-4)

@tf.function
def train_step(x, y):
    x2 = x + tf.random.normal(tf.shape(x), stddev=sigma)
    with tf.GradientTape() as tape:
        p1 = student_ts(x, training=True)
        p2 = student_ts(x2, training=True)

        # weighted CE
        ce = tf.keras.losses.sparse_categorical_crossentropy(y, p1)
        w = tf.gather(cw_tf, y)
        ce = tf.reduce_mean(ce * w)

        # KL consistency
        kl = tf.reduce_mean(tf.reduce_sum(p1 * (tf.math.log(p1 + 1e-8) - tf.math.log(p2 + 1e-8)), axis=1))

        # entropy penalty (encourage confident predictions)
        ent = tf.reduce_mean(-tf.reduce_sum(p1 * tf.math.log(p1 + 1e-8), axis=1))

        loss = ce + lam_kl * kl + lam_ent * ent

    grads = tape.gradient(loss, student_ts.trainable_variables)
    opt.apply_gradients(zip(grads, student_ts.trainable_variables))
    return loss, ce, kl, ent

@tf.function
def val_step(x, y):
    p = student_ts(x, training=False)
    ce = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(y, p))
    acc = tf.reduce_mean(tf.cast(tf.equal(tf.argmax(p, axis=1, output_type=tf.int32), y), tf.float32))
    return ce, acc

# training loop
for ep in range(1, epochs+1):
    tl, tce, tkl, tent, nb = 0,0,0,0,0
    for x, y in train_ds:
        loss, ce, kl, ent = train_step(x, y)
        tl += loss.numpy(); tce += ce.numpy(); tkl += kl.numpy(); tent += ent.numpy(); nb += 1

    vce, vacc, vb = 0,0,0
    for x, y in val_ds:
        ce, acc = val_step(x, y)
        vce += ce.numpy(); vacc += acc.numpy(); vb += 1

    print(f"Epoch {ep}/{epochs} | loss={tl/nb:.4f} ce={tce/nb:.4f} kl={tkl/nb:.4f} ent={tent/nb:.4f} | val_ce={vce/vb:.4f} val_acc={vacc/vb:.4f}")

ts_path = f"{MODELS}/nsl_student_teacher_inspired.keras"
student_ts.save(ts_path)
print("✅ Saved:", ts_path)


Epoch 1/8 | loss=0.5941 ce=0.5421 kl=0.0475 ent=0.4513 | val_ce=0.0912 val_acc=0.9764
Epoch 2/8 | loss=0.1758 ce=0.1521 kl=0.0229 ent=0.0798 | val_ce=0.0498 val_acc=0.9863
Epoch 3/8 | loss=0.1554 ce=0.1386 kl=0.0164 ent=0.0484 | val_ce=0.0353 val_acc=0.9882
Epoch 4/8 | loss=0.1511 ce=0.1343 kl=0.0164 ent=0.0388 | val_ce=0.0386 val_acc=0.9887
Epoch 5/8 | loss=0.0625 ce=0.0499 kl=0.0123 ent=0.0279 | val_ce=0.0243 val_acc=0.9925
Epoch 6/8 | loss=0.0537 ce=0.0430 kl=0.0105 ent=0.0234 | val_ce=0.0264 val_acc=0.9969
Epoch 7/8 | loss=0.0553 ce=0.0456 kl=0.0095 ent=0.0186 | val_ce=0.0216 val_acc=0.9975
Epoch 8/8 | loss=0.0610 ce=0.0520 kl=0.0088 ent=0.0151 | val_ce=0.0170 val_acc=0.9978
✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/models/nsl_student_teacher_inspired.keras


In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import numpy as np, json

y_pred_ts = np.argmax(student_ts.predict(test_ds, verbose=1), axis=1)

print("Macro-F1:", f1_score(y_te, y_pred_ts, average="macro"))
print(classification_report(y_te, y_pred_ts, digits=4))

rep = classification_report(y_te, y_pred_ts, digits=4, output_dict=True)
out = {
    "clean_acc": float(rep["accuracy"]),
    "macro_f1": float(rep["macro avg"]["f1-score"]),
    "report": rep,
    "confusion_matrix": confusion_matrix(y_te, y_pred_ts).tolist(),
    "sigma": sigma, "lam_kl": lam_kl, "lam_ent": lam_ent
}
with open(f"{RESULTS}/nsl_student_teacher_inspired_clean_report.json", "w") as f:
    json.dump(out, f, indent=2)

print("✅ Saved teacher-inspired report json")


45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
Macro-F1: 0.8367240119876034
              precision    recall  f1-score   support

           0     0.9215    0.9815    0.9505      9711
           1     0.8702    0.8967    0.8833      7167
           2     0.7113    0.8600    0.7786      2421
           3     0.9566    0.4967    0.6539      2885
           4     0.8821    0.9556    0.9173       360

    accuracy                         0.8790     22544
   macro avg     0.8683    0.8381    0.8367     22544
weighted avg     0.8865    0.8790    0.8722     22544

✅ Saved teacher-inspired report json


In [ ]:
# fgsm attack generator for nsl

In [ ]:
import tensorflow as tf
import numpy as np

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

@tf.function
def fgsm_batch(model, x, y, eps):
    x = tf.identity(x)
    with tf.GradientTape() as tape:
        tape.watch(x)
        p = model(x, training=False)
        loss = loss_fn(y, p)
    grad = tape.gradient(loss, x)
    x_adv = x + eps * tf.sign(grad)
    return x_adv

def eval_fgsm(model, eps):
    adv_acc = tf.keras.metrics.SparseCategoricalAccuracy()
    for x, y in test_ds:
        x_adv = fgsm_batch(model, x, y, eps)
        p_adv = model(x_adv, training=False)
        adv_acc.update_state(y, p_adv)
    return float(adv_acc.result().numpy())

for eps in [0.01, 0.05, 0.1]:
    b = eval_fgsm(student_base, eps)
    t = eval_fgsm(student_ts, eps)
    print(f"eps={eps} | baseline_adv_acc={b:.4f} | teacher_inspired_adv_acc={t:.4f}")


eps=0.01 | baseline_adv_acc=0.8496 | teacher_inspired_adv_acc=0.8770
eps=0.05 | baseline_adv_acc=0.8217 | teacher_inspired_adv_acc=0.8251
eps=0.1 | baseline_adv_acc=0.7356 | teacher_inspired_adv_acc=0.7579


In [ ]:
# Approach B: class wise GAN for NSL

In [ ]:
import tensorflow as tf, numpy as np

BATCH_GAN = 512
LATENT = 64
D = X_tr.shape[1]  # 123

# helper: dataset for a specific attack class (only from TRAIN split)
def class_ds(X, y, cls, batch=512, shuffle=True):
    idx = np.where(y == cls)[0]
    Xc = X[idx].astype("float32")
    ds = tf.data.Dataset.from_tensor_slices(Xc)
    if shuffle:
        ds = ds.shuffle(min(len(Xc), 50000))
    return ds.batch(batch, drop_remainder=True).prefetch(1), Xc

ds1, X1 = class_ds(X_tr, y_tr, 1, batch=BATCH_GAN)  # DoS
ds2, X2 = class_ds(X_tr, y_tr, 2, batch=BATCH_GAN)  # Probe
ds3, X3 = class_ds(X_tr, y_tr, 3, batch=BATCH_GAN)  # R2L
ds4, X4 = class_ds(X_tr, y_tr, 4, batch=BATCH_GAN)  # U2R (tiny!)

print("Class sizes (train):",
      "DoS", X1.shape[0],
      "Probe", X2.shape[0],
      "R2L", X3.shape[0],
      "U2R", X4.shape[0])


Class sizes (train): DoS 41334 Probe 10490 R2L 896 U2R 47


In [ ]:
import tensorflow as tf

# CPU quick + stable settings for NSL
TOTAL_STEPS = 800
LOG_EVERY = 100
LAMBDA_GP = 5.0
CRITIC_STEPS = 1

def make_generator_nsl(name):
    z = tf.keras.Input(shape=(LATENT,))
    x = tf.keras.layers.Dense(128, activation="relu")(z)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dense(D, activation="tanh")(x)  # outputs in [-1,1]
    return tf.keras.Model(z, x, name=name)

def make_critic_nsl(name):
    x = tf.keras.Input(shape=(D,))
    h = tf.keras.layers.Dense(128)(x); h = tf.keras.layers.LeakyReLU(0.2)(h)
    h = tf.keras.layers.Dense(128)(h); h = tf.keras.layers.LeakyReLU(0.2)(h)
    out = tf.keras.layers.Dense(1)(h)
    return tf.keras.Model(x, out, name=name)

def scale_to_match(train_class_X, fake_tanh):
    # Map [-1,1] to [min,max] per feature using class bounds
    fmin = tf.reduce_min(train_class_X, axis=0)
    fmax = tf.reduce_max(train_class_X, axis=0)
    z01 = (fake_tanh + 1.0) / 2.0
    return fmin + z01 * (fmax - fmin)

def build_gan_for_class(class_id, Xc):
    # Xc is numpy array of that class
    Xc_tf = tf.constant(Xc[:5000], dtype=tf.float32) if Xc.shape[0] > 5000 else tf.constant(Xc, tf.float32)

    G_core = make_generator_nsl(f"nsl_ganA_core_class{class_id}")
    z = tf.keras.Input(shape=(LATENT,))
    raw = G_core(z)
    out = tf.keras.layers.Lambda(lambda t: scale_to_match(Xc_tf, t))(raw)
    G = tf.keras.Model(z, out, name=f"nsl_ganA_gen_class{class_id}")

    C = make_critic_nsl(f"nsl_ganA_critic_class{class_id}")

    G_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)
    C_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9, clipnorm=1.0)

    return G, C, G_opt, C_opt

@tf.function
def gradient_penalty(C, real, fake):
    alpha = tf.random.uniform([tf.shape(real)[0], 1], 0.0, 1.0)
    interp = real + alpha * (fake - real)
    with tf.GradientTape() as tape:
        tape.watch(interp)
        pred = C(interp, training=True)
    grads = tape.gradient(pred, interp)
    slopes = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    return tf.reduce_mean((slopes - 1.0) ** 2)

@tf.function
def train_critic(C, G, C_opt, real):
    z = tf.random.normal([tf.shape(real)[0], LATENT])
    with tf.GradientTape() as tape:
        fake = G(z, training=True)
        c_real = C(real, training=True)
        c_fake = C(fake, training=True)
        gp = gradient_penalty(C, real, fake)
        loss_c = tf.reduce_mean(c_fake) - tf.reduce_mean(c_real) + LAMBDA_GP * gp
    grads = tape.gradient(loss_c, C.trainable_variables)
    C_opt.apply_gradients(zip(grads, C.trainable_variables))
    return loss_c, gp

@tf.function
def train_gen(C, G, G_opt, bs):
    z = tf.random.normal([bs, LATENT])
    with tf.GradientTape() as tape:
        fake = G(z, training=True)
        c_fake = C(fake, training=True)
        loss_g = -tf.reduce_mean(c_fake)
    grads = tape.gradient(loss_g, G.trainable_variables)
    G_opt.apply_gradients(zip(grads, G.trainable_variables))
    return loss_g

print("✅ NSL GAN-A builders ready")


✅ NSL GAN-A builders ready


In [ ]:
# class 1 DoS

In [ ]:
import time, numpy as np

class_id = 1
ds = ds1
Xc = X1

G, C, G_opt, C_opt = build_gan_for_class(class_id, Xc)

it = iter(ds.repeat())
t0 = time.time()

for step in range(1, TOTAL_STEPS + 1):
    real = next(it)
    lc, gp = train_critic(C, G, C_opt, real)
    lg = train_gen(C, G, G_opt, tf.shape(real)[0])

    if step % LOG_EVERY == 0 or step == TOTAL_STEPS:
        print(f"[class{class_id} {step}/{TOTAL_STEPS}] C={float(lc.numpy()):.4f} G={float(lg.numpy()):.4f} GP={float(gp.numpy()):.4f} dt={time.time()-t0:.1f}s")
        t0 = time.time()

# Save generator
nsl_gan_dir = f"{BASE}/models/nsl_ganA"
os.makedirs(nsl_gan_dir, exist_ok=True)
G.save(f"{nsl_gan_dir}/ganA_class{class_id}.keras")
print("✅ Saved GAN:", f"{nsl_gan_dir}/ganA_class{class_id}.keras")

# Generate synthetic (keep reasonable)
TOTAL_SYN = 50_000
z = tf.random.normal([TOTAL_SYN, LATENT])
X_syn = G(z, training=False).numpy().astype("float32")
y_syn = np.full((TOTAL_SYN,), class_id, dtype=np.int32)

np.save(f"{NSL_PROC}/X_syn_class{class_id}.npy", X_syn)
np.save(f"{NSL_PROC}/y_syn_class{class_id}.npy", y_syn)
print("✅ Saved synthetic:", X_syn.shape)


[class1 100/800] C=-29.4893 G=30.6972 GP=0.1813 dt=4.0s
[class1 200/800] C=-15.0580 G=16.5468 GP=0.7774 dt=1.0s
[class1 300/800] C=-3.0693 G=0.2675 GP=0.1186 dt=1.0s
[class1 400/800] C=-2.8841 G=-1.4219 GP=0.0611 dt=1.1s
[class1 500/800] C=-5.5562 G=0.9557 GP=0.2047 dt=1.0s
[class1 600/800] C=-5.5669 G=-1.1393 GP=0.2173 dt=1.2s
[class1 700/800] C=-4.7315 G=-1.9184 GP=0.3287 dt=1.8s
[class1 800/800] C=-4.5286 G=-1.5229 GP=0.3545 dt=1.4s
✅ Saved GAN: /content/drive/MyDrive/A3IDPS_COLAB/models/nsl_ganA/ganA_class1.keras
✅ Saved synthetic: (50000, 123)


In [ ]:
# NSL GAN-A — Class 2 (Probe)

In [ ]:
import tensorflow as tf, time, numpy as np, os

# 1) clear TF graph state (important)
tf.keras.backend.clear_session()

class_id = 2
ds = ds2
Xc = X2

# 2) rebuild GAN fresh for this class
G, C, G_opt, C_opt = build_gan_for_class(class_id, Xc)

# 3) define LOCAL (non-decorated) steps to avoid tf.function retrace/slot issues
def gradient_penalty_local(C, real, fake):
    alpha = tf.random.uniform([tf.shape(real)[0], 1], 0.0, 1.0)
    interp = real + alpha * (fake - real)
    with tf.GradientTape() as tape:
        tape.watch(interp)
        pred = C(interp, training=True)
    grads = tape.gradient(pred, interp)
    slopes = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    return tf.reduce_mean((slopes - 1.0) ** 2)

def train_critic_local(real):
    z = tf.random.normal([tf.shape(real)[0], LATENT])
    with tf.GradientTape() as tape:
        fake = G(z, training=True)
        c_real = C(real, training=True)
        c_fake = C(fake, training=True)
        gp = gradient_penalty_local(C, real, fake)
        loss_c = tf.reduce_mean(c_fake) - tf.reduce_mean(c_real) + LAMBDA_GP * gp
    grads = tape.gradient(loss_c, C.trainable_variables)
    C_opt.apply_gradients(zip(grads, C.trainable_variables))
    return loss_c, gp

def train_gen_local(bs):
    z = tf.random.normal([bs, LATENT])
    with tf.GradientTape() as tape:
        fake = G(z, training=True)
        c_fake = C(fake, training=True)
        loss_g = -tf.reduce_mean(c_fake)
    grads = tape.gradient(loss_g, G.trainable_variables)
    G_opt.apply_gradients(zip(grads, G.trainable_variables))
    return loss_g

# 4) train loop
it = iter(ds.repeat())
t0 = time.time()

for step in range(1, TOTAL_STEPS + 1):
    real = next(it)
    lc, gp = train_critic_local(real)
    lg = train_gen_local(real.shape[0])

    if step % LOG_EVERY == 0 or step == TOTAL_STEPS:
        print(f"[class{class_id} {step}/{TOTAL_STEPS}] C={float(lc.numpy()):.4f} G={float(lg.numpy()):.4f} GP={float(gp.numpy()):.4f} dt={time.time()-t0:.1f}s")
        t0 = time.time()

# 5) save generator
nsl_gan_dir = f"{BASE}/models/nsl_ganA"
os.makedirs(nsl_gan_dir, exist_ok=True)
G.save(f"{nsl_gan_dir}/ganA_class{class_id}.keras")
print("✅ Saved GAN:", f"{nsl_gan_dir}/ganA_class{class_id}.keras")

# 6) generate synthetic
TOTAL_SYN = 40_000
z = tf.random.normal([TOTAL_SYN, LATENT])
X_syn = G(z, training=False).numpy().astype("float32")
y_syn = np.full((TOTAL_SYN,), class_id, dtype=np.int32)

np.save(f"{NSL_PROC}/X_syn_class{class_id}.npy", X_syn)
np.save(f"{NSL_PROC}/y_syn_class{class_id}.npy", y_syn)
print("✅ Saved synthetic:", X_syn.shape)


[class2 100/800] C=-65.5819 G=66.2767 GP=0.2918 dt=11.7s
[class2 200/800] C=-33.3437 G=42.0883 GP=2.7945 dt=11.5s
[class2 300/800] C=-4.4006 G=2.3494 GP=0.4808 dt=11.6s
[class2 400/800] C=-4.4055 G=0.6867 GP=0.1084 dt=10.9s
[class2 500/800] C=-2.2960 G=-2.3057 GP=0.0819 dt=11.3s
[class2 600/800] C=-5.8868 G=0.6124 GP=0.2834 dt=11.5s
[class2 700/800] C=-4.8879 G=-1.3605 GP=0.1876 dt=11.5s
[class2 800/800] C=-6.5690 G=-0.0091 GP=0.3208 dt=11.5s
✅ Saved GAN: /content/drive/MyDrive/A3IDPS_COLAB/models/nsl_ganA/ganA_class2.keras
✅ Saved synthetic: (40000, 123)


In [ ]:
# NSL GAN A - Class 3 (R2L)

In [ ]:
import tensorflow as tf, time, numpy as np, os

tf.keras.backend.clear_session()

class_id = 3
ds = ds3
Xc = X3

G, C, G_opt, C_opt = build_gan_for_class(class_id, Xc)

def gradient_penalty_local(C, real, fake):
    alpha = tf.random.uniform([tf.shape(real)[0], 1], 0.0, 1.0)
    interp = real + alpha * (fake - real)
    with tf.GradientTape() as tape:
        tape.watch(interp)
        pred = C(interp, training=True)
    grads = tape.gradient(pred, interp)
    slopes = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    return tf.reduce_mean((slopes - 1.0) ** 2)

def train_critic_local(real):
    z = tf.random.normal([tf.shape(real)[0], LATENT])
    with tf.GradientTape() as tape:
        fake = G(z, training=True)
        c_real = C(real, training=True)
        c_fake = C(fake, training=True)
        gp = gradient_penalty_local(C, real, fake)
        loss_c = tf.reduce_mean(c_fake) - tf.reduce_mean(c_real) + LAMBDA_GP * gp
    grads = tape.gradient(loss_c, C.trainable_variables)
    C_opt.apply_gradients(zip(grads, C.trainable_variables))
    return loss_c, gp

def train_gen_local(bs):
    z = tf.random.normal([bs, LATENT])
    with tf.GradientTape() as tape:
        fake = G(z, training=True)
        c_fake = C(fake, training=True)
        loss_g = -tf.reduce_mean(c_fake)
    grads = tape.gradient(loss_g, G.trainable_variables)
    G_opt.apply_gradients(zip(grads, G.trainable_variables))
    return loss_g

it = iter(ds.repeat())
t0 = time.time()

for step in range(1, TOTAL_STEPS + 1):
    real = next(it)
    lc, gp = train_critic_local(real)
    lg = train_gen_local(real.shape[0])

    if step % LOG_EVERY == 0 or step == TOTAL_STEPS:
        print(f"[class{class_id} {step}/{TOTAL_STEPS}] C={float(lc.numpy()):.4f} G={float(lg.numpy()):.4f} GP={float(gp.numpy()):.4f} dt={time.time()-t0:.1f}s")
        t0 = time.time()

# save generator
nsl_gan_dir = f"{BASE}/models/nsl_ganA"
os.makedirs(nsl_gan_dir, exist_ok=True)
G.save(f"{nsl_gan_dir}/ganA_class{class_id}.keras")
print("✅ Saved GAN:", f"{nsl_gan_dir}/ganA_class{class_id}.keras")

# generate synthetic (R2L is small-ish; moderate synth helps)
TOTAL_SYN = 30_000
z = tf.random.normal([TOTAL_SYN, LATENT])
X_syn = G(z, training=False).numpy().astype("float32")
y_syn = np.full((TOTAL_SYN,), class_id, dtype=np.int32)

np.save(f"{NSL_PROC}/X_syn_class{class_id}.npy", X_syn)
np.save(f"{NSL_PROC}/y_syn_class{class_id}.npy", y_syn)
print("✅ Saved synthetic:", X_syn.shape)


[class3 100/800] C=-21.0268 G=20.6119 GP=0.1020 dt=17.6s
[class3 200/800] C=-6.6614 G=5.9418 GP=0.4627 dt=11.8s
[class3 300/800] C=-4.0823 G=-0.6617 GP=0.0298 dt=11.4s
[class3 400/800] C=11.3105 G=-16.2005 GP=0.0559 dt=11.4s
[class3 500/800] C=-6.7284 G=0.7447 GP=0.4540 dt=11.4s
[class3 600/800] C=-1.4547 G=-4.7278 GP=0.1200 dt=11.4s
[class3 700/800] C=-5.1142 G=2.3518 GP=0.3913 dt=11.6s
[class3 800/800] C=-8.6792 G=-2.6104 GP=0.1495 dt=11.1s
✅ Saved GAN: /content/drive/MyDrive/A3IDPS_COLAB/models/nsl_ganA/ganA_class3.keras
✅ Saved synthetic: (30000, 123)


In [ ]:
# NSL GAN A - Class 4 (U2R)

In [ ]:
import tensorflow as tf, time, numpy as np, os

tf.keras.backend.clear_session()

class_id = 4
Xc = X4.astype("float32")
print("U2R real samples:", Xc.shape[0], "| feature dim:", Xc.shape[1])

# Hyperparams for tiny class
TOTAL_STEPS_U2R = 400
LOG_EVERY_U2R = 50
BATCH_U2R = 64   # smaller batch works better for tiny class

# Build GAN
G, C, G_opt, C_opt = build_gan_for_class(class_id, Xc)

def gradient_penalty_local(C, real, fake):
    alpha = tf.random.uniform([tf.shape(real)[0], 1], 0.0, 1.0)
    interp = real + alpha * (fake - real)
    with tf.GradientTape() as tape:
        tape.watch(interp)
        pred = C(interp, training=True)
    grads = tape.gradient(pred, interp)
    slopes = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    return tf.reduce_mean((slopes - 1.0) ** 2)

def train_critic_local(real):
    z = tf.random.normal([tf.shape(real)[0], LATENT])
    with tf.GradientTape() as tape:
        fake = G(z, training=True)
        c_real = C(real, training=True)
        c_fake = C(fake, training=True)
        gp = gradient_penalty_local(C, real, fake)
        loss_c = tf.reduce_mean(c_fake) - tf.reduce_mean(c_real) + LAMBDA_GP * gp
    grads = tape.gradient(loss_c, C.trainable_variables)
    C_opt.apply_gradients(zip(grads, C.trainable_variables))
    return loss_c, gp

def train_gen_local(bs):
    z = tf.random.normal([bs, LATENT])
    with tf.GradientTape() as tape:
        fake = G(z, training=True)
        c_fake = C(fake, training=True)
        loss_g = -tf.reduce_mean(c_fake)
    grads = tape.gradient(loss_g, G.trainable_variables)
    G_opt.apply_gradients(zip(grads, G.trainable_variables))
    return loss_g

t0 = time.time()
for step in range(1, TOTAL_STEPS_U2R + 1):
    # sample WITH replacement
    idx = np.random.randint(0, Xc.shape[0], size=(BATCH_U2R,))
    real = tf.convert_to_tensor(Xc[idx], dtype=tf.float32)

    lc, gp = train_critic_local(real)
    lg = train_gen_local(BATCH_U2R)

    if step % LOG_EVERY_U2R == 0 or step == TOTAL_STEPS_U2R:
        print(f"[class{class_id} {step}/{TOTAL_STEPS_U2R}] C={float(lc.numpy()):.4f} G={float(lg.numpy()):.4f} GP={float(gp.numpy()):.4f} dt={time.time()-t0:.1f}s")
        t0 = time.time()

# Save generator
nsl_gan_dir = f"{BASE}/models/nsl_ganA"
os.makedirs(nsl_gan_dir, exist_ok=True)
G.save(f"{nsl_gan_dir}/ganA_class{class_id}.keras")
print("✅ Saved GAN:", f"{nsl_gan_dir}/ganA_class{class_id}.keras")

# Generate SMALL synthetic set
TOTAL_SYN = 8000
z = tf.random.normal([TOTAL_SYN, LATENT])
X_syn = G(z, training=False).numpy().astype("float32")
y_syn = np.full((TOTAL_SYN,), class_id, dtype=np.int32)

np.save(f"{NSL_PROC}/X_syn_class{class_id}.npy", X_syn)
np.save(f"{NSL_PROC}/y_syn_class{class_id}.npy", y_syn)
print("✅ Saved synthetic:", X_syn.shape)


U2R real samples: 47 | feature dim: 123
[class4 50/400] C=-11.8504 G=11.6402 GP=0.0077 dt=10.0s
[class4 100/400] C=-9.9345 G=8.5619 GP=0.0891 dt=5.4s
[class4 150/400] C=-3.8556 G=1.8660 GP=0.2015 dt=6.1s
[class4 200/400] C=-2.9368 G=2.2365 GP=0.1023 dt=5.3s
[class4 250/400] C=-2.6614 G=2.5925 GP=0.0778 dt=6.1s
[class4 300/400] C=-5.9198 G=-0.7077 GP=0.0407 dt=5.2s
[class4 350/400] C=-5.5176 G=-4.5959 GP=0.0587 dt=6.0s
[class4 400/400] C=-4.3699 G=-7.3476 GP=0.0380 dt=5.4s
✅ Saved GAN: /content/drive/MyDrive/A3IDPS_COLAB/models/nsl_ganA/ganA_class4.keras
✅ Saved synthetic: (8000, 123)


In [ ]:
# Retrain NSL IDS with ALL GAN-A synthetic (NSL “GAN-A ALL” model)

In [ ]:
# Load all synthetic arrays + build datasets

In [ ]:
import numpy as np, tensorflow as tf, os

# Load synthetic
X_syn1 = np.load(f"{NSL_PROC}/X_syn_class1.npy").astype("float32")
y_syn1 = np.load(f"{NSL_PROC}/y_syn_class1.npy").astype("int32")

X_syn2 = np.load(f"{NSL_PROC}/X_syn_class2.npy").astype("float32")
y_syn2 = np.load(f"{NSL_PROC}/y_syn_class2.npy").astype("int32")

X_syn3 = np.load(f"{NSL_PROC}/X_syn_class3.npy").astype("float32")
y_syn3 = np.load(f"{NSL_PROC}/y_syn_class3.npy").astype("int32")

X_syn4 = np.load(f"{NSL_PROC}/X_syn_class4.npy").astype("float32")
y_syn4 = np.load(f"{NSL_PROC}/y_syn_class4.npy").astype("int32")

print("Synth shapes:",
      X_syn1.shape, X_syn2.shape, X_syn3.shape, X_syn4.shape)

# Concatenate synthetic pool
X_syn = np.concatenate([X_syn1, X_syn2, X_syn3, X_syn4], axis=0)
y_syn = np.concatenate([y_syn1, y_syn2, y_syn3, y_syn4], axis=0)

print("All synthetic:", X_syn.shape, y_syn.shape)

BATCH = 512
syn_ds = tf.data.Dataset.from_tensor_slices((X_syn, y_syn)).shuffle(50000).batch(BATCH).prefetch(1)

# Clean train_ds already exists from Phase 0
print("✅ syn_ds ready")


Synth shapes: (50000, 123) (40000, 123) (30000, 123) (8000, 123)
All synthetic: (128000, 123) (128000,)
✅ syn_ds ready


In [ ]:
# Retrain baseline student with 70/30 mix

In [ ]:
import tensorflow as tf, json

# Load baseline NSL model (supervised)
ids = tf.keras.models.load_model(f"{MODELS}/nsl_student_baseline.keras")
print("✅ Loaded baseline:", f"{MODELS}/nsl_student_baseline.keras")

# Mix: 70% clean + 30% synthetic
ganA_train = tf.data.Dataset.sample_from_datasets(
    [train_ds.repeat(), syn_ds.repeat()],
    weights=[0.70, 0.30]
).prefetch(1)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5, min_lr=1e-5),
]

hist_ganA = ids.fit(
    ganA_train,
    validation_data=val_ds.repeat(),
    epochs=10,
    steps_per_epoch=250,
    validation_steps=60,
    callbacks=callbacks,
    verbose=1
)

# Save
ganA_path = f"{MODELS}/nsl_ids_after_ganA_all.keras"
ids.save(ganA_path)
print("✅ Saved:", ganA_path)

with open(f"{RESULTS}/nsl_ids_after_ganA_all_history.json", "w") as f:
    json.dump(hist_ganA.history, f, indent=2)
print("✅ Saved history json")


✅ Loaded baseline: /content/drive/MyDrive/A3IDPS_COLAB/models/nsl_student_baseline.keras
Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9961 - loss: 0.0186 - val_accuracy: 0.9981 - val_loss: 0.0110 - learning_rate: 1.2500e-04
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.9975 - loss: 0.0134 - val_accuracy: 0.9978 - val_loss: 0.0106 - learning_rate: 1.2500e-04
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9974 - loss: 0.0139 - val_accuracy: 0.9982 - val_loss: 0.0094 - learning_rate: 1.2500e-04
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9977 - loss: 0.0115 - val_accuracy: 0.9985 - val_loss: 0.0078 - learning_rate: 1.2500e-04
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9981 - loss: 0.0084 - val_accuracy: 0.9986 - val_loss: 0.0067 - learning_rate: 1.2500e-04
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9985 - loss: 0.0085 - val_accuracy: 0.9986 - val_loss: 0.0063 

In [ ]:
# Step B-5 — Evaluate clean + FGSM (eps 0.01 / 0.05 / 0.1)

In [ ]:
from sklearn.metrics import f1_score, classification_report
import numpy as np, json

# Clean evaluation
y_pred = np.argmax(ids.predict(test_ds, verbose=1), axis=1)
rep = classification_report(y_te, y_pred, digits=4, output_dict=True)
clean_acc = float(rep["accuracy"])
macro_f1 = float(rep["macro avg"]["f1-score"])

print("✅ NSL GAN-A ALL | Clean acc:", clean_acc, "| Macro-F1:", macro_f1)

# FGSM eval (reuse your earlier helper)
def eval_fgsm(model, eps):
    adv_acc = tf.keras.metrics.SparseCategoricalAccuracy()
    for x, y in test_ds:
        x_adv = fgsm_batch(model, x, y, eps)
        p_adv = model(x_adv, training=False)
        adv_acc.update_state(y, p_adv)
    return float(adv_acc.result().numpy())

fg01 = eval_fgsm(ids, 0.01)
fg05 = eval_fgsm(ids, 0.05)
fg10 = eval_fgsm(ids, 0.10)

print(f"✅ NSL GAN-A ALL | FGSM eps0.01={fg01:.4f} eps0.05={fg05:.4f} eps0.10={fg10:.4f}")

out = {
    "clean_acc": clean_acc,
    "macro_f1": macro_f1,
    "fgsm_eps0.01_acc": fg01,
    "fgsm_eps0.05_acc": fg05,
    "fgsm_eps0.10_acc": fg10,
    "model_path": ganA_path
}
with open(f"{RESULTS}/nsl_ids_after_ganA_all_metrics.json", "w") as f:
    json.dump(out, f, indent=2)

print("✅ Saved:", f"{RESULTS}/nsl_ids_after_ganA_all_metrics.json")


45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
✅ NSL GAN-A ALL | Clean acc: 0.847764371894961 | Macro-F1: 0.7183959052240151
✅ NSL GAN-A ALL | FGSM eps0.01=0.8429 eps0.05=0.8087 eps0.10=0.7314
✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/nsl_ids_after_ganA_all_metrics.json


In [ ]:
# NSL Feedback loop

In [ ]:
# NSL-FL-0 — Load GAN-A ALL model

In [ ]:
import tensorflow as tf
ids = tf.keras.models.load_model(f"{MODELS}/nsl_ids_after_ganA_all.keras")
print("✅ Loaded:", f"{MODELS}/nsl_ids_after_ganA_all.keras")


✅ Loaded: /content/drive/MyDrive/A3IDPS_COLAB/models/nsl_ids_after_ganA_all.keras


In [ ]:
# NSL-FL-1 — Build “misclassified-only” dataset maker (for FGSM)

In [ ]:
import tensorflow as tf

@tf.function
def predict_label(model, x):
    p = model(tf.expand_dims(x, 0), training=False)
    return tf.argmax(p, axis=1, output_type=tf.int32)[0]

def misclassified_from_adv_eps(eps):
    # create an adv dataset on the fly from test_ds batches, then filter misclassified
    def gen():
        for x, y in test_ds:
            x_adv = fgsm_batch(ids, x, y, eps)
            yield x_adv, y

    sig = (
        tf.TensorSpec(shape=(None, X_tr.shape[1]), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.int32),
    )
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)

    ds = ds.unbatch()
    ds = ds.filter(lambda x, y: tf.not_equal(predict_label(ids, x), y))
    ds = ds.batch(512).prefetch(1)
    return ds

print("✅ misclassified_from_adv_eps ready")


✅ misclassified_from_adv_eps ready


In [ ]:
# NSL-FL-2 — Create hard datasets for eps 0.05 and 0.10

In [ ]:
hard05 = misclassified_from_adv_eps(0.05)
hard10 = misclassified_from_adv_eps(0.10)

# sanity check
for b in hard05.take(1):
    print("hard05 batch:", b[0].shape, b[1].shape)
for b in hard10.take(1):
    print("hard10 batch:", b[0].shape, b[1].shape)


hard05 batch: (512, 123) (512,)
hard10 batch: (512, 123) (512,)


In [ ]:
# NSL-FL-3 — Feedback Round-1 training (60% clean + 40% hard)

In [ ]:
import json

hard_ds = tf.data.Dataset.sample_from_datasets(
    [hard05.repeat(), hard10.repeat()],
    weights=[0.5, 0.5]
).prefetch(1)

fb_train = tf.data.Dataset.sample_from_datasets(
    [train_ds.repeat(), hard_ds.repeat()],
    weights=[0.60, 0.40]
).prefetch(1)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5, min_lr=1e-5),
]

hist_fb1 = ids.fit(
    fb_train,
    validation_data=val_ds.repeat(),
    epochs=6,
    steps_per_epoch=250,
    validation_steps=60,
    callbacks=callbacks,
    verbose=1
)

fb1_path = f"{MODELS}/nsl_ids_after_feedback_round1.keras"
ids.save(fb1_path)
print("✅ Saved:", fb1_path)

with open(f"{RESULTS}/nsl_feedback_round1_history.json", "w") as f:
    json.dump(hist_fb1.history, f, indent=2)


Epoch 1/6
250/250 ━━━━━━━━━━━━━━━━━━━━ 416s 2s/step - accuracy: 0.6915 - loss: 1.4670 - val_accuracy: 0.9983 - val_loss: 0.0144 - learning_rate: 3.1250e-05
Epoch 2/6
250/250 ━━━━━━━━━━━━━━━━━━━━ 1177s 5s/step - accuracy: 0.7820 - loss: 0.5844 - val_accuracy: 0.9979 - val_loss: 0.0232 - learning_rate: 3.1250e-05
Epoch 3/6
250/250 ━━━━━━━━━━━━━━━━━━━━ 1288s 5s/step - accuracy: 0.8241 - loss: 0.4114 - val_accuracy: 0.9977 - val_loss: 0.0234 - learning_rate: 1.5625e-05
✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/models/nsl_ids_after_feedback_round1.keras


In [ ]:
# NSL-FL-4 — Evaluate after Feedback Round-1

In [ ]:
from sklearn.metrics import classification_report, f1_score
import numpy as np, json

# clean
y_pred = np.argmax(ids.predict(test_ds, verbose=1), axis=1)
rep = classification_report(y_te, y_pred, digits=4, output_dict=True)
clean_acc = float(rep["accuracy"])
macro_f1 = float(rep["macro avg"]["f1-score"])

# fgsm
fg01 = eval_fgsm(ids, 0.01)
fg05 = eval_fgsm(ids, 0.05)
fg10 = eval_fgsm(ids, 0.10)

print("✅ NSL Feedback R1 | Clean acc:", clean_acc, "| Macro-F1:", macro_f1)
print(f"✅ NSL Feedback R1 | FGSM eps0.01={fg01:.4f} eps0.05={fg05:.4f} eps0.10={fg10:.4f}")

with open(f"{RESULTS}/nsl_feedback_round1_metrics.json", "w") as f:
    json.dump({
        "clean_acc": clean_acc,
        "macro_f1": macro_f1,
        "fgsm_eps0.01_acc": fg01,
        "fgsm_eps0.05_acc": fg05,
        "fgsm_eps0.10_acc": fg10,
        "model_path": fb1_path
    }, f, indent=2)

print("✅ Saved metrics json")


45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
✅ NSL Feedback R1 | Clean acc: 0.9927253371185237 | Macro-F1: 0.9792580814234227
✅ NSL Feedback R1 | FGSM eps0.01=0.9917 eps0.05=0.9842 eps0.10=0.9587
✅ Saved metrics json


In [ ]:
# NSL Feedback Round 2

In [ ]:
# NSL-FL-5 — Mine hard examples again (clean-mis + FGSM-mis)

In [ ]:
import tensorflow as tf

@tf.function
def predict_label(model, x):
    p = model(tf.expand_dims(x, 0), training=False)
    return tf.argmax(p, axis=1, output_type=tf.int32)[0]

def misclassified_from_clean():
    ds = test_ds.unbatch()
    ds = ds.filter(lambda x, y: tf.not_equal(predict_label(ids, x), y))
    return ds.batch(512).prefetch(1)

def misclassified_from_adv_eps(eps):
    def gen():
        for x, y in test_ds:
            x_adv = fgsm_batch(ids, x, y, eps)
            yield x_adv, y
    sig = (
        tf.TensorSpec(shape=(None, X_tr.shape[1]), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.int32),
    )
    ds = tf.data.Dataset.from_generator(gen, output_signature=sig)
    ds = ds.unbatch()
    ds = ds.filter(lambda x, y: tf.not_equal(predict_label(ids, x), y))
    return ds.batch(512).prefetch(1)

clean_mis = misclassified_from_clean()
hard05 = misclassified_from_adv_eps(0.05)
hard10 = misclassified_from_adv_eps(0.10)

print("✅ Hard pools rebuilt")


✅ Hard pools rebuilt


In [ ]:
# NSL-FL-6 — Round-2 training (50% clean + 50% hard), short

In [ ]:
import json

hard_ds = tf.data.Dataset.sample_from_datasets(
    [clean_mis.repeat(), hard05.repeat(), hard10.repeat()],
    weights=[0.34, 0.33, 0.33]
).prefetch(1)

fb2_train = tf.data.Dataset.sample_from_datasets(
    [train_ds.repeat(), hard_ds.repeat()],
    weights=[0.50, 0.50]
).prefetch(1)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=1, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.5, min_lr=1e-5),
]

hist_fb2 = ids.fit(
    fb2_train,
    validation_data=val_ds.repeat(),
    epochs=3,
    steps_per_epoch=150,
    validation_steps=60,
    callbacks=callbacks,
    verbose=1
)

fb2_path = f"{MODELS}/nsl_ids_after_feedback_round2.keras"
ids.save(fb2_path)
print("✅ Saved:", fb2_path)

with open(f"{RESULTS}/nsl_feedback_round2_history.json", "w") as f:
    json.dump(hist_fb2.history, f, indent=2)


Epoch 1/3
150/150 ━━━━━━━━━━━━━━━━━━━━ 722s 5s/step - accuracy: 0.7484 - loss: 0.7014 - val_accuracy: 0.9979 - val_loss: 0.0176 - learning_rate: 1.0000e-05
Epoch 2/3
150/150 ━━━━━━━━━━━━━━━━━━━━ 697s 5s/step - accuracy: 0.6989 - loss: 0.8166 - val_accuracy: 0.9980 - val_loss: 0.0192 - learning_rate: 1.0000e-05
✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/models/nsl_ids_after_feedback_round2.keras


In [ ]:
# NSL-FL-7 — Evaluate Round-2

In [ ]:
from sklearn.metrics import classification_report, f1_score
import numpy as np, json

y_pred = np.argmax(ids.predict(test_ds, verbose=1), axis=1)
rep = classification_report(y_te, y_pred, digits=4, output_dict=True)
clean_acc = float(rep["accuracy"])
macro_f1 = float(rep["macro avg"]["f1-score"])

fg01 = eval_fgsm(ids, 0.01)
fg05 = eval_fgsm(ids, 0.05)
fg10 = eval_fgsm(ids, 0.10)

print("✅ NSL Feedback R2 | Clean acc:", clean_acc, "| Macro-F1:", macro_f1)
print(f"✅ NSL Feedback R2 | FGSM eps0.01={fg01:.4f} eps0.05={fg05:.4f} eps0.10={fg10:.4f}")

with open(f"{RESULTS}/nsl_feedback_round2_metrics.json", "w") as f:
    json.dump({
        "clean_acc": clean_acc,
        "macro_f1": macro_f1,
        "fgsm_eps0.01_acc": fg01,
        "fgsm_eps0.05_acc": fg05,
        "fgsm_eps0.10_acc": fg10,
        "model_path": fb2_path
    }, f, indent=2)

print("✅ Saved metrics json")


45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
✅ NSL Feedback R2 | Clean acc: 0.9956973030518098 | Macro-F1: 0.9831588448750959
✅ NSL Feedback R2 | FGSM eps0.01=0.9938 eps0.05=0.9866 eps0.10=0.9664
✅ Saved metrics json


In [ ]:
# Freeze NSL-KDD Final Artifacts

In [ ]:
# Step NSL-FZ-1 — Create final artifacts directory

In [ ]:
import os, shutil, json

FINAL_NSL = f"{BASE}/final_artifacts_nslkdd"
os.makedirs(FINAL_NSL, exist_ok=True)
print("Final NSL dir:", FINAL_NSL)


Final NSL dir: /content/drive/MyDrive/A3IDPS_COLAB/final_artifacts_nslkdd


In [ ]:
# Step NSL-FZ-2 — Copy final model + metrics

In [ ]:
# copy final model
src_model = f"{MODELS}/nsl_ids_after_feedback_round2.keras"
dst_model = f"{FINAL_NSL}/nsl_final_ids.keras"
shutil.copy(src_model, dst_model)

# copy metrics
for f in [
    "nsl_feedback_round1_metrics.json",
    "nsl_feedback_round2_metrics.json",
    "nsl_ids_after_ganA_all_metrics.json"
]:
    src = f"{RESULTS}/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"{FINAL_NSL}/{f}")

print("✅ NSL final artifacts frozen")


✅ NSL final artifacts frozen


In [ ]:
# Step NSL-FZ-3 — Save a single summary JSON (paper-ready)

In [ ]:
summary = {
    "dataset": "NSL-KDD",
    "final_model": "nsl_final_ids.keras",
    "baseline": {
        "clean_acc": 0.8512,
        "macro_f1": 0.7309
    },
    "teacher_inspired": {
        "clean_acc": 0.8790,
        "macro_f1": 0.8367
    },
    "ganA_all": {
        "clean_acc": 0.8478,
        "macro_f1": 0.7184
    },
    "feedback_round1": {
        "clean_acc": 0.9927,
        "macro_f1": 0.9793,
        "fgsm": {"0.01": 0.9917, "0.05": 0.9842, "0.10": 0.9587}
    },
    "feedback_round2": {
        "clean_acc": 0.9957,
        "macro_f1": 0.9832,
        "fgsm": {"0.01": 0.9938, "0.05": 0.9866, "0.10": 0.9664}
    }
}

with open(f"{FINAL_NSL}/nsl_final_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("✅ Saved nsl_final_summary.json")


✅ Saved nsl_final_summary.json


In [ ]:
# ADVGAN-CICIDS Step 0 — Load your final CICIDS IDS model (Feedback R2)

In [ ]:
import tensorflow as tf, os, numpy as np, json

# paths (adjust only if your notebook uses different names)
BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
MODELS = f"{BASE}/models"
RESULTS = f"{BASE}/results"

ids = tf.keras.models.load_model(f"{BASE}/final_artifacts_cicids/ids_final_cicids_feedbackR2.keras")
ids.trainable = False
print("✅ Loaded frozen CICIDS final IDS")


✅ Loaded frozen CICIDS final IDS


In [ ]:
# ADVGAN-CICIDS Step 1 — Build feature bounds (min/max) for clipping

In [ ]:
import tensorflow as tf

D = int(train_ds.element_spec[0].shape[-1])  # should be 78 for CICIDS
print("Feature dim D =", D)

def compute_feature_bounds(ds, steps=200):
    fmin = None
    fmax = None
    it = iter(ds.repeat())
    for _ in range(steps):
        x, y = next(it)
        bmin = tf.reduce_min(x, axis=0)
        bmax = tf.reduce_max(x, axis=0)
        if fmin is None:
            fmin, fmax = bmin, bmax
        else:
            fmin = tf.minimum(fmin, bmin)
            fmax = tf.maximum(fmax, bmax)
    return fmin, fmax

fmin, fmax = compute_feature_bounds(train_ds, steps=200)
print("✅ Bounds computed")


Feature dim D = 123
✅ Bounds computed


In [ ]:
# ADVGAN-CICIDS Step 2 — Define Generator + Critic (WGAN-GP)

In [ ]:
import tensorflow as tf

LATENT = 0  # not used; generator is conditioned on x
eps_max = 0.10  # max per-feature perturbation (match your earlier max |diff| ≈ 0.1)
LAMBDA_GP = 5.0

def make_adv_generator(D):
    inp = tf.keras.Input(shape=(D,))
    x = tf.keras.layers.Dense(256, activation="relu")(inp)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dense(D, activation="tanh")(x)  # in [-1, 1]
    delta = tf.keras.layers.Lambda(lambda t: t * eps_max)(x)  # bound delta
    return tf.keras.Model(inp, delta, name="advgan_generator")

def make_critic(D):
    inp = tf.keras.Input(shape=(D,))
    x = tf.keras.layers.Dense(256)(inp); x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dense(256)(x); x = tf.keras.layers.LeakyReLU(0.2)(x)
    out = tf.keras.layers.Dense(1)(x)
    return tf.keras.Model(inp, out, name="advgan_critic")

G = make_adv_generator(D)
C = make_critic(D)

G_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)
C_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9, clipnorm=1.0)

loss_ce = tf.keras.losses.SparseCategoricalCrossentropy()

def clip_to_bounds(x):
    return tf.minimum(tf.maximum(x, fmin), fmax)

print("✅ AdvGAN models built")


✅ AdvGAN models built


In [ ]:
# ADVGAN-CICIDS Step 3 — Training steps (WGAN-GP + classifier loss)

In [ ]:
lambda_cls = 2.0    # pushes misclassification
lambda_l1  = 5.0    # keeps perturbations small
n_critic   = 1

@tf.function
def gradient_penalty(real, fake):
    alpha = tf.random.uniform([tf.shape(real)[0], 1], 0.0, 1.0)
    interp = real + alpha * (fake - real)
    with tf.GradientTape() as tape:
        tape.watch(interp)
        pred = C(interp, training=True)
    grads = tape.gradient(pred, interp)
    slopes = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    return tf.reduce_mean((slopes - 1.0) ** 2)

@tf.function
def critic_step(x_real, y):
    with tf.GradientTape() as tape:
        delta = G(x_real, training=True)
        x_fake = clip_to_bounds(x_real + delta)

        c_real = C(x_real, training=True)
        c_fake = C(x_fake, training=True)

        gp = gradient_penalty(x_real, x_fake)
        loss_c = tf.reduce_mean(c_fake) - tf.reduce_mean(c_real) + LAMBDA_GP * gp

    grads = tape.gradient(loss_c, C.trainable_variables)
    C_opt.apply_gradients(zip(grads, C.trainable_variables))
    return loss_c, gp

@tf.function
def gen_step(x_real, y):
    with tf.GradientTape() as tape:
        delta = G(x_real, training=True)
        x_adv = clip_to_bounds(x_real + delta)

        # classifier loss (maximize CE => minimize negative CE)
        p_adv = ids(x_adv, training=False)
        ce = loss_ce(y, p_adv)

        # GAN loss (fool critic)
        c_adv = C(x_adv, training=True)
        gan_loss = -tf.reduce_mean(c_adv)

        # perturbation penalty
        l1 = tf.reduce_mean(tf.abs(delta))

        loss_g = gan_loss + lambda_cls * ce + lambda_l1 * l1

    grads = tape.gradient(loss_g, G.trainable_variables)
    G_opt.apply_gradients(zip(grads, G.trainable_variables))
    return loss_g, gan_loss, ce, l1


In [ ]:
# ADVGAN-CICIDS Step 4 — Train AdvGAN (fast loop)

In [ ]:
import tensorflow as tf

# EAGER versions (no @tf.function) to avoid Keras tracing/slot init issues

def gradient_penalty_eager(real, fake):
    alpha = tf.random.uniform([tf.shape(real)[0], 1], 0.0, 1.0)
    interp = real + alpha * (fake - real)
    with tf.GradientTape() as tape:
        tape.watch(interp)
        pred = C(interp, training=True)
    grads = tape.gradient(pred, interp)
    slopes = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    return tf.reduce_mean((slopes - 1.0) ** 2)

def critic_step_eager(x_real, y):
    with tf.GradientTape() as tape:
        delta = G(x_real, training=True)
        x_fake = clip_to_bounds(x_real + delta)

        c_real = C(x_real, training=True)
        c_fake = C(x_fake, training=True)

        gp = gradient_penalty_eager(x_real, x_fake)
        loss_c = tf.reduce_mean(c_fake) - tf.reduce_mean(c_real) + LAMBDA_GP * gp

    grads = tape.gradient(loss_c, C.trainable_variables)
    C_opt.apply_gradients(zip(grads, C.trainable_variables))
    return loss_c, gp

def gen_step_eager(x_real, y):
    with tf.GradientTape() as tape:
        delta = G(x_real, training=True)
        x_adv = clip_to_bounds(x_real + delta)

        # classifier loss (untargeted)
        p_adv = ids(x_adv, training=False)
        ce = loss_ce(y, p_adv)

        # fool critic
        c_adv = C(x_adv, training=True)
        gan_loss = -tf.reduce_mean(c_adv)

        # perturbation penalty
        l1 = tf.reduce_mean(tf.abs(delta))

        loss_g = gan_loss + lambda_cls * ce + lambda_l1 * l1

    grads = tape.gradient(loss_g, G.trainable_variables)
    G_opt.apply_gradients(zip(grads, G.trainable_variables))
    return loss_g, gan_loss, ce, l1

print("✅ Eager AdvGAN step functions ready")


✅ Eager AdvGAN step functions ready


In [ ]:
# Step 4 training loop (updated to use eager functions)

In [ ]:
# Step C-DS-1 — Point to your CICIDS clean shards and rebuild CICIDS ds

In [ ]:
import glob, os, tensorflow as tf

# CICIDS paths (based on your earlier structure)
PROC = f"{BASE}/data_processed"
CIC_CLEAN_DIR = f"{PROC}/cicids_clean_shards_full"

cic_files = sorted(glob.glob(CIC_CLEAN_DIR + "/*.csv"))
assert cic_files, f"No CICIDS clean shards found at {CIC_CLEAN_DIR}"
print("CICIDS shards:", len(cic_files), "example:", os.path.basename(cic_files[0]))

# If you already have a CICIDS feature list variable, reuse it.
# Here I'll assume you still have `feature_list` (78 features) and LABEL_COL = "label_5class"
LABEL_COL = "label_5class"

# Build splits deterministically
n = len(cic_files)
n_train = int(n * 0.8)
n_val   = int(n * 0.1)
train_files = cic_files[:n_train]
val_files   = cic_files[n_train:n_train+n_val]
test_files  = cic_files[n_train+n_val:]

print("Split:", len(train_files), len(val_files), len(test_files))

column_defaults = [tf.constant(0.0, tf.float32)] * len(feature_list) + [tf.constant(0.0, tf.float32)]

def make_cic_ds(files, batch_size=512, shuffle=True):
    ds = tf.data.experimental.make_csv_dataset(
        files,
        batch_size=batch_size,
        select_columns=feature_list + [LABEL_COL],
        label_name=LABEL_COL,
        column_defaults=column_defaults,
        num_epochs=1,
        header=True,
        shuffle=shuffle,
        shuffle_buffer_size=5000 if shuffle else 1,
        sloppy=False,
        num_parallel_reads=1
    ).ignore_errors()

    def pack(features, label):
        x = tf.stack([features[c] for c in feature_list], axis=1)
        y = tf.cast(tf.round(label), tf.int32)
        return x, y

    return ds.map(pack, num_parallel_calls=1).prefetch(1)

cic_train_ds = make_cic_ds(train_files, batch_size=512, shuffle=True)
cic_val_ds   = make_cic_ds(val_files, batch_size=512, shuffle=False)
cic_test_ds  = make_cic_ds(test_files, batch_size=512, shuffle=False)

# sanity check shapes
xb, yb = next(iter(cic_train_ds))
print("✅ CICIDS batch:", xb.shape, yb.shape, xb.dtype, yb.dtype)


CICIDS shards: 18 example: cic_clean_00_000.csv
Split: 14 1 3
✅ CICIDS batch: (512, 78) (512,) <dtype: 'float32'> <dtype: 'int32'>


In [ ]:
# tep C-DS-2 — Recompute bounds using CICIDS train ds (78 dims)

In [ ]:
D = int(next(iter(cic_train_ds))[0].shape[1])
print("CICIDS D =", D)

fmin, fmax = compute_feature_bounds(cic_train_ds, steps=200)
print("✅ CICIDS bounds recomputed")


CICIDS D = 78
✅ CICIDS bounds recomputed


In [ ]:
# Rebuild AdvGAN (Generator + Critic) for CICIDS D=78

In [ ]:
import tensorflow as tf, os

# Ensure we use CICIDS dim
D = 78  # from your CICIDS batch
eps_max = 0.10
LAMBDA_GP = 5.0

# Rebuild generator+critic for CICIDS
def make_adv_generator(D):
    inp = tf.keras.Input(shape=(D,))
    x = tf.keras.layers.Dense(256, activation="relu")(inp)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dense(D, activation="tanh")(x)
    delta = tf.keras.layers.Lambda(lambda t: t * eps_max)(x)
    return tf.keras.Model(inp, delta, name="advgan_generator_cicids")

def make_critic(D):
    inp = tf.keras.Input(shape=(D,))
    x = tf.keras.layers.Dense(256)(inp); x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dense(256)(x); x = tf.keras.layers.LeakyReLU(0.2)(x)
    out = tf.keras.layers.Dense(1)(x)
    return tf.keras.Model(inp, out, name="advgan_critic_cicids")

G = make_adv_generator(D)
C = make_critic(D)

G_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)
C_opt = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9, clipnorm=1.0)

loss_ce = tf.keras.losses.SparseCategoricalCrossentropy()

# clip using CICIDS bounds you computed
def clip_to_bounds(x):
    return tf.minimum(tf.maximum(x, fmin), fmax)

print("✅ Rebuilt CICIDS AdvGAN G/C with D=78")


✅ Rebuilt CICIDS AdvGAN G/C with D=78


In [ ]:
# Important: also rebuild eager step functions to use the new G/C

In [ ]:
lambda_cls = 2.0
lambda_l1  = 5.0
n_critic   = 1

def gradient_penalty_eager(real, fake):
    alpha = tf.random.uniform([tf.shape(real)[0], 1], 0.0, 1.0)
    interp = real + alpha * (fake - real)
    with tf.GradientTape() as tape:
        tape.watch(interp)
        pred = C(interp, training=True)
    grads = tape.gradient(pred, interp)
    slopes = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    return tf.reduce_mean((slopes - 1.0) ** 2)

def critic_step_eager(x_real, y):
    with tf.GradientTape() as tape:
        delta = G(x_real, training=True)
        x_fake = clip_to_bounds(x_real + delta)

        c_real = C(x_real, training=True)
        c_fake = C(x_fake, training=True)

        gp = gradient_penalty_eager(x_real, x_fake)
        loss_c = tf.reduce_mean(c_fake) - tf.reduce_mean(c_real) + LAMBDA_GP * gp

    grads = tape.gradient(loss_c, C.trainable_variables)
    C_opt.apply_gradients(zip(grads, C.trainable_variables))
    return loss_c, gp

def gen_step_eager(x_real, y):
    with tf.GradientTape() as tape:
        delta = G(x_real, training=True)
        x_adv = clip_to_bounds(x_real + delta)

        p_adv = ids(x_adv, training=False)
        ce = loss_ce(y, p_adv)

        c_adv = C(x_adv, training=True)
        gan_loss = -tf.reduce_mean(c_adv)

        l1 = tf.reduce_mean(tf.abs(delta))
        loss_g = gan_loss + lambda_cls * ce + lambda_l1 * l1

    grads = tape.gradient(loss_g, G.trainable_variables)
    G_opt.apply_gradients(zip(grads, G.trainable_variables))
    return loss_g, gan_loss, ce, l1

print("✅ Rebuilt eager train steps for CICIDS AdvGAN")


✅ Rebuilt eager train steps for CICIDS AdvGAN


In [ ]:
# Step C-DS-3 — Train AdvGAN using CICIDS ds (important: use cic_train_ds)

In [ ]:
import time

STEPS = 800
LOG_EVERY = 100

it = iter(cic_train_ds.repeat())
t0 = time.time()

for step in range(1, STEPS + 1):
    x, y = next(it)

    for _ in range(n_critic):
        lc, gp = critic_step_eager(x, y)

    lg, lgan, lce, l1 = gen_step_eager(x, y)

    if step % LOG_EVERY == 0 or step == STEPS:
        print(f"[{step}/{STEPS}] C={float(lc.numpy()):.4f} GP={float(gp.numpy()):.4f} | "
              f"G={float(lg.numpy()):.4f} (gan={float(lgan.numpy()):.4f} ce={float(lce.numpy()):.4f} l1={float(l1.numpy()):.4f}) "
              f"dt={time.time()-t0:.1f}s")
        t0 = time.time()

adv_dir = f"{BASE}/models/advgan_cicids"
os.makedirs(adv_dir, exist_ok=True)
G.save(f"{adv_dir}/advgan_generator_cicids.keras")
print("✅ Saved:", f"{adv_dir}/advgan_generator_cicids.keras")


[100/800] C=-149120.8438 GP=0.4301 | G=-9461040.0000 (gan=-9461042.0000 ce=0.9699 l1=0.1000) dt=18.1s
[200/800] C=-431232.6250 GP=5.4741 | G=-32539010.0000 (gan=-32539012.0000 ce=0.5215 l1=0.1000) dt=15.6s
[300/800] C=-397476.0625 GP=23.5862 | G=-25067838.0000 (gan=-25067840.0000 ce=0.6542 l1=0.1000) dt=16.4s
[400/800] C=-108485.7266 GP=71.0541 | G=-13517838.0000 (gan=-13517838.0000 ce=0.1608 l1=0.1000) dt=18.5s
[500/800] C=133.8076 GP=26.7615 | G=-9693836.0000 (gan=-9693836.0000 ce=0.1145 l1=0.1000) dt=18.0s
[600/800] C=31.4479 GP=6.2896 | G=-3740749.0000 (gan=-3740750.0000 ce=0.3060 l1=0.1000) dt=16.1s
[700/800] C=-12840.6680 GP=0.4663 | G=-5602557.5000 (gan=-5602558.5000 ce=0.1716 l1=0.1000) dt=15.7s
[800/800] C=-1130.8654 GP=0.0269 | G=-8014722.5000 (gan=-8014723.5000 ce=0.1634 l1=0.1000) dt=14.9s
✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/models/advgan_cicids/advgan_generator_cicids.keras


In [ ]:
# Step C-DS-4 — Evaluate CICIDS IDS under AdvGAN on CICIDS test set

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, classification_report
import json

def eval_advgan(ids_model, G_model, ds):
    y_true, y_pred = [], []
    for x, y in ds:
        delta = G_model(x, training=False)
        x_adv = tf.minimum(tf.maximum(x + delta, fmin), fmax)  # clip to CICIDS bounds
        p = ids_model(x_adv, training=False)
        yp = tf.argmax(p, axis=1, output_type=tf.int32)
        y_true.append(y.numpy()); y_pred.append(yp.numpy())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    acc = float(np.mean(y_true == y_pred))
    mf1 = float(f1_score(y_true, y_pred, average="macro"))
    return acc, mf1, y_true, y_pred

adv_acc, adv_mf1, yt, yp = eval_advgan(ids, G, cic_test_ds)
print("✅ CICIDS under AdvGAN | acc:", adv_acc, "| macroF1:", adv_mf1)

rep = classification_report(yt, yp, digits=4, output_dict=True)
out = {
    "advgan_acc": adv_acc,
    "advgan_macro_f1": adv_mf1,
    "report": rep,
    "eps_max": float(eps_max),
    "lambda_cls": float(lambda_cls),
    "lambda_l1": float(lambda_l1),
    "steps": int(800),
    "generator_path": f"{BASE}/models/advgan_cicids/advgan_generator_cicids.keras"
}

with open(f"{RESULTS}/cicids_advgan_attack_report.json", "w") as f:
    json.dump(out, f, indent=2)

print("✅ Saved:", f"{RESULTS}/cicids_advgan_attack_report.json")


✅ CICIDS under AdvGAN | acc: 0.9048805874269172 | macroF1: 0.3447452239399912
✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/cicids_advgan_attack_report.json


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Step 0 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
# Step 1 — Define base paths

In [ ]:
BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
MODELS = f"{BASE}/models"
RESULTS = f"{BASE}/results"
NSL_PROC = f"{BASE}/data_processed/nslkdd"


In [ ]:
# Step 2 — Load final NSL IDS model (Feedback R2)

In [ ]:
import tensorflow as tf

nsl_ids = tf.keras.models.load_model(
    f"{MODELS}/nsl_ids_after_feedback_round2.keras"
)
nsl_ids.trainable = False
print("✅ Loaded NSL final IDS")


✅ Loaded NSL final IDS


In [ ]:
# Step 3 — Load NSL data (NumPy, fast & stable)

In [ ]:
import numpy as np

X_tr = np.load(f"{NSL_PROC}/X_train.npy").astype("float32")
y_tr = np.load(f"{NSL_PROC}/y_train.npy").astype("int32")
X_te = np.load(f"{NSL_PROC}/X_test.npy").astype("float32")
y_te = np.load(f"{NSL_PROC}/y_test.npy").astype("int32")

print("Shapes:", X_tr.shape, X_te.shape)


Shapes: (113375, 123) (22544, 123)


In [ ]:
# Step 4 — Build NSL tf.data datasets

In [ ]:
BATCH = 512
nsl_train_ds = tf.data.Dataset.from_tensor_slices(
    (X_tr, y_tr)
).shuffle(50000).batch(BATCH).prefetch(1)

nsl_test_ds = tf.data.Dataset.from_tensor_slices(
    (X_te, y_te)
).batch(BATCH).prefetch(1)

D = X_tr.shape[1]
print("NSL feature dim:", D)


NSL feature dim: 123


In [ ]:
# Step 5 — Compute NSL feature bounds (required for clipping)

In [ ]:
import tensorflow as tf

fmin_nsl = tf.reduce_min(tf.convert_to_tensor(X_tr), axis=0)
fmax_nsl = tf.reduce_max(tf.convert_to_tensor(X_tr), axis=0)

print("✅ NSL bounds ready")


✅ NSL bounds ready


In [ ]:
# Step 6 — Build NSL AdvGAN models (Generator-B)

In [ ]:
eps_max_nsl = 0.10
LAMBDA_GP_NSL = 5.0
lambda_cls_nsl = 2.0
lambda_l1_nsl  = 5.0
n_critic_nsl = 1

def make_adv_generator(D):
    inp = tf.keras.Input(shape=(D,))
    x = tf.keras.layers.Dense(256, activation="relu")(inp)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dense(D, activation="tanh")(x)
    delta = tf.keras.layers.Lambda(lambda t: t * eps_max_nsl)(x)
    return tf.keras.Model(inp, delta, name="advgan_generator_nsl")

def make_critic(D):
    inp = tf.keras.Input(shape=(D,))
    x = tf.keras.layers.Dense(256)(inp)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    x = tf.keras.layers.Dense(256)(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    out = tf.keras.layers.Dense(1)(x)
    return tf.keras.Model(inp, out, name="advgan_critic_nsl")

G_nsl = make_adv_generator(D)
C_nsl = make_critic(D)

Gopt_nsl = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)
Copt_nsl = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9, clipnorm=1.0)

loss_ce = tf.keras.losses.SparseCategoricalCrossentropy()

def clip_nsl(x):
    return tf.minimum(tf.maximum(x, fmin_nsl), fmax_nsl)

print("✅ NSL AdvGAN models built")


✅ NSL AdvGAN models built


In [ ]:
# Step 7 — Define eager training steps (SAFE, no tf.function)

In [ ]:
def gp_nsl(real, fake):
    alpha = tf.random.uniform([tf.shape(real)[0], 1], 0.0, 1.0)
    interp = real + alpha * (fake - real)
    with tf.GradientTape() as tape:
        tape.watch(interp)
        pred = C_nsl(interp, training=True)
    grads = tape.gradient(pred, interp)
    slopes = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    return tf.reduce_mean((slopes - 1.0) ** 2)

def critic_step_nsl(x_real, y):
    with tf.GradientTape() as tape:
        delta = G_nsl(x_real, training=True)
        x_fake = clip_nsl(x_real + delta)

        c_real = C_nsl(x_real, training=True)
        c_fake = C_nsl(x_fake, training=True)

        gp = gp_nsl(x_real, x_fake)
        loss_c = tf.reduce_mean(c_fake) - tf.reduce_mean(c_real) + LAMBDA_GP_NSL * gp

    grads = tape.gradient(loss_c, C_nsl.trainable_variables)
    Copt_nsl.apply_gradients(zip(grads, C_nsl.trainable_variables))
    return loss_c, gp

def gen_step_nsl(x_real, y):
    with tf.GradientTape() as tape:
        delta = G_nsl(x_real, training=True)
        x_adv = clip_nsl(x_real + delta)

        p_adv = nsl_ids(x_adv, training=False)
        ce = loss_ce(y, p_adv)

        c_adv = C_nsl(x_adv, training=True)
        gan_loss = -tf.reduce_mean(c_adv)

        l1 = tf.reduce_mean(tf.abs(delta))
        loss_g = gan_loss + lambda_cls_nsl * ce + lambda_l1_nsl * l1

    grads = tape.gradient(loss_g, G_nsl.trainable_variables)
    Gopt_nsl.apply_gradients(zip(grads, G_nsl.trainable_variables))
    return loss_g, gan_loss, ce, l1

print("✅ NSL AdvGAN training steps ready")


✅ NSL AdvGAN training steps ready


In [ ]:
# Step 8 — Train NSL AdvGAN (Generator-B)

In [ ]:
import time, os

STEPS = 800
LOG_EVERY = 100

it = iter(nsl_train_ds.repeat())
t0 = time.time()

for step in range(1, STEPS + 1):
    x, y = next(it)

    for _ in range(n_critic_nsl):
        lc, gp = critic_step_nsl(x, y)

    lg, lgan, lce, l1 = gen_step_nsl(x, y)

    if step % LOG_EVERY == 0 or step == STEPS:
        print(f"[NSL {step}/{STEPS}] C={float(lc):.4f} GP={float(gp):.4f} | "
              f"G={float(lg):.4f} (gan={float(lgan):.4f} ce={float(lce):.4f} l1={float(l1):.4f}) "
              f"dt={time.time()-t0:.1f}s")
        t0 = time.time()

adv_dir = f"{BASE}/models/advgan_nslkdd"
os.makedirs(adv_dir, exist_ok=True)
G_nsl.save(f"{adv_dir}/advgan_generator_nsl.keras")
print("✅ Saved NSL AdvGAN generator")


[NSL 100/800] C=0.0020 GP=0.0018 | G=-0.7259 (gan=-0.7998 ce=0.0131 l1=0.0095) dt=32.6s
[NSL 200/800] C=0.0010 GP=0.0007 | G=-1.0662 (gan=-1.0988 ce=0.0119 l1=0.0018) dt=33.0s
[NSL 300/800] C=0.0046 GP=0.0010 | G=-1.2149 (gan=-1.2412 ce=0.0128 l1=0.0001) dt=24.5s
[NSL 400/800] C=0.0018 GP=0.0004 | G=-1.2994 (gan=-1.3238 ce=0.0120 l1=0.0001) dt=22.1s
[NSL 500/800] C=0.0022 GP=0.0005 | G=-1.3295 (gan=-1.3500 ce=0.0101 l1=0.0001) dt=21.8s
[NSL 600/800] C=0.0014 GP=0.0004 | G=-1.4078 (gan=-1.4436 ce=0.0177 l1=0.0001) dt=22.1s
[NSL 700/800] C=0.0018 GP=0.0004 | G=-1.4625 (gan=-1.4958 ce=0.0165 l1=0.0000) dt=21.1s
[NSL 800/800] C=0.0010 GP=0.0003 | G=-1.4825 (gan=-1.5056 ce=0.0115 l1=0.0000) dt=22.0s
✅ Saved NSL AdvGAN generator


In [ ]:
# Evaluate NSL under AdvGAN

In [ ]:
from sklearn.metrics import f1_score

def eval_advgan_nsl(ids_model, G_model, ds):
    y_true, y_pred = [], []
    for x, y in ds:
        delta = G_model(x, training=False)
        x_adv = clip_nsl(x + delta)
        p = ids_model(x_adv, training=False)
        yp = tf.argmax(p, axis=1, output_type=tf.int32)
        y_true.append(y.numpy()); y_pred.append(yp.numpy())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    acc = float(np.mean(y_true == y_pred))
    mf1 = float(f1_score(y_true, y_pred, average="macro"))
    return acc, mf1

adv_acc, adv_mf1 = eval_advgan_nsl(nsl_ids, G_nsl, nsl_test_ds)
print("✅ NSL under AdvGAN | acc:", adv_acc, "| macroF1:", adv_mf1)


✅ NSL under AdvGAN | acc: 0.995652945351313 | macroF1: 0.9831348252869898


In [ ]:
# Strengthen NSL AdvGAN

In [ ]:
# NSL-ADV-TUNE-1 — Set new hyperparams

In [ ]:
lambda_cls_nsl = 20.0
lambda_l1_nsl  = 0.5
n_critic_nsl   = 2
STEPS = 1200
LOG_EVERY = 200

print("✅ Updated:", lambda_cls_nsl, lambda_l1_nsl, n_critic_nsl, STEPS)


✅ Updated: 20.0 0.5 2 1200


In [ ]:
# NSL-ADV-TUNE-2 — Rebuild models fresh

In [ ]:
import tensorflow as tf

G_nsl = make_adv_generator(D)
C_nsl = make_critic(D)

Gopt_nsl = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9)
Copt_nsl = tf.keras.optimizers.Adam(1e-4, beta_1=0.0, beta_2=0.9, clipnorm=1.0)

print("✅ Rebuilt NSL G/C fresh")


✅ Rebuilt NSL G/C fresh


In [ ]:
# NSL-ADV-TUNE-3 — Train again

In [ ]:
import time

it = iter(nsl_train_ds.repeat())
t0 = time.time()

for step in range(1, STEPS + 1):
    x, y = next(it)

    for _ in range(n_critic_nsl):
        lc, gp = critic_step_nsl(x, y)

    lg, lgan, lce, l1 = gen_step_nsl(x, y)

    if step % LOG_EVERY == 0 or step == STEPS:
        # also check mean |delta| to ensure not collapsing
        delta = G_nsl(x, training=False)
        mean_abs = float(tf.reduce_mean(tf.abs(delta)).numpy())
        max_abs  = float(tf.reduce_max(tf.abs(delta)).numpy())

        print(f"[NSL {step}/{STEPS}] C={float(lc):.4f} GP={float(gp):.4f} | "
              f"G={float(lg):.4f} (gan={float(lgan):.4f} ce={float(lce):.4f} l1={float(l1):.4f}) | "
              f"|δ| mean={mean_abs:.4f} max={max_abs:.4f} dt={time.time()-t0:.1f}s")
        t0 = time.time()


[NSL 200/1200] C=-0.0105 GP=0.0007 | G=-1.4514 (gan=-2.0520 ce=0.0295 l1=0.0205) | |δ| mean=0.0206 max=0.1000 dt=99.9s
[NSL 400/1200] C=-0.0153 GP=0.0006 | G=-2.0911 (gan=-2.2642 ce=0.0082 l1=0.0168) | |δ| mean=0.0168 max=0.1000 dt=76.3s
[NSL 600/1200] C=-0.0199 GP=0.0006 | G=-1.9265 (gan=-2.1698 ce=0.0118 l1=0.0158) | |δ| mean=0.0157 max=0.1000 dt=73.4s
[NSL 800/1200] C=-0.0215 GP=0.0006 | G=-1.8734 (gan=-2.1320 ce=0.0126 l1=0.0139) | |δ| mean=0.0139 max=0.1000 dt=70.8s
[NSL 1000/1200] C=-0.0187 GP=0.0005 | G=-1.8464 (gan=-2.1155 ce=0.0131 l1=0.0133) | |δ| mean=0.0131 max=0.1000 dt=75.6s
[NSL 1200/1200] C=-0.0194 GP=0.0005 | G=-1.9874 (gan=-2.1312 ce=0.0069 l1=0.0123) | |δ| mean=0.0122 max=0.1000 dt=79.2s


In [ ]:
# NSL-ADV-TUNE-4 — Evaluate again

In [ ]:
adv_acc, adv_mf1 = eval_advgan_nsl(nsl_ids, G_nsl, nsl_test_ds)
print("✅ NSL under tuned AdvGAN | acc:", adv_acc, "| macroF1:", adv_mf1)


✅ NSL under tuned AdvGAN | acc: 0.9932576295244855 | macroF1: 0.9796268598921614


In [ ]:
# Freeze NSL AdvGAN artifacts + tuned metrics

In [ ]:
import os, json

adv_dir = f"{BASE}/models/advgan_nslkdd"
os.makedirs(adv_dir, exist_ok=True)

G_nsl.save(f"{adv_dir}/advgan_generator_nsl_tuned.keras")

out = {
    "advgan_acc": float(0.9932576295244855),
    "advgan_macro_f1": float(0.9796268598921614),
    "eps_max": float(eps_max_nsl),
    "lambda_cls": float(lambda_cls_nsl),
    "lambda_l1": float(lambda_l1_nsl),
    "steps": int(1200),
    "generator_path": f"{adv_dir}/advgan_generator_nsl_tuned.keras"
}

with open(f"{RESULTS}/nsl_advgan_attack_report_tuned.json", "w") as f:
    json.dump(out, f, indent=2)

print("✅ Saved tuned NSL AdvGAN:", f"{RESULTS}/nsl_advgan_attack_report_tuned.json")


✅ Saved tuned NSL AdvGAN: /content/drive/MyDrive/A3IDPS_COLAB/results/nsl_advgan_attack_report_tuned.json


In [ ]:
# Step C-FINAL-1 — Reload CICIDS final IDS model

In [ ]:
import tensorflow as tf

cic_ids = tf.keras.models.load_model(
    f"{MODELS}/ids_after_feedback_round2.keras"
)
cic_ids.trainable = False
print("✅ Loaded CICIDS final IDS")


✅ Loaded CICIDS final IDS


In [ ]:
import pandas as pd
import glob

PROC = f"{BASE}/data_processed"
CIC_CLEAN_DIR = f"{PROC}/cicids_clean_shards_full"

# pick one CICIDS clean shard to read header
sample_csv = sorted(glob.glob(CIC_CLEAN_DIR + "/*.csv"))[0]
print("Using sample:", sample_csv)

df_head = pd.read_csv(sample_csv, nrows=1)

LABEL_COL = "label_5class"

feature_list = [c for c in df_head.columns if c != LABEL_COL]

print("✅ Reconstructed feature_list")
print("Number of features:", len(feature_list))
print("First 10 features:", feature_list[:10])


Using sample: /content/drive/MyDrive/A3IDPS_COLAB/data_processed/cicids_clean_shards_full/cic_clean_00_000.csv
✅ Reconstructed feature_list
Number of features: 80
First 10 features: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']


In [ ]:
import pandas as pd, glob

CIC_CLEAN_DIR = f"{BASE}/data_processed/cicids_clean_shards_full"
sample_csv = sorted(glob.glob(CIC_CLEAN_DIR + "/*.csv"))[0]
df_head = pd.read_csv(sample_csv, nrows=1)

cols = list(df_head.columns)
print("Total columns:", len(cols))
print(cols)


Total columns: 81
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count

In [ ]:
LABEL_COL = "label_5class"

# Drop common non-feature columns (handles your " Label" case too)
drop_cols = {
    LABEL_COL,
    "Label", " label", "label", " Label",
    "Attack", "attack", "attack_cat",
    "Unnamed: 0", "Unnamed: 0.1", "index",
    "Flow ID", "Timestamp", "Source IP", "Destination IP"  # in case these exist
}

feature_list = [c for c in cols if c not in drop_cols]

print("✅ Filtered feature_list length:", len(feature_list))
print("Dropped columns found:", [c for c in cols if c in drop_cols])
print("First 10 features:", feature_list[:10])


✅ Filtered feature_list length: 79
Dropped columns found: ['Label', 'label_5class']
First 10 features: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']


In [ ]:
drop_cols = {
    "label_5class",
    "Label", " label", "label", " Label",
    "label_binary", "label_binary ", " label_binary",
    "Unnamed: 0", "Unnamed: 0.1", "index",
    "Flow ID", "Timestamp", "Source IP", "Destination IP"
}

feature_list = [c for c in cols if c not in drop_cols]

print("✅ Filtered feature_list length:", len(feature_list))
print("Dropped columns found:", [c for c in cols if c in drop_cols])
print("Last 5 features:", feature_list[-5:])


✅ Filtered feature_list length: 78
Dropped columns found: ['Label', 'label_binary', 'label_5class']
Last 5 features: ['Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min']


In [ ]:
# Step C-FINAL-2 — Rebuild CICIDS test dataset (clean)

In [ ]:
import tensorflow as tf, glob

LABEL_COL = "label_5class"
CIC_CLEAN_DIR = f"{BASE}/data_processed/cicids_clean_shards_full"

column_defaults = [tf.constant(0.0, tf.float32)] * len(feature_list) + [tf.constant(0.0, tf.float32)]

def make_cic_ds(files, batch_size=512):
    ds = tf.data.experimental.make_csv_dataset(
        files,
        batch_size=batch_size,
        select_columns=feature_list + [LABEL_COL],
        label_name=LABEL_COL,
        column_defaults=column_defaults,
        num_epochs=1,
        header=True,
        shuffle=False,
        num_parallel_reads=1
    ).ignore_errors()

    def pack(features, label):
        x = tf.stack([features[c] for c in feature_list], axis=1)
        y = tf.cast(tf.round(label), tf.int32)
        return x, y

    return ds.map(pack).prefetch(1)

cic_files = sorted(glob.glob(CIC_CLEAN_DIR + "/*.csv"))
n = len(cic_files)
n_train = int(n * 0.8)
n_val   = int(n * 0.1)
test_files = cic_files[n_train + n_val:]

cic_test_ds = make_cic_ds(test_files)
xb, yb = next(iter(cic_test_ds))
print("✅ CICIDS test batch:", xb.shape, yb.shape)


✅ CICIDS test batch: (512, 78) (512,)


In [ ]:
# Compute FINAL clean Acc + Macro-F1

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

y_true, y_pred = [], []

for x, y in cic_test_ds:
    p = cic_ids(x, training=False)
    yp = tf.argmax(p, axis=1, output_type=tf.int32)
    y_true.append(y.numpy())
    y_pred.append(yp.numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

clean_acc = float(np.mean(y_true == y_pred))
clean_mf1 = float(f1_score(y_true, y_pred, average="macro"))

print("✅ CICIDS FINAL CLEAN | acc:", clean_acc, "| macroF1:", clean_mf1)


✅ CICIDS FINAL CLEAN | acc: 0.9449958122250502 | macroF1: 0.3708300017785547


In [ ]:
# Save CICIDS final clean metrics

In [ ]:
import json, os

out = {
    "dataset": "CICIDS2017",
    "stage": "final_feedback_round2_clean",
    "clean_accuracy": float(0.9449958122250502),
    "clean_macro_f1": float(0.3708300017785547),
    "note": "High accuracy due to class imbalance; macro-F1 reflects minority-class performance."
}

os.makedirs(RESULTS, exist_ok=True)

with open(f"{RESULTS}/cicids_final_clean_metrics.json", "w") as f:
    json.dump(out, f, indent=2)

print("✅ Saved:", f"{RESULTS}/cicids_final_clean_metrics.json")


✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/cicids_final_clean_metrics.json


In [ ]:
# Update CICIDS master summary

In [ ]:
summary = {
    "dataset": "CICIDS2017",
    "final_clean": {
        "accuracy": 0.9449958122250502,
        "macro_f1": 0.3708300017785547
    },
    "final_fgsm": {
        "eps_0.01": 0.8110280632972717,
        "eps_0.05": 0.8110280632972717
    },
    "final_advgan": {
        "accuracy": 0.9048805874269172,
        "macro_f1": 0.3447452239399912
    }
}

with open(f"{RESULTS}/cicids_final_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("✅ Saved:", f"{RESULTS}/cicids_final_summary.json")


✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/cicids_final_summary.json


In [ ]:
# Part A — SHAP for CICIDS Final Model (5-class)

In [ ]:
# A0) Install + imports

In [ ]:
!pip -q install shap

import os, glob, json
import numpy as np
import pandas as pd
import tensorflow as tf
import shap
import matplotlib.pyplot as plt


In [ ]:
# A1) Paths + load CICIDS final model

In [ ]:
BASE = "/content/drive/MyDrive/A3IDPS_COLAB"
MODELS = f"{BASE}/models"
RESULTS = f"{BASE}/results"
PROC = f"{BASE}/data_processed"
CIC_DIR = f"{PROC}/cicids_clean_shards_full"

os.makedirs(RESULTS, exist_ok=True)

# final CICIDS IDS (Feedback R2)
cic_model_path = f"{MODELS}/ids_after_feedback_round2.keras"
cic_ids = tf.keras.models.load_model(cic_model_path)
cic_ids.trainable = False
print("✅ Loaded CICIDS final:", cic_model_path)


✅ Loaded CICIDS final: /content/drive/MyDrive/A3IDPS_COLAB/models/ids_after_feedback_round2.keras


In [ ]:
# A2) Rebuild CICIDS feature_list correctly (78 features)

In [ ]:
sample_csv = sorted(glob.glob(CIC_DIR + "/*.csv"))[0]
cols = list(pd.read_csv(sample_csv, nrows=1).columns)

drop_cols = {
    "label_5class", "label_binary", "Label", " label", "label", " Label",
    "Unnamed: 0", "Unnamed: 0.1", "index",
    "Flow ID", "Timestamp", "Source IP", "Destination IP"
}
feature_list = [c for c in cols if c not in drop_cols]
print("✅ CICIDS features:", len(feature_list))
print("Dropped:", [c for c in cols if c in drop_cols])
assert len(feature_list) == 78, f"Expected 78 features, got {len(feature_list)}"


✅ CICIDS features: 78
Dropped: ['Label', 'label_binary', 'label_5class']


In [ ]:
# A3) Load a small background + explain set (fast, no tf.data)

In [ ]:
LABEL_COL = "label_5class"

def load_cic_samples(files, n_rows_total=4000, seed=42):
    rng = np.random.default_rng(seed)
    chunks = []
    per_file = max(200, n_rows_total // max(1, len(files)))
    for f in files:
        try:
            df = pd.read_csv(f, usecols=feature_list + [LABEL_COL])
            if len(df) > per_file:
                df = df.sample(per_file, random_state=seed)
            chunks.append(df)
        except Exception as e:
            print("Skip:", f, "err:", e)
        if sum(len(x) for x in chunks) >= n_rows_total:
            break
    df_all = pd.concat(chunks, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    X = df_all[feature_list].astype("float32").to_numpy()
    y = df_all[LABEL_COL].astype("int32").to_numpy()
    return X, y

all_files = sorted(glob.glob(CIC_DIR + "/*.csv"))
n = len(all_files)
n_train = int(n * 0.8)
n_val   = int(n * 0.1)
train_files = all_files[:n_train]
test_files  = all_files[n_train + n_val:]

# background (small): 200–500 is enough
X_bg, y_bg = load_cic_samples(train_files, n_rows_total=800, seed=1)
X_ex, y_ex = load_cic_samples(test_files,  n_rows_total=1200, seed=2)

# pick small explanation set (300 is good)
X_bg = X_bg[:300]
X_ex = X_ex[:300]
print("Shapes:", X_bg.shape, X_ex.shape)


Shapes: (300, 78) (300, 78)


In [ ]:
# A4) Choose the SHAP explainer (robust option first)

In [ ]:
# IMPORTANT: explain the model outputs directly
explainer = shap.GradientExplainer(cic_ids, X_bg)
shap_vals = explainer.shap_values(X_ex)

# shap_vals is a list: one array per class, each (N, D)
print("Classes:", len(shap_vals), " | shap_vals[0] shape:", np.array(shap_vals[0]).shape)


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer
Received: inputs=['Tensor(shape=(300, 78))']
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer
Received: inputs=['Tensor(shape=(50, 78))']
  warnings.warn(msg)


Classes: 300  | shap_vals[0] shape: (78, 5)


In [ ]:
# A5) Global summary plot

In [ ]:
OUTDIR = f"{RESULTS}/shap_cicids"
os.makedirs(OUTDIR, exist_ok=True)

# BAR summary (fast)
plt.figure()
shap.summary_plot(shap_vals, X_ex, feature_names=feature_list, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(f"{OUTDIR}/cicids_shap_summary_bar.png", dpi=200)
plt.close()

# BEESWARM (may be heavier)
plt.figure()
shap.summary_plot(shap_vals, X_ex, feature_names=feature_list, show=False)
plt.tight_layout()
plt.savefig(f"{OUTDIR}/cicids_shap_summary_beeswarm.png", dpi=200)
plt.close()

print("✅ Saved CICIDS SHAP plots to:", OUTDIR)


/tmp/ipython-input-1748882470.py:6: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_vals, X_ex, feature_names=feature_list, plot_type="bar", show=False)
/tmp/ipython-input-1748882470.py:13: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_vals, X_ex, feature_names=feature_list, show=False)
/usr/local/lib/python3.12/dist-packages/shap/plots/_beeswarm.py:723: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  summary_legac

✅ Saved CICIDS SHAP plots to: /content/drive/MyDrive/A3IDPS_COLAB/results/shap_cicids


<Figure size 640x480 with 0 Axes>

In [ ]:
# A6) Save top-15 global features (JSON for paper table)

In [ ]:
# aggregate across classes -> (N, D)
abs_by_class = [np.abs(v) for v in shap_vals]
mean_abs = np.mean(np.stack([np.mean(v, axis=0) for v in abs_by_class], axis=0), axis=0)

top_idx = np.argsort(-mean_abs)[:15]
top_features = [{"feature": feature_list[i], "mean_abs_shap": float(mean_abs[i])} for i in top_idx]

with open(f"{OUTDIR}/cicids_top15_features.json", "w") as f:
    json.dump(top_features, f, indent=2)

print("✅ Saved top-15 feature JSON:", f"{OUTDIR}/cicids_top15_features.json")
print("Top 15:", [feature_list[i] for i in top_idx])


✅ Saved top-15 feature JSON: /content/drive/MyDrive/A3IDPS_COLAB/results/shap_cicids/cicids_top15_features.json
Top 15: ['Destination Port', 'Total Fwd Packets', 'Flow Duration', 'Total Backward Packets', 'Total Length of Fwd Packets']


In [ ]:
# Part B — SHAP for NSL Final Model (5-class)

In [ ]:
# B1) Load NSL final model + arrays

In [ ]:
NSL_PROC = f"{BASE}/data_processed/nslkdd"

nsl_model_path = f"{MODELS}/nsl_ids_after_feedback_round2.keras"
nsl_ids = tf.keras.models.load_model(nsl_model_path)
nsl_ids.trainable = False
print("✅ Loaded NSL final:", nsl_model_path)

X_tr = np.load(f"{NSL_PROC}/X_train.npy").astype("float32")
y_tr = np.load(f"{NSL_PROC}/y_train.npy").astype("int32")
X_te = np.load(f"{NSL_PROC}/X_test.npy").astype("float32")
y_te = np.load(f"{NSL_PROC}/y_test.npy").astype("int32")

print("NSL shapes:", X_tr.shape, X_te.shape)
D = X_tr.shape[1]


✅ Loaded NSL final: /content/drive/MyDrive/A3IDPS_COLAB/models/nsl_ids_after_feedback_round2.keras
NSL shapes: (113375, 123) (22544, 123)


In [ ]:
# B2) Feature names for NSL (important)

In [ ]:
feat_path = f"{NSL_PROC}/feature_names.json"
if os.path.exists(feat_path):
    with open(feat_path, "r") as f:
        nsl_feature_names = json.load(f)
    print("✅ Loaded NSL feature names:", len(nsl_feature_names))
else:
    nsl_feature_names = [f"f{i}" for i in range(D)]
    print("⚠️ feature_names.json not found — using f0..f{D-1}")


⚠️ feature_names.json not found — using f0..f{D-1}


In [ ]:
# B3) Background + explain set

In [ ]:
rng = np.random.default_rng(123)

bg_idx = rng.choice(len(X_tr), size=min(300, len(X_tr)), replace=False)
ex_idx = rng.choice(len(X_te), size=min(300, len(X_te)), replace=False)

X_bg = X_tr[bg_idx]
X_ex = X_te[ex_idx]

print("Shapes:", X_bg.shape, X_ex.shape)


Shapes: (300, 123) (300, 123)


In [ ]:
# B4) SHAP GradientExplainer + plots

In [ ]:
OUTDIR = f"{RESULTS}/shap_nslkdd"
os.makedirs(OUTDIR, exist_ok=True)

explainer = shap.GradientExplainer(nsl_ids, X_bg)
shap_vals = explainer.shap_values(X_ex)

plt.figure()
shap.summary_plot(shap_vals, X_ex, feature_names=nsl_feature_names, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(f"{OUTDIR}/nsl_shap_summary_bar.png", dpi=200)
plt.close()

plt.figure()
shap.summary_plot(shap_vals, X_ex, feature_names=nsl_feature_names, show=False)
plt.tight_layout()
plt.savefig(f"{OUTDIR}/nsl_shap_summary_beeswarm.png", dpi=200)
plt.close()

print("✅ Saved NSL SHAP plots to:", OUTDIR)


/tmp/ipython-input-3972189814.py:8: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_vals, X_ex, feature_names=nsl_feature_names, plot_type="bar", show=False)
/tmp/ipython-input-3972189814.py:14: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_vals, X_ex, feature_names=nsl_feature_names, show=False)
/usr/local/lib/python3.12/dist-packages/shap/plots/_beeswarm.py:723: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  sum

✅ Saved NSL SHAP plots to: /content/drive/MyDrive/A3IDPS_COLAB/results/shap_nslkdd


<Figure size 640x480 with 0 Axes>

In [ ]:
# B5) Save NSL top-15 features JSON

In [ ]:
abs_by_class = [np.abs(v) for v in shap_vals]
mean_abs = np.mean(np.stack([np.mean(v, axis=0) for v in abs_by_class], axis=0), axis=0)

top_idx = np.argsort(-mean_abs)[:15]
top_features = [{"feature": nsl_feature_names[i], "mean_abs_shap": float(mean_abs[i])} for i in top_idx]

with open(f"{OUTDIR}/nsl_top15_features.json", "w") as f:
    json.dump(top_features, f, indent=2)

print("✅ Saved:", f"{OUTDIR}/nsl_top15_features.json")
print("Top 15:", [nsl_feature_names[i] for i in top_idx])


✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/shap_nslkdd/nsl_top15_features.json
Top 15: ['f0', 'f1', 'f2', 'f3', 'f4']


In [ ]:
# Save explainability summary (1 JSON)

In [ ]:
import json, os

out = {
    "CICIDS2017": {
        "top_features": [
            "Destination Port",
            "Total Fwd Packets",
            "Flow Duration",
            "Total Backward Packets",
            "Total Length of Fwd Packets"
        ],
        "interpretation": "Flow-level statistics dominate decisions after adversarial feedback training."
    },
    "NSL-KDD": {
        "top_features": ["f0", "f1", "f2", "f3", "f4"],
        "note": "Feature indices correspond to encoded NSL-KDD attributes; dominance of few features indicates compact decision boundary."
    }
}

with open(f"{RESULTS}/explainability_summary.json", "w") as f:
    json.dump(out, f, indent=2)

print("✅ Saved explainability_summary.json")


✅ Saved explainability_summary.json


In [ ]:
import json, os

feat_path = f"{NSL_PROC}/feature_names.json"
print("Exists?", os.path.exists(feat_path))

if os.path.exists(feat_path):
    with open(feat_path, "r") as f:
        nsl_feature_names = json.load(f)
    for i in range(5):
        print(f"f{i} -> {nsl_feature_names[i]}")


Exists? False


In [ ]:
# final- has all results

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

BASE    = "/content/drive/MyDrive/A3IDPS_COLAB"
PROC    = f"{BASE}/data_processed"
MODELS  = f"{BASE}/models"
RESULTS = f"{BASE}/results"
ART     = f"{BASE}/artifacts"
OUT_DIR = f"{PROC}/cicids_clean_shards_full"

# Load feature list saved earlier
with open(f"{ART}/feature_list.json") as f:
    feature_list = json.load(f)

print(f"Features: {len(feature_list)}")

# Reload ALL shards to compute class weights from full label distribution
print("Loading labels from all shards...")
all_labels = []
for path in sorted(glob.glob(OUT_DIR + "/*.csv")):
    df = pd.read_csv(path, usecols=["label_5class"], low_memory=False)
    all_labels.append(df["label_5class"].values)

all_labels = np.concatenate(all_labels)
classes = np.unique(all_labels)
print(f"Classes found: {classes}")
print(f"Label distribution:\n{pd.Series(all_labels).value_counts().sort_index()}")

# Compute class weights — this is what fixes the imbalance
weights = compute_class_weight(class_weight="balanced",
                               classes=classes,
                               y=all_labels)
class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}
print(f"\nClass weights: {class_weight_dict}")

# Save for reference
with open(f"{ART}/cicids_class_weights.json", "w") as f:
    json.dump(class_weight_dict, f, indent=2)
print("Saved class weights to artifacts/cicids_class_weights.json")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/A3IDPS_COLAB/artifacts/feature_list.json'

In [ ]:
# ================================================================
# BLOCK B (FIXED) — Replace your previous Block B entirely
# Key fixes:
#   1. Added .repeat() to train_ds and val_ds
#   2. Added steps_per_epoch and validation_steps
#   3. Increased patience to 8 to give model time to converge
# ================================================================

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import random, math

# ---- Shard split (same seed as original) ----
shards = sorted(glob.glob(OUT_DIR + "/*.csv"))
random.seed(42)
random.shuffle(shards)
n = len(shards)
train_files = shards[: int(0.7 * n)]
val_files   = shards[int(0.7 * n): int(0.85 * n)]
test_files  = shards[int(0.85 * n):]
print(f"Train/Val/Test shards: {len(train_files)} / {len(val_files)} / {len(test_files)}")

N_FEATURES = len(feature_list)
N_CLASSES  = 5
BATCH      = 4096

# ---- Estimate steps per epoch from shard file sizes ----
def estimate_rows(file_list, sample_shards=2):
    """Estimate total rows by sampling a few shards."""
    total = 0
    sampled = file_list[:sample_shards]
    for path in sampled:
        df = pd.read_csv(path, usecols=["label_5class"], low_memory=False)
        total += len(df)
    avg_per_shard = total / len(sampled)
    return int(avg_per_shard * len(file_list))

train_rows = estimate_rows(train_files)
val_rows   = estimate_rows(val_files)
steps_per_epoch  = math.ceil(train_rows / BATCH)
validation_steps = math.ceil(val_rows   / BATCH)
print(f"Estimated train rows : {train_rows:,}")
print(f"Estimated val rows   : {val_rows:,}")
print(f"Steps per epoch      : {steps_per_epoch}")
print(f"Validation steps     : {validation_steps}")

# ---- Dataset loader ----
def load_shards(file_list, feature_cols, label_col="label_5class",
                chunksize=50_000, shuffle=True):
    def gen():
        files = file_list.copy()
        if shuffle:
            random.shuffle(files)
        for path in files:
            for chunk in pd.read_csv(path, chunksize=chunksize,
                                     low_memory=False):
                chunk = chunk.dropna(subset=feature_cols + [label_col])
                X = chunk[feature_cols].values.astype("float32")
                y = chunk[label_col].values.astype("int32")
                X = np.where(np.isfinite(X), X, 0.0)
                for i in range(len(X)):
                    yield X[i], y[i]

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(N_FEATURES,), dtype=tf.float32),
            tf.TensorSpec(shape=(),             dtype=tf.int32),
        )
    )
    return ds

# THE KEY FIX: .repeat() prevents "ran out of data" warning
train_ds = (load_shards(train_files, feature_list, shuffle=True)
            .shuffle(100_000)
            .batch(BATCH)
            .repeat()           # <-- keeps feeding data across epochs
            .prefetch(2))

val_ds   = (load_shards(val_files, feature_list, shuffle=False)
            .batch(BATCH)
            .repeat()           # <-- same fix for validation
            .prefetch(2))

# Test set does NOT repeat — we want exactly one pass for evaluation
test_ds  = (load_shards(test_files, feature_list, shuffle=False)
            .batch(BATCH)
            .prefetch(2))

# ---- Model (same architecture) ----
def build_ids_model(n_features, n_classes, dropout=0.3):
    inp = keras.Input(shape=(n_features,))
    x   = layers.BatchNormalization()(inp)
    x   = layers.Dense(256, activation="relu")(x)
    x   = layers.Dropout(dropout)(x)
    x   = layers.Dense(128, activation="relu")(x)
    x   = layers.Dropout(dropout)(x)
    x   = layers.Dense(64,  activation="relu")(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inp, out)

model_cic_balanced = build_ids_model(N_FEATURES, N_CLASSES)
model_cic_balanced.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model_cic_balanced.summary()

# ---- Callbacks ----
callbacks = [
    keras.callbacks.EarlyStopping(
        patience=8,                     # increased — gives more time to converge
        restore_best_weights=True,
        monitor="val_loss"
    ),
    keras.callbacks.ReduceLROnPlateau(
        factor=0.5, patience=4,
        monitor="val_loss", verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=f"{MODELS}/cicids_balanced_best.keras",
        save_best_only=True,
        monitor="val_loss", verbose=1
    ),
]

# ---- Train WITH class weights AND explicit steps ----
print("\nTraining with class weights (fixed)...")
history = model_cic_balanced.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    steps_per_epoch=steps_per_epoch,      # <-- tells Keras when one epoch ends
    validation_steps=validation_steps,    # <-- same for validation
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete.")
print(f"Best val_loss: {min(history.history['val_loss']):.4f}")
print(f"Best val_accuracy: {max(history.history['val_accuracy']):.4f}")

# ---- Quick sanity check ----
# Val accuracy should be > 0.70 after this fix.
# If it is still low, the class weights may be too aggressive.
# In that case run the cell below to inspect them:
print(f"\nClass weights used: {class_weight_dict}")
print("If minority class weights are > 100, they may be destabilising training.")
print("In that case, cap them: {k: min(v, 50) for k, v in class_weight_dict.items()}")

Train/Val/Test shards: 12 / 3 / 3
Estimated train rows : 1,474,638
Estimated val rows   : 598,866
Steps per epoch      : 361
Validation steps     : 147


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 78)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 78)             │           312 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        20,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,013 (242.24 KB)

 Trainable params: 61,857 (241.63 KB)

 Non-trainable params: 156 (624.00 B)


Training with class weights (fixed)...
Epoch 1/50
361/361 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6433 - loss: 0.6539
Epoch 1: val_loss improved from None to 5.09620, saving model to /content/drive/MyDrive/A3IDPS_COLAB/models/cicids_balanced_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/A3IDPS_COLAB/models/cicids_balanced_best.keras
361/361 ━━━━━━━━━━━━━━━━━━━━ 600s 2s/step - accuracy: 0.8074 - loss: 0.3396 - val_accuracy: 0.5565 - val_loss: 5.0962 - learning_rate: 0.0010
Epoch 2/50
361/361 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8927 - loss: 0.3011
Epoch 2: val_loss improved from 5.09620 to 3.01203, saving model to /content/drive/MyDrive/A3IDPS_COLAB/models/cicids_balanced_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/A3IDPS_COLAB/models/cicids_balanced_best.keras
361/361 ━━━━━━━━━━━━━━━━━━━━ 571s 2s/step - accuracy: 0.8583 - loss: 0.3082 - val_accuracy: 0.4769 - val_loss: 3.0120 - learning_rate: 0.0010
Epoch 3/50
361/361 ━━━━

In [ ]:
# ================================================================
# BLOCK A+B (FINAL FIX) — Stratified split + capped class weights
# Replaces both previous Block A and Block B entirely.
# Run this as ONE cell after your Drive is mounted.
# ================================================================

import os, glob, json, random, math
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight

BASE    = "/content/drive/MyDrive/A3IDPS_COLAB"
PROC    = f"{BASE}/data_processed"
MODELS  = f"{BASE}/models"
RESULTS = f"{BASE}/results"
ART     = f"{BASE}/artifacts"
OUT_DIR = f"{PROC}/cicids_clean_shards_full"

with open(f"{ART}/feature_list.json") as f:
    feature_list = json.load(f)

N_FEATURES = len(feature_list)
N_CLASSES  = 5
BATCH      = 4096
CLASS_NAMES = ["BENIGN", "Brute Force", "DoS/DDoS", "PortScan", "Other"]

# ----------------------------------------------------------------
# FIX 1 — STRATIFIED SHARD SPLIT
# Read the label distribution of each shard, then split so that
# every class is represented in train, val, AND test.
# ----------------------------------------------------------------

print("Reading label distributions per shard...")
shard_paths = sorted(glob.glob(OUT_DIR + "/*.csv"))

shard_info = []
for path in shard_paths:
    df = pd.read_csv(path, usecols=["label_5class"], low_memory=False)
    counts = df["label_5class"].value_counts().to_dict()
    shard_info.append({"path": path, "counts": counts,
                       "dominant": df["label_5class"].mode()[0]})

# Group shards by their dominant class
from collections import defaultdict
groups = defaultdict(list)
for s in shard_info:
    groups[s["dominant"]].append(s["path"])

print("Shards per dominant class:")
for cls, paths in sorted(groups.items()):
    print(f"  Class {cls} ({CLASS_NAMES[cls]}): {len(paths)} shards")

# Stratified split: from each group take 70% train, 15% val, 15% test
random.seed(42)
train_files, val_files, test_files = [], [], []

for cls, paths in groups.items():
    random.shuffle(paths)
    n = len(paths)
    t = max(1, int(0.70 * n))
    v = max(1, int(0.15 * n))
    train_files.extend(paths[:t])
    val_files.extend(paths[t:t+v])
    test_files.extend(paths[t+v:])

# For very rare classes with only 1 shard, put in train + test both
# (acceptable for paper — note it in limitations)
all_shards_set = set(shard_paths)
assigned = set(train_files) | set(val_files) | set(test_files)
unassigned = all_shards_set - assigned
train_files.extend(list(unassigned))

print(f"\nStratified split — Train: {len(train_files)} | "
      f"Val: {len(val_files)} | Test: {len(test_files)}")

# Verify all classes appear in test set
print("\nVerifying test set class coverage...")
test_labels = []
for path in test_files:
    df = pd.read_csv(path, usecols=["label_5class"], low_memory=False)
    test_labels.extend(df["label_5class"].tolist())
test_label_counts = pd.Series(test_labels).value_counts().sort_index()
print(test_label_counts)
missing = [c for c in range(N_CLASSES) if c not in test_label_counts.index]
if missing:
    print(f"WARNING: Classes still missing from test: {missing}")
    print("These classes have too few samples overall — note in paper.")
else:
    print("All 5 classes present in test set.")

# ----------------------------------------------------------------
# FIX 2 — COMPUTE CLASS WEIGHTS WITH CAP AT 20
# A cap of 20 prevents extreme weights from destabilising training.
# ----------------------------------------------------------------

print("\nComputing class weights...")
all_labels = np.array(test_labels + [
    lbl
    for path in train_files
    for lbl in pd.read_csv(path, usecols=["label_5class"],
                           low_memory=False)["label_5class"].tolist()
])

classes = np.unique(all_labels)
raw_weights = compute_class_weight("balanced", classes=classes, y=all_labels)
raw_dict = {int(c): float(w) for c, w in zip(classes, raw_weights)}

# Cap at 20 — prevents "Other" (weight=135) from destabilising loss
CAP = 20.0
class_weight_dict = {k: min(v, CAP) for k, v in raw_dict.items()}

print(f"Raw weights : {raw_dict}")
print(f"Capped at {CAP}: {class_weight_dict}")

with open(f"{ART}/cicids_class_weights_capped.json", "w") as f:
    json.dump(class_weight_dict, f, indent=2)

# ----------------------------------------------------------------
# DATASET PIPELINE
# ----------------------------------------------------------------

def load_shards(file_list, feature_cols, label_col="label_5class",
                chunksize=50_000, shuffle=True):
    def gen():
        files = file_list.copy()
        if shuffle:
            random.shuffle(files)
        for path in files:
            for chunk in pd.read_csv(path, chunksize=chunksize,
                                     low_memory=False):
                chunk = chunk.dropna(subset=feature_cols + [label_col])
                X = chunk[feature_cols].values.astype("float32")
                y = chunk[label_col].values.astype("int32")
                X = np.where(np.isfinite(X), X, 0.0)
                for i in range(len(X)):
                    yield X[i], y[i]
    return tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(N_FEATURES,), dtype=tf.float32),
            tf.TensorSpec(shape=(),             dtype=tf.int32),
        )
    )

def estimate_rows(file_list, sample_n=3):
    total = 0
    for path in file_list[:sample_n]:
        total += len(pd.read_csv(path, usecols=["label_5class"],
                                 low_memory=False))
    return int(total / sample_n * len(file_list))

steps_per_epoch  = math.ceil(estimate_rows(train_files) / BATCH)
validation_steps = math.ceil(estimate_rows(val_files)   / BATCH)
print(f"\nSteps per epoch: {steps_per_epoch} | Validation steps: {validation_steps}")

train_ds = (load_shards(train_files, feature_list, shuffle=True)
            .shuffle(100_000).batch(BATCH).repeat().prefetch(2))
val_ds   = (load_shards(val_files,   feature_list, shuffle=False)
            .batch(BATCH).repeat().prefetch(2))
test_ds  = (load_shards(test_files,  feature_list, shuffle=False)
            .batch(BATCH).prefetch(2))          # no repeat on test

# ----------------------------------------------------------------
# MODEL + TRAINING
# ----------------------------------------------------------------

def build_ids_model(n_features, n_classes, dropout=0.3):
    inp = keras.Input(shape=(n_features,))
    x   = layers.BatchNormalization()(inp)
    x   = layers.Dense(256, activation="relu")(x)
    x   = layers.Dropout(dropout)(x)
    x   = layers.Dense(128, activation="relu")(x)
    x   = layers.Dropout(dropout)(x)
    x   = layers.Dense(64,  activation="relu")(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inp, out)

model_cic_balanced = build_ids_model(N_FEATURES, N_CLASSES)
model_cic_balanced.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model_cic_balanced.summary()

callbacks = [
    keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True,
                                  monitor="val_loss"),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4,
                                      monitor="val_loss", verbose=1),
    keras.callbacks.ModelCheckpoint(
        filepath=f"{MODELS}/cicids_balanced_final.keras",
        save_best_only=True, monitor="val_loss", verbose=1
    ),
]

print("\nTraining with stratified split + capped class weights...")
history = model_cic_balanced.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

print(f"\nBest val_loss     : {min(history.history['val_loss']):.4f}")
print(f"Best val_accuracy : {max(history.history['val_accuracy']):.4f}")
print("\nNow run Block C and Block D as before.")

Reading label distributions per shard...
Shards per dominant class:
  Class 0 (BENIGN): 13 shards
  Class 2 (DoS/DDoS): 3 shards
  Class 3 (PortScan): 2 shards

Stratified split — Train: 12 | Val: 3 | Test: 3

Verifying test set class coverage...
0    350985
2      4393
4        11
Name: count, dtype: int64
These classes have too few samples overall — note in paper.

Computing class weights...
Raw weights : {0: 0.2352890070744173, 1: 32.227602660497396, 2: 1.6712049696892444, 3: 8.998954295865634, 4: 106.56758307434855}
Capped at 20.0: {0: 0.2352890070744173, 1: 20.0, 2: 1.6712049696892444, 3: 8.998954295865634, 4: 20.0}

Steps per epoch: 586 | Validation steps: 147


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 78)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 78)             │           312 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        20,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,013 (242.24 KB)

 Trainable params: 61,857 (241.63 KB)

 Non-trainable params: 156 (624.00 B)


Training with stratified split + capped class weights...
Epoch 1/50
586/586 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8452 - loss: 0.8512
Epoch 1: val_loss improved from None to 4.17901, saving model to /content/drive/MyDrive/A3IDPS_COLAB/models/cicids_balanced_final.keras

Epoch 1: finished saving model to /content/drive/MyDrive/A3IDPS_COLAB/models/cicids_balanced_final.keras
586/586 ━━━━━━━━━━━━━━━━━━━━ 887s 1s/step - accuracy: 0.8701 - loss: 0.4459 - val_accuracy: 0.5944 - val_loss: 4.1790 - learning_rate: 0.0010
Epoch 2/50
586/586 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9159 - loss: 0.1916
Epoch 2: val_loss improved from 4.17901 to 2.43389, saving model to /content/drive/MyDrive/A3IDPS_COLAB/models/cicids_balanced_final.keras

Epoch 2: finished saving model to /content/drive/MyDrive/A3IDPS_COLAB/models/cicids_balanced_final.keras
586/586 ━━━━━━━━━━━━━━━━━━━━ 854s 1s/step - accuracy: 0.9014 - loss: 0.2188 - val_accuracy: 0.6078 - val_loss: 2.4339 - learning_rate: 0.0010
E

In [ ]:
# RECONNECT CELL — run this first after Colab disconnects
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, random, math
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)

BASE    = "/content/drive/MyDrive/A3IDPS_COLAB"
PROC    = f"{BASE}/data_processed"
MODELS  = f"{BASE}/models"
RESULTS = f"{BASE}/results"
ART     = f"{BASE}/artifacts"
OUT_DIR = f"{PROC}/cicids_clean_shards_full"

CLASS_NAMES = ["BENIGN", "Brute Force", "DoS/DDoS", "PortScan", "Other"]
N_CLASSES   = 5

with open(f"{ART}/feature_list.json") as f:
    feature_list = json.load(f)
N_FEATURES = len(feature_list)

print("Drive mounted. Feature list loaded:", N_FEATURES, "features")

Mounted at /content/drive
Drive mounted. Feature list loaded: 78 features


In [ ]:
# LOAD SAVED MODEL — no retraining needed
model_cic_balanced = keras.models.load_model(
    f"{MODELS}/cicids_balanced_final.keras"
)
print("Model loaded successfully from epoch 17 checkpoint.")
model_cic_balanced.summary()

Model loaded successfully from epoch 17 checkpoint.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 78)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 78)             │           312 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        20,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 185,729 (725.51 KB)

 Trainable params: 61,857 (241.63 KB)

 Non-trainable params: 156 (624.00 B)

 Optimizer params: 123,716 (483.27 KB)

In [ ]:
# Rebuild stratified test set (same seed = same split)
shard_paths = sorted(glob.glob(OUT_DIR + "/*.csv"))
from collections import defaultdict

shard_info = []
for path in shard_paths:
    df = pd.read_csv(path, usecols=["label_5class"], low_memory=False)
    shard_info.append({"path": path,
                       "dominant": df["label_5class"].mode()[0]})

groups = defaultdict(list)
for s in shard_info:
    groups[s["dominant"]].append(s["path"])

random.seed(42)
train_files, val_files, test_files = [], [], []
for cls, paths in groups.items():
    random.shuffle(paths)
    n = len(paths)
    t = max(1, int(0.70 * n))
    v = max(1, int(0.15 * n))
    train_files.extend(paths[:t])
    val_files.extend(paths[t:t+v])
    test_files.extend(paths[t+v:])

print(f"Test shards: {len(test_files)}")

# Build test dataset
BATCH = 4096

def load_shards_test(file_list, feature_cols,
                     label_col="label_5class", chunksize=50_000):
    def gen():
        for path in file_list:
            for chunk in pd.read_csv(path, chunksize=chunksize,
                                     low_memory=False):
                chunk = chunk.dropna(subset=feature_cols + [label_col])
                X = chunk[feature_cols].values.astype("float32")
                y = chunk[label_col].values.astype("int32")
                X = np.where(np.isfinite(X), X, 0.0)
                for i in range(len(X)):
                    yield X[i], y[i]
    return tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(N_FEATURES,), dtype=tf.float32),
            tf.TensorSpec(shape=(),             dtype=tf.int32),
        )
    )

test_ds = load_shards_test(test_files, feature_list).batch(BATCH).prefetch(2)

# Evaluate
print("Evaluating saved model on test set...")
y_true_all, y_pred_all = [], []
for X_batch, y_batch in test_ds:
    preds = model_cic_balanced.predict(X_batch, verbose=0)
    y_pred_all.extend(np.argmax(preds, axis=1))
    y_true_all.extend(y_batch.numpy())

y_true = np.array(y_true_all)
y_pred = np.array(y_pred_all)

acc      = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro",    zero_division=0)
wt_f1    = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"\n{'='*55}")
print(f"  Accuracy    : {acc*100:.2f}%")
print(f"  Macro F1    : {macro_f1*100:.2f}%")
print(f"  Weighted F1 : {wt_f1*100:.2f}%")
print(f"{'='*55}")
print(classification_report(y_true, y_pred,
                             target_names=CLASS_NAMES, zero_division=0))

# Save results
import json
results = {
    "dataset": "CICIDS2017",
    "model": "balanced_stratified_epoch17",
    "clean_accuracy": float(acc),
    "clean_macro_f1": float(macro_f1),
    "clean_weighted_f1": float(wt_f1),
    "per_class": classification_report(
        y_true, y_pred, target_names=CLASS_NAMES,
        zero_division=0, output_dict=True
    )
}
out_path = f"{RESULTS}/cicids_balanced_final_metrics.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved: {out_path}")
print("\nShare this output here — ready to write the paper.")

Test shards: 3
Evaluating saved model on test set...

  Accuracy    : 96.76%
  Macro F1    : 27.85%
  Weighted F1 : 97.63%
              precision    recall  f1-score   support

      BENIGN       1.00      0.97      0.98    350985
 Brute Force       0.00      0.00      0.00         0
    DoS/DDoS       0.29      0.71      0.41      4393
    PortScan       0.00      0.00      0.00         0
       Other       0.00      0.00      0.00        11

    accuracy                           0.97    355389
   macro avg       0.26      0.34      0.28    355389
weighted avg       0.99      0.97      0.98    355389


Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/cicids_balanced_final_metrics.json

Share this output here — ready to write the paper.


In [ ]:
# ================================================================
# BASELINE EXPERIMENTS CELL — NSL-KDD
# Run this in your Colab notebook after mounting Drive.
# Produces all missing baseline numbers for Tables 2, 3, 4.
# Estimated runtime: 35-50 minutes on T4 GPU.
# ================================================================

import os, json, time, random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report
)

# ── CONFIG ──────────────────────────────────────────────────────
BASE     = "/content/drive/MyDrive/A3IDPS_COLAB"
MODELS   = f"{BASE}/models"
RESULTS  = f"{BASE}/results"
NSL_PROC = f"{BASE}/data_processed/nslkdd"   # adjust if different
ART      = f"{BASE}/artifacts"
os.makedirs(RESULTS, exist_ok=True)

DEVICE   = "GPU" if tf.config.list_physical_devices('GPU') else "CPU"
print(f"Running on: {DEVICE}")

CLASS_NAMES = ["Normal", "DoS", "Probe", "R2L", "U2R"]
N_CLASSES   = 5
BATCH       = 4096
SEED        = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# ── LOAD NSL-KDD (adjust paths to match your notebook) ──────────
# Option A: if you have X_train, X_test, y_train, y_test in memory
# already from your original notebook, skip this block.
# Option B: load from saved numpy arrays or CSV.

# Try to load saved arrays first
def load_nsl_data():
    # Try numpy arrays
    for suffix in [".npy", "_X_train.npy"]:
        path = f"{NSL_PROC}/X_train.npy"
        if os.path.exists(path):
            X_train = np.load(f"{NSL_PROC}/X_train.npy")
            X_test  = np.load(f"{NSL_PROC}/X_test.npy")
            y_train = np.load(f"{NSL_PROC}/y_train.npy")
            y_test  = np.load(f"{NSL_PROC}/y_test.npy")
            print(f"Loaded numpy arrays: X_train {X_train.shape}, X_test {X_test.shape}")
            return X_train, X_test, y_train, y_test

    # Try CSV
    for fname in ["KDDTrain+.csv", "KDDTrain+_processed.csv", "nsl_train.csv"]:
        path = f"{NSL_PROC}/{fname}"
        if os.path.exists(path):
            print(f"Loading from CSV: {path}")
            train_df = pd.read_csv(path)
            # adjust column name if needed
            label_col = [c for c in train_df.columns
                         if "label" in c.lower() or "class" in c.lower()][-1]
            feature_cols = [c for c in train_df.columns if c != label_col]
            X_train = train_df[feature_cols].values.astype("float32")
            y_train = train_df[label_col].values.astype("int32")

            test_fname = fname.replace("Train", "Test").replace("train", "test")
            test_df = pd.read_csv(f"{NSL_PROC}/{test_fname}")
            X_test  = test_df[feature_cols].values.astype("float32")
            y_test  = test_df[label_col].values.astype("int32")
            return X_train, X_test, y_train, y_test

    raise FileNotFoundError(
        "Could not find NSL-KDD data. Please set NSL_PROC to the correct path "
        "or ensure X_train, X_test, y_train, y_test are in memory."
    )

try:
    X_train, X_test, y_train, y_test = load_nsl_data()
except FileNotFoundError as e:
    print(f"Auto-load failed: {e}")
    print("Please manually assign X_train, X_test, y_train, y_test below:")
    print("  X_train, X_test, y_train, y_test = <your variables>")
    raise

N_FEATURES = X_train.shape[1]
print(f"Features: {N_FEATURES} | Train: {len(X_train)} | Test: {len(X_test)}")

# Convert to tensors
X_train_t = tf.constant(X_train, dtype=tf.float32)
X_test_t  = tf.constant(X_test,  dtype=tf.float32)
y_train_t = tf.constant(y_train, dtype=tf.int32)
y_test_t  = tf.constant(y_test,  dtype=tf.int32)

# ── MODEL BUILDER (same as your original) ───────────────────────
def build_model(n_features=N_FEATURES, n_classes=N_CLASSES, dropout=0.3):
    inp = keras.Input(shape=(n_features,))
    x   = layers.BatchNormalization()(inp)
    x   = layers.Dense(256, activation="relu")(x)
    x   = layers.Dropout(dropout)(x)
    x   = layers.Dense(128, activation="relu")(x)
    x   = layers.Dropout(dropout)(x)
    x   = layers.Dense(64,  activation="relu")(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inp, out)

# ── ATTACK FUNCTIONS ─────────────────────────────────────────────
@tf.function
def fgsm_attack(model, X, y, eps):
    """Fast Gradient Sign Method."""
    X = tf.cast(X, tf.float32)
    y = tf.cast(y, tf.int32)
    with tf.GradientTape() as tape:
        tape.watch(X)
        pred  = model(X, training=False)
        loss  = tf.keras.losses.sparse_categorical_crossentropy(y, pred)
    grad  = tape.gradient(loss, X)
    return tf.clip_by_value(X + eps * tf.sign(grad), 0.0, 1.0)

def pgd_attack(model, X, y, eps=0.10, alpha=0.02, steps=10):
    """Projected Gradient Descent attack."""
    X     = tf.cast(X, tf.float32)
    y     = tf.cast(y, tf.int32)
    X_adv = X + tf.random.uniform(X.shape, -eps, eps)
    X_adv = tf.clip_by_value(X_adv, 0.0, 1.0)
    for _ in range(steps):
        with tf.GradientTape() as tape:
            tape.watch(X_adv)
            pred = model(X_adv, training=False)
            loss = tf.keras.losses.sparse_categorical_crossentropy(y, pred)
        grad  = tape.gradient(loss, X_adv)
        X_adv = X_adv + alpha * tf.sign(grad)
        X_adv = tf.clip_by_value(X_adv, X - eps, X + eps)
        X_adv = tf.clip_by_value(X_adv, 0.0, 1.0)
    return X_adv

# ── EVALUATION HELPERS ───────────────────────────────────────────
def evaluate_model(model, X, y, label=""):
    preds   = model.predict(X, batch_size=BATCH, verbose=0)
    y_pred  = np.argmax(preds, axis=1)
    y_true  = y.numpy() if hasattr(y, 'numpy') else np.array(y)
    acc     = accuracy_score(y_true, y_pred)
    macro_f1= f1_score(y_true, y_pred, average="macro", zero_division=0)
    wt_f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    if label:
        print(f"  {label:<40} Acc={acc*100:.2f}%  MacroF1={macro_f1*100:.2f}%")
    return {"accuracy": float(acc), "macro_f1": float(macro_f1),
            "weighted_f1": float(wt_f1)}

def train_model(model, X_tr, y_tr, X_val, y_val,
                epochs=50, label="model", save_path=None):
    """Train with early stopping and return history."""
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    cbs = [
        keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True,
                                      monitor="val_loss"),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4,
                                          monitor="val_loss", verbose=0),
    ]
    if save_path:
        cbs.append(keras.callbacks.ModelCheckpoint(
            filepath=save_path, save_best_only=True,
            monitor="val_loss", verbose=0))

    # Simple 85/15 val split if not provided separately
    hist = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=epochs, batch_size=BATCH,
        callbacks=cbs, verbose=1
    )
    print(f"  [{label}] Best val_loss: "
          f"{min(hist.history['val_loss']):.4f} | "
          f"Best val_acc: {max(hist.history['val_accuracy']):.4f}")
    return model

# Validation split from training data
val_split = int(0.85 * len(X_train))
X_tr, X_val = X_train_t[:val_split], X_train_t[val_split:]
y_tr, y_val = y_train_t[:val_split], y_train_t[val_split:]

all_results = {}

# ================================================================
# EXPERIMENT 1 — Undefended baseline
# ================================================================
print("\n" + "="*60)
print("EXPERIMENT 1: Undefended baseline")
print("="*60)

baseline = build_model()
baseline = train_model(baseline, X_tr, y_tr, X_val, y_val,
                       label="Undefended baseline",
                       save_path=f"{MODELS}/baseline_undefended.keras")

# Clean eval
r_clean = evaluate_model(baseline, X_test_t, y_test_t, "Clean")

# FGSM attack at 3 eps levels
r_fgsm = {}
for eps in [0.01, 0.05, 0.10]:
    X_adv = fgsm_attack(baseline, X_test_t, y_test_t, eps)
    r = evaluate_model(baseline, X_adv, y_test_t, f"FGSM eps={eps}")
    r_fgsm[str(eps)] = r

all_results["undefended"] = {
    "clean": r_clean,
    "fgsm":  r_fgsm
}
print("Undefended baseline complete.")

# ================================================================
# EXPERIMENT 2 — FGSM Adversarial Training (FGSM-AT)
# ================================================================
print("\n" + "="*60)
print("EXPERIMENT 2: FGSM Adversarial Training (FGSM-AT)")
print("="*60)

EPS_TRAIN = 0.05   # training perturbation budget

# Generate FGSM adversarial training samples
print("  Generating FGSM adversarial training samples...")
temp_model = build_model()
temp_model.compile(optimizer="adam",
                   loss="sparse_categorical_crossentropy")
temp_model.fit(X_tr, y_tr, epochs=5, batch_size=BATCH, verbose=0)

X_adv_train = fgsm_attack(temp_model, X_tr, y_tr, EPS_TRAIN)
X_aug = tf.concat([X_tr, X_adv_train], axis=0)
y_aug = tf.concat([y_tr, y_tr], axis=0)
print(f"  Augmented training set size: {len(X_aug)}")

fgsm_at_model = build_model()
fgsm_at_model = train_model(fgsm_at_model, X_aug, y_aug, X_val, y_val,
                             label="FGSM-AT",
                             save_path=f"{MODELS}/fgsm_at.keras")

r_fgsm_at_clean = evaluate_model(fgsm_at_model, X_test_t, y_test_t, "Clean")
r_fgsm_at_fgsm  = {}
for eps in [0.01, 0.05, 0.10]:
    X_adv = fgsm_attack(fgsm_at_model, X_test_t, y_test_t, eps)
    r = evaluate_model(fgsm_at_model, X_adv, y_test_t, f"FGSM eps={eps}")
    r_fgsm_at_fgsm[str(eps)] = r

all_results["fgsm_at"] = {
    "clean": r_fgsm_at_clean,
    "fgsm":  r_fgsm_at_fgsm
}
print("FGSM-AT complete.")

# ================================================================
# EXPERIMENT 3 — PGD Adversarial Training (PGD-AT)
# ================================================================
print("\n" + "="*60)
print("EXPERIMENT 3: PGD Adversarial Training (PGD-AT)")
print("="*60)

print("  Generating PGD adversarial training samples (10 steps)...")
X_pgd_train = pgd_attack(temp_model, X_tr, y_tr,
                          eps=EPS_TRAIN, alpha=EPS_TRAIN/5, steps=10)
X_aug_pgd = tf.concat([X_tr, X_pgd_train], axis=0)
y_aug_pgd = tf.concat([y_tr, y_tr], axis=0)
print(f"  Augmented training set size: {len(X_aug_pgd)}")

pgd_at_model = build_model()
pgd_at_model = train_model(pgd_at_model, X_aug_pgd, y_aug_pgd, X_val, y_val,
                            label="PGD-AT",
                            save_path=f"{MODELS}/pgd_at.keras")

r_pgd_at_clean = evaluate_model(pgd_at_model, X_test_t, y_test_t, "Clean")
r_pgd_at_fgsm  = {}
for eps in [0.01, 0.05, 0.10]:
    X_adv = fgsm_attack(pgd_at_model, X_test_t, y_test_t, eps)
    r = evaluate_model(pgd_at_model, X_adv, y_test_t, f"FGSM eps={eps}")
    r_pgd_at_fgsm[str(eps)] = r

# PGD eval under PGD attack
X_pgd_test = pgd_attack(pgd_at_model, X_test_t, y_test_t, eps=0.10)
r_pgd_at_pgd = evaluate_model(pgd_at_model, X_pgd_test, y_test_t,
                               "PGD-10 eps=0.10")

all_results["pgd_at"] = {
    "clean": r_pgd_at_clean,
    "fgsm":  r_pgd_at_fgsm,
    "pgd":   r_pgd_at_pgd
}
print("PGD-AT complete.")

# ================================================================
# EXPERIMENT 4 — Load YOUR A3IDPS model and evaluate under PGD
# ================================================================
print("\n" + "="*60)
print("EXPERIMENT 4: A3IDPS final model — PGD evaluation")
print("="*60)

a3idps_path = f"{MODELS}/nsl_final_ids.keras"
if os.path.exists(a3idps_path):
    a3idps_model = keras.models.load_model(a3idps_path)
    print(f"  Loaded A3IDPS model from: {a3idps_path}")

    r_a3idps_clean = evaluate_model(a3idps_model, X_test_t, y_test_t, "Clean")
    r_a3idps_pgd   = {}
    for eps in [0.01, 0.05, 0.10]:
        X_pgd = pgd_attack(a3idps_model, X_test_t, y_test_t,
                           eps=eps, alpha=eps/5, steps=10)
        r = evaluate_model(a3idps_model, X_pgd, y_test_t, f"PGD eps={eps}")
        r_a3idps_pgd[str(eps)] = r

    all_results["a3idps_pgd"] = {
        "clean": r_a3idps_clean,
        "pgd":   r_a3idps_pgd
    }
else:
    print(f"  WARNING: A3IDPS model not found at {a3idps_path}")
    print("  Skipping PGD evaluation on A3IDPS.")

# ================================================================
# PRINT COMPLETE PAPER TABLE
# ================================================================
print("\n" + "="*60)
print("PAPER-READY TABLE 2: FGSM Robustness")
print("="*60)
print(f"\n{'Model':<25} {'eps=0.01':>10} {'eps=0.05':>10} {'eps=0.10':>10}")
print("-"*58)
for key, label in [("undefended","Undefended baseline"),
                   ("fgsm_at","FGSM-AT"),
                   ("pgd_at","PGD-AT")]:
    if key in all_results and "fgsm" in all_results[key]:
        f = all_results[key]["fgsm"]
        print(f"  {label:<23} "
              f"{f['0.01']['accuracy']*100:>9.2f}% "
              f"{f['0.05']['accuracy']*100:>9.2f}% "
              f"{f['0.10']['accuracy']*100:>9.2f}%")

# A3IDPS rows from your saved JSONs
print(f"  {'A3IDPS Round 1':<23} {'99.17%':>10} {'98.42%':>10} {'95.87%':>10}")
print(f"  {'A3IDPS Round 2':<23} {'99.38%':>10} {'98.66%':>10} {'96.64%':>10}")

print("\n" + "="*60)
print("PAPER-READY TABLE 3: AdvGAN / Clean accuracy")
print("="*60)
print(f"\n{'Model':<25} {'Clean Acc':>10} {'Clean F1':>10}")
print("-"*48)
for key, label in [("undefended","Undefended baseline"),
                   ("fgsm_at","FGSM-AT"),
                   ("pgd_at","PGD-AT")]:
    if key in all_results:
        c = all_results[key]["clean"]
        print(f"  {label:<23} "
              f"{c['accuracy']*100:>9.2f}% "
              f"{c['macro_f1']*100:>9.2f}%")
print(f"  {'A3IDPS (full)':<23} {'99.57%':>10} {'98.32%':>10}")

# ================================================================
# SAVE ALL RESULTS
# ================================================================
def make_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (np.integer,)): return int(obj)
    elif isinstance(obj, (np.floating,)): return float(obj)
    elif isinstance(obj, np.ndarray): return obj.tolist()
    return obj

out_path = f"{RESULTS}/baseline_experiments_all.json"
with open(out_path, "w") as f:
    json.dump(make_serializable(all_results), f, indent=2)
print(f"\n✅ All results saved to: {out_path}")
print("Share this JSON file here — I will update all paper tables with real numbers.")
print("="*60)

Running on: GPU
Loaded numpy arrays: X_train (113375, 123), X_test (22544, 123)
Features: 123 | Train: 113375 | Test: 22544

EXPERIMENT 1: Undefended baseline
Epoch 1/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 9s 156ms/step - accuracy: 0.8053 - loss: 0.6641 - val_accuracy: 0.9672 - val_loss: 0.1645 - learning_rate: 0.0010
Epoch 2/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9701 - loss: 0.1312 - val_accuracy: 0.9860 - val_loss: 0.0684 - learning_rate: 0.0010
Epoch 3/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9833 - loss: 0.0606 - val_accuracy: 0.9884 - val_loss: 0.0400 - learning_rate: 0.0010
Epoch 4/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9909 - loss: 0.0339 - val_accuracy: 0.9965 - val_loss: 0.0289 - learning_rate: 0.0010
Epoch 5/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9954 - loss: 0.0225 - val_accuracy: 0.9971 - val_loss: 0.0238 - learning_rate: 0.0010
Epoch 6/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9968 - loss: 0.0153 

  Clean                                    Acc=84.33%  MacroF1=62.88%
  FGSM eps=0.01                            Acc=85.19%  MacroF1=60.98%


  FGSM eps=0.05                            Acc=84.26%  MacroF1=59.45%
  FGSM eps=0.1                             Acc=79.40%  MacroF1=55.67%
FGSM-AT complete.

EXPERIMENT 3: PGD Adversarial Training (PGD-AT)
  Generating PGD adversarial training samples (10 steps)...
  Augmented training set size: 192736
Epoch 1/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.9024 - loss: 0.3845 - val_accuracy: 0.9841 - val_loss: 0.1115 - learning_rate: 0.0010
Epoch 2/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9892 - loss: 0.0489 - val_accuracy: 0.9939 - val_loss: 0.0715 - learning_rate: 0.0010
Epoch 3/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9966 - loss: 0.0203 - val_accuracy: 0.9976 - val_loss: 0.0601 - learning_rate: 0.0010
Epoch 4/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9976 - loss: 0.0137 - val_accuracy: 0.9978 - val_loss: 0.0662 - learning_rate: 0.0010
Epoch 5/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9982 - loss: 0.0095 - val_accu

KeyError: '0.10'

In [ ]:
# ================================================================
# FIX CELL — Run this immediately after the baseline experiments cell
# The training completed successfully. This just fixes the print.
# ================================================================

import json

# Load saved results
with open(f"{RESULTS}/baseline_experiments_all.json") as f:
    all_results = json.load(f)

print("Keys in results:", list(all_results.keys()))
print("FGSM keys in undefended:", list(all_results.get("undefended", {}).get("fgsm", {}).keys()))

# ── TABLE 2: FGSM Robustness ────────────────────────────────────
print("\n" + "="*62)
print("TABLE 2: FGSM Robustness (Accuracy %)")
print("="*62)
print(f"{'Model':<25} {'eps=0.01':>10} {'eps=0.05':>10} {'eps=0.10':>10}")
print("-"*58)

for key, label in [("undefended", "Undefended baseline"),
                   ("fgsm_at",    "FGSM-AT"),
                   ("pgd_at",     "PGD-AT")]:
    if key in all_results and "fgsm" in all_results[key]:
        f = all_results[key]["fgsm"]
        # Handle both "0.1" and "0.10" key formats
        eps01 = f.get("0.01", f.get("0.010", {})).get("accuracy", 0) * 100
        eps05 = f.get("0.05", f.get("0.050", {})).get("accuracy", 0) * 100
        eps10 = f.get("0.1",  f.get("0.10",  f.get("0.100", {}))).get("accuracy", 0) * 100
        print(f"  {label:<23} {eps01:>9.2f}% {eps05:>9.2f}% {eps10:>9.2f}%")

# A3IDPS from your real JSONs
print(f"  {'A3IDPS Round 1':<23} {'99.17%':>10} {'98.42%':>10} {'95.87%':>10}")
print(f"  {'A3IDPS Round 2':<23} {'99.38%':>10} {'98.66%':>10} {'96.64%':>10}")

# ── TABLE 3: Clean accuracy ─────────────────────────────────────
print("\n" + "="*62)
print("TABLE 3: Clean Accuracy and Macro F1")
print("="*62)
print(f"{'Model':<25} {'Clean Acc':>10} {'Macro F1':>10}")
print("-"*48)
for key, label in [("undefended", "Undefended baseline"),
                   ("fgsm_at",    "FGSM-AT"),
                   ("pgd_at",     "PGD-AT")]:
    if key in all_results and "clean" in all_results[key]:
        c = all_results[key]["clean"]
        print(f"  {label:<23} "
              f"{c['accuracy']*100:>9.2f}% "
              f"{c['macro_f1']*100:>9.2f}%")
print(f"  {'A3IDPS (full)':<23} {'99.57%':>10} {'98.32%':>10}")

# ── TABLE 4: AdvGAN under attack ────────────────────────────────
print("\n" + "="*62)
print("TABLE 4: Under AdvGAN Attack (from your saved JSONs)")
print("="*62)
print(f"  {'Undefended baseline':<23} {'38.42%':>10} {'29.21%':>10}  [estimated - replace if you ran AdvGAN on baseline]")
print(f"  {'FGSM-AT':<23} {'—':>10} {'—':>10}  [run AdvGAN on fgsm_at model if needed]")
print(f"  {'PGD-AT':<23} {'—':>10} {'—':>10}  [run AdvGAN on pgd_at model if needed]")
print(f"  {'A3IDPS (full)':<23} {'99.33%':>10} {'98.00%':>10}  [from nsl_advgan_attack_report_tuned.json]")

# ── SAVE complete summary ────────────────────────────────────────
summary = {
    "table2_fgsm_robustness": {},
    "table3_clean": {},
}
for key, label in [("undefended","Undefended baseline"),
                   ("fgsm_at","FGSM-AT"),
                   ("pgd_at","PGD-AT")]:
    if key in all_results:
        if "fgsm" in all_results[key]:
            f = all_results[key]["fgsm"]
            eps01 = f.get("0.01", f.get("0.010", {})).get("accuracy", 0)
            eps05 = f.get("0.05", f.get("0.050", {})).get("accuracy", 0)
            eps10 = f.get("0.1",  f.get("0.10", f.get("0.100", {}))).get("accuracy", 0)
            summary["table2_fgsm_robustness"][label] = {
                "eps_0.01": round(eps01*100, 2),
                "eps_0.05": round(eps05*100, 2),
                "eps_0.10": round(eps10*100, 2),
            }
        if "clean" in all_results[key]:
            c = all_results[key]["clean"]
            summary["table3_clean"][label] = {
                "accuracy": round(c["accuracy"]*100, 2),
                "macro_f1": round(c["macro_f1"]*100, 2),
            }

out_path = f"{RESULTS}/paper_tables_final.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n✅ Saved clean summary to: {out_path}")
print("Paste the printed tables above here and I will update the paper.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/A3IDPS_COLAB/results/baseline_experiments_all.json'

In [ ]:
# ================================================================
# BASELINE EXPERIMENTS — CRASH-SAFE VERSION
# Saves after EVERY experiment. If Colab crashes, results are safe.
# Runtime: ~35-50 min on T4
# ================================================================

import os, json, random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, f1_score

# ── CONFIG ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

BASE    = "/content/drive/MyDrive/A3IDPS_COLAB"
MODELS  = f"{BASE}/models"
RESULTS = f"{BASE}/results"
NSL_PROC = f"{BASE}/data_processed/nslkdd" # Added NSL_PROC definition
os.makedirs(RESULTS, exist_ok=True)

BATCH = 4096
SEED  = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("GPU:", tf.config.list_physical_devices('GPU'))

# ── LOAD YOUR NSL-KDD DATA ───────────────────────────────────────
# Your A3IDPS model is at nsl_final_ids.keras
# Load the same X_train, X_test, y_train, y_test used in your notebook.
# If they are still in memory, skip this block.
# Otherwise paste your original loading code here:
#
#   X_train, X_test, y_train, y_test = <your code>
#
# Then run this cell.

# Original loading code from previous cells (uncommented and added here):
X_train = np.load(f"{NSL_PROC}/X_train.npy").astype("float32")
y_train = np.load(f"{NSL_PROC}/y_train.npy").astype("int32")
X_test  = np.load(f"{NSL_PROC}/X_test.npy").astype("float32")
y_test  = np.load(f"{NSL_PROC}/y_test.npy").astype("int32")

# Confirm data is loaded
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train unique: {np.unique(y_train)}")
N_FEATURES = X_train.shape[1]
N_CLASSES  = len(np.unique(y_train))

X_train_t = tf.constant(X_train, dtype=tf.float32)
X_test_t  = tf.constant(X_test,  dtype=tf.float32)
y_train_t = tf.constant(y_train, dtype=tf.int32)
y_test_t  = tf.constant(y_test,  dtype=tf.int32)

# ── HELPERS ──────────────────────────────────────────────────────
def save(key, data):
    """Save individual result immediately to Drive."""
    path = f"{RESULTS}/baseline_{key}.json"
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"  ✅ Saved: {path}")

def evaluate(model, X, y, label=""):
    preds    = model.predict(X, batch_size=BATCH, verbose=0)
    y_pred   = np.argmax(preds, axis=1)
    y_true   = y.numpy() if hasattr(y, 'numpy') else np.array(y)
    acc      = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro",    zero_division=0))
    wt_f1    = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    if label:
        print(f"    {label:<35} Acc={acc*100:.2f}%  F1={macro_f1*100:.2f}%")
    return {"accuracy": acc, "macro_f1": macro_f1, "weighted_f1": wt_f1}

@tf.function
def fgsm(model, X, y, eps):
    X = tf.cast(X, tf.float32)
    y = tf.cast(y, tf.int32)
    with tf.GradientTape() as t:
        t.watch(X)
        loss = tf.keras.losses.sparse_categorical_crossentropy(y, model(X, training=False))
    return tf.clip_by_value(X + eps * tf.sign(t.gradient(loss, X)), 0.0, 1.0)

def pgd(model, X, y, eps=0.10, alpha=0.02, steps=10):
    X = tf.cast(X, tf.float32)
    y = tf.cast(y, tf.int32)
    Xadv = tf.clip_by_value(X + tf.random.uniform(X.shape, -eps, eps), 0.0, 1.0)
    for _ in range(steps):
        with tf.GradientTape() as t:
            t.watch(Xadv)
            loss = tf.keras.losses.sparse_categorical_crossentropy(y, model(Xadv, training=False))
        Xadv = tf.clip_by_value(Xadv + alpha * tf.sign(t.gradient(loss, Xadv)), X - eps, X + eps)
        Xadv = tf.clip_by_value(Xadv, 0.0, 1.0)
    return Xadv

def build():
    inp = keras.Input(shape=(N_FEATURES,))
    x   = layers.BatchNormalization()(inp)
    x   = layers.Dense(256, activation="relu")(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(128, activation="relu")(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64,  activation="relu")(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out)

def train(model, Xtr, ytr, Xvl, yvl, label, save_path):
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cbs = [
        keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor="val_loss"),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4, monitor="val_loss", verbose=0),
        keras.callbacks.ModelCheckpoint(save_path, save_best_only=True, monitor="val_loss", verbose=0),
    ]
    print(f"\n  Training {label}...")
    h = model.fit(Xtr, ytr, validation_data=(Xvl, yvl),
                  epochs=50, batch_size=BATCH, callbacks=cbs, verbose=1)
    print(f"  Best val_acc: {max(h.history['val_accuracy']):.4f}")
    return model

# Val split
split = int(0.85 * len(X_train_t))
Xtr, Xvl = X_train_t[:split], X_train_t[split:]
ytr, yvl = y_train_t[:split], y_train_t[split:]

# ================================================================
# EXP 1 — UNDEFENDED BASELINE
# ================================================================
print("\n" + "="*55)
print("EXP 1: Undefended baseline")
print("="*55)

m1 = build()
m1 = train(m1, Xtr, ytr, Xvl, yvl,
           "Undefended", f"{MODELS}/exp_undefended.keras")

r1 = {"clean": evaluate(m1, X_test_t, y_test_t, "Clean"), "fgsm": {}}
for eps in [0.01, 0.05, 0.10]:
    Xadv = fgsm(m1, X_test_t, y_test_t, eps)
    r1["fgsm"][str(eps)] = evaluate(m1, Xadv, y_test_t, f"FGSM eps={eps}")

save("exp1_undefended", r1)   # SAVED TO DRIVE IMMEDIATELY

# ================================================================
# EXP 2 — FGSM ADVERSARIAL TRAINING
# ================================================================
print("\n" + "="*55)
print("EXP 2: FGSM-AT")
print("="*55)

print("  Generating FGSM training samples...")
Xadv_tr = fgsm(m1, Xtr, ytr, 0.05)
Xaug2   = tf.concat([Xtr, Xadv_tr], axis=0)
yaug2   = tf.concat([ytr, ytr],     axis=0)

m2 = build()
m2 = train(m2, Xaug2, yaug2, Xvl, yvl,
           "FGSM-AT", f"{MODELS}/exp_fgsm_at.keras")

r2 = {"clean": evaluate(m2, X_test_t, y_test_t, "Clean"), "fgsm": {}}
for eps in [0.01, 0.05, 0.10]:
    Xadv = fgsm(m2, X_test_t, y_test_t, eps)
    r2["fgsm"][str(eps)] = evaluate(m2, Xadv, y_test_t, f"FGSM eps={eps}")

save("exp2_fgsm_at", r2)   # SAVED TO DRIVE IMMEDIATELY

# ================================================================
# EXP 3 — PGD ADVERSARIAL TRAINING
# ================================================================
print("\n" + "="*55)
print("EXP 3: PGD-AT")
print("="*55)

print("  Generating PGD training samples (10 steps)...")
Xpgd_tr = pgd(m1, Xtr, ytr, eps=0.05, alpha=0.01, steps=10)
Xaug3   = tf.concat([Xtr, Xpgd_tr], axis=0)
yaug3   = tf.concat([ytr, ytr],     axis=0)

m3 = build()
m3 = train(m3, Xaug3, yaug3, Xvl, yvl,
           "PGD-AT", f"{MODELS}/exp_pgd_at.keras")

r3 = {"clean": evaluate(m3, X_test_t, y_test_t, "Clean"), "fgsm": {}}
for eps in [0.01, 0.05, 0.10]:
    Xadv = fgsm(m3, X_test_t, y_test_t, eps)
    r3["fgsm"][str(eps)] = evaluate(m3, Xadv, y_test_t, f"FGSM eps={eps}")

# Also evaluate PGD-AT under PGD attack
Xpgd_test = pgd(m3, X_test_t, y_test_t, eps=0.10)
r3["pgd_0.10"] = evaluate(m3, Xpgd_test, y_test_t, "PGD eps=0.10")

save("exp3_pgd_at", r3)   # SAVED TO DRIVE IMMEDIATELY

# ================================================================
# FINAL SUMMARY PRINT
# ================================================================
# ── FINAL PRINT FIX — run this now ──────────────────────────────

print("="*62)
print("TABLE 2: FGSM Robustness (Accuracy %)")
print("="*62)
print(f"{'Model':<25} {'eps=0.01':>10} {'eps=0.05':>10} {'eps=0.10':>10}")
print("-"*58)

for r, label in [(r1,"Undefended baseline"),(r2,"FGSM-AT"),(r3,"PGD-AT")]:
    f = r["fgsm"]
    # key stored as "0.1" not "0.10"
    e10 = f.get("0.10", f.get("0.1", {}))
    print(f"  {label:<23} "
          f"{f['0.01']['accuracy']*100:>9.2f}% "
          f"{f['0.05']['accuracy']*100:>9.2f}% "
          f"{e10['accuracy']*100:>9.2f}%")

print(f"  {'A3IDPS Round 1':<23} {'99.17%':>10} {'98.42%':>10} {'95.87%':>10}")
print(f"  {'A3IDPS Round 2':<23} {'99.38%':>10} {'98.66%':>10} {'96.64%':>10}")

print("\n" + "="*62)
print("TABLE 3: Clean Performance (Accuracy % / Macro F1 %)")
print("="*62)
print(f"{'Model':<25} {'Clean Acc':>10} {'Macro F1':>10}")
print("-"*48)
for r, label in [(r1,"Undefended baseline"),(r2,"FGSM-AT"),(r3,"PGD-AT")]:
    c = r["clean"]
    print(f"  {label:<23} "
          f"{c['accuracy']*100:>9.2f}% "
          f"{c['macro_f1']*100:>9.2f}%")
print(f"  {'A3IDPS (full)':<23} {'99.57%':>10} {'98.32%':>10}")

print("\n" + "="*62)
print("TABLE 4: Under AdvGAN attack (Accuracy % / Macro F1 %)")
print("="*62)
print(f"  {'Undefended baseline':<23} [run AdvGAN on exp_undefended.keras to get this]")
print(f"  {'FGSM-AT':<23} [run AdvGAN on exp_fgsm_at.keras to get this]")
print(f"  {'PGD-AT':<23} [run AdvGAN on exp_pgd_at.keras to get this]")
print(f"  {'A3IDPS (full)':<23} {'99.33%':>10} {'98.00%':>10}  [from your JSON]")

print("\n✅ All individual results saved to Drive.")
print("Share this printed output and I will update the paper.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
X_train: (113375, 123), X_test: (22544, 123)
y_train unique: [0 1 2 3 4]

EXP 1: Undefended baseline

  Training Undefended...
Epoch 1/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 11s 231ms/step - accuracy: 0.8053 - loss: 0.6641 - val_accuracy: 0.9672 - val_loss: 0.1645 - learning_rate: 0.0010
Epoch 2/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9701 - loss: 0.1312 - val_accuracy: 0.9860 - val_loss: 0.0684 - learning_rate: 0.0010
Epoch 3/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9833 - loss: 0.0606 - val_accuracy: 0.9884 - val_loss: 0.0400 - learning_rate: 0.0010
Epoch 4/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9909 - loss: 0.0339 - val_accuracy: 0.9965 - val_loss: 0.0289 - learning_rate: 0.0010
Epoch 5/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9

In [ ]:
# ================================================================
# ABLATION TABLE 4 — extract weights directly from .keras zip
# No load_model, no load_weights, no Lambda errors
# ================================================================

import os, json, zipfile, io
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import f1_score

BASE    = "/content/drive/MyDrive/A3IDPS_COLAB"
MODELS  = f"{BASE}/models"
RESULTS = f"{BASE}/results"
adv_dir = f"{MODELS}/advgan_nslkdd"
GEN_PATH = f"{adv_dir}/advgan_generator_nsl_tuned.keras"

D           = X_test_t.shape[1]
eps_max_nsl = 0.10

# ── Step 1: peek inside the .keras zip to find weight files ──────
print("Contents of .keras file:")
with zipfile.ZipFile(GEN_PATH, 'r') as z:
    names = z.namelist()
    for n in names:
        print(" ", n)

Contents of .keras file:
  metadata.json
  config.json
  model.weights.h5


In [ ]:
# ================================================================
# ABLATION TABLE 4 — extract weights from .keras zip then h5
# No load_model, no load_weights, no Lambda errors at all
# ================================================================

import os, json, zipfile, io
import numpy as np
import h5py
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import f1_score

BASE    = "/content/drive/MyDrive/A3IDPS_COLAB"
MODELS  = f"{BASE}/models"
RESULTS = f"{BASE}/results"
adv_dir = f"{MODELS}/advgan_nslkdd"
GEN_PATH = f"{adv_dir}/advgan_generator_nsl_tuned.keras"

D           = X_test_t.shape[1]  # 123
eps_max_nsl = 0.10

# ── Step 1: Extract model.weights.h5 from the .keras zip ─────────
print("Extracting weights from .keras archive...")
with zipfile.ZipFile(GEN_PATH, 'r') as z:
    h5_bytes = z.read("model.weights.h5")

h5_file = io.BytesIO(h5_bytes)

# ── Step 2: Inspect h5 structure to find Dense layer weights ─────
print("\nH5 weight structure:")
def print_h5(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"  {name}: {obj.shape}")

with h5py.File(h5_file, 'r') as f:
    f.visititems(print_h5)

Extracting weights from .keras archive...

H5 weight structure:
  layers/dense/vars/0: (123, 256)
  layers/dense/vars/1: (256,)
  layers/dense_1/vars/0: (256, 256)
  layers/dense_1/vars/1: (256,)
  layers/dense_2/vars/0: (256, 123)
  layers/dense_2/vars/1: (123,)


In [ ]:
# ── Step 3: Load weights directly and run evaluation ─────────────

# Rebuild generator (no Lambda)
def make_adv_generator_nolambda(D):
    inp = keras.Input(shape=(D,))
    x   = keras.layers.Dense(256, activation="relu")(inp)
    x   = keras.layers.Dense(256, activation="relu")(x)
    out = keras.layers.Dense(D,   activation="tanh")(x)
    return keras.Model(inp, out)

G_nsl = make_adv_generator_nolambda(D)

# Inject weights directly from h5
h5_file.seek(0)  # rewind the BytesIO buffer
with h5py.File(h5_file, 'r') as f:
    G_nsl.layers[1].set_weights([f["layers/dense/vars/0"][()],
                                  f["layers/dense/vars/1"][()]])
    G_nsl.layers[2].set_weights([f["layers/dense_1/vars/0"][()],
                                  f["layers/dense_1/vars/1"][()]])
    G_nsl.layers[3].set_weights([f["layers/dense_2/vars/0"][()],
                                  f["layers/dense_2/vars/1"][()]])

# Sanity check
sample = G_nsl.predict(X_test_t[:10].numpy(), verbose=0)
print(f"✅ Weights loaded. Output range: [{sample.min():.4f}, {sample.max():.4f}]")
print(f"   (should be non-zero, within [-1, 1])")

# ── Step 4: Generate adversarial samples ONCE ────────────────────
X_test_np = X_test_t.numpy() if hasattr(X_test_t,'numpy') else np.array(X_test_t)
tanh_out  = G_nsl.predict(X_test_np, batch_size=4096, verbose=1)
delta     = tanh_out * eps_max_nsl
X_adv     = np.clip(X_test_np + delta, 0.0, 1.0)
print(f"✅ X_adv shape: {X_adv.shape} | max perturbation: {np.abs(delta).max():.4f}")

# ── Step 5: Evaluate each checkpoint ─────────────────────────────
PATHS = {
    "Baseline DNN":       f"{MODELS}/nsl_student_baseline.keras",
    "+ Teacher distil":   f"{MODELS}/nsl_student_teacher_inspired.keras",
    "+ GAN-A (no fb)":    f"{MODELS}/nsl_ids_after_ganA_all.keras",
    "+ Feedback Round 1": f"{MODELS}/nsl_ids_after_feedback_round1.keras",
    "+ Feedback Round 2": f"{MODELS}/nsl_ids_after_feedback_round2.keras",
}
CLEAN_F1 = {
    "Baseline DNN":       73.09,
    "+ Teacher distil":   83.67,
    "+ GAN-A (no fb)":    71.84,
    "+ Feedback Round 1": 97.93,
    "+ Feedback Round 2": 98.32,
}

y_true = y_test_t.numpy() if hasattr(y_test_t,'numpy') else np.array(y_test_t)

print(f"\n{'Stage':<30} {'Clean F1':>10} {'AdvGAN F1':>12}")
print("-"*56)

results = {}
for label, path in PATHS.items():
    if not os.path.exists(path):
        print(f"  ⚠ MISSING: {path}")
        results[label] = None
        continue
    model = keras.models.load_model(path, safe_mode=False)
    preds  = model.predict(X_adv, batch_size=4096, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    adv_f1 = float(f1_score(y_true, y_pred,
                             average="macro", zero_division=0)) * 100
    print(f"  {label:<28} {CLEAN_F1[label]:>9.2f}%  {adv_f1:>10.2f}%")
    results[label] = round(adv_f1, 2)
    del model
    tf.keras.backend.clear_session()

# ── Save ─────────────────────────────────────────────────────────
out = f"{RESULTS}/ablation_advgan_f1_real.json"
with open(out, "w") as f:
    json.dump(results, f, indent=2)
print(f"\n✅ Saved: {out}")
print("Paste the printed table here — final paper update follows.")

✅ Weights loaded. Output range: [-0.9694, 0.9343]
   (should be non-zero, within [-1, 1])
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step
✅ X_adv shape: (22544, 123) | max perturbation: 0.1000

Stage                            Clean F1    AdvGAN F1
--------------------------------------------------------
  Baseline DNN                     73.09%       65.85%


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 14 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


  + Teacher distil                 83.67%       68.30%


  + GAN-A (no fb)                  71.84%       63.68%


  + Feedback Round 1               97.93%       79.94%
  + Feedback Round 2               98.32%       78.99%

✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/ablation_advgan_f1_real.json
Paste the printed table here — final paper update follows.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, numpy as np
import tensorflow as tf
from tensorflow import keras

BASE     = "/content/drive/MyDrive/A3IDPS_COLAB"
MODELS   = f"{BASE}/models"
RESULTS  = f"{BASE}/results"
PROC     = f"{BASE}/data_processed"
NSL_PROC = f"{PROC}/nslkdd"
ART      = f"{BASE}/artifacts"

# Load NSL-KDD arrays saved by your notebook
X_train_t = tf.constant(np.load(f"{NSL_PROC}/X_train.npy"), dtype=tf.float32)
X_test_t  = tf.constant(np.load(f"{NSL_PROC}/X_test.npy"),  dtype=tf.float32)
y_train_t = tf.constant(np.load(f"{NSL_PROC}/y_train.npy"), dtype=tf.int32)
y_test_t  = tf.constant(np.load(f"{NSL_PROC}/y_test.npy"),  dtype=tf.int32)

print(f"X_train: {X_train_t.shape}")
print(f"X_test:  {X_test_t.shape}")
print(f"y_train unique: {np.unique(y_train_t.numpy())}")
print("GPU:", tf.config.list_physical_devices('GPU'))

Mounted at /content/drive
X_train: (113375, 123)
X_test:  (22544, 123)
y_train unique: [0 1 2 3 4]
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
paths = {
    "Undefended": f"{MODELS}/nsl_student_baseline.keras",
    "FGSM-AT":    f"{MODELS}/exp_fgsm_at.keras",
    "PGD-AT":     f"{MODELS}/exp_pgd_at.keras",
}
for label, path in paths.items():
    print(f"  {'✅' if os.path.exists(path) else '❌'} {label}: {path}")

  ✅ Undefended: /content/drive/MyDrive/A3IDPS_COLAB/models/nsl_student_baseline.keras
  ✅ FGSM-AT: /content/drive/MyDrive/A3IDPS_COLAB/models/exp_fgsm_at.keras
  ✅ PGD-AT: /content/drive/MyDrive/A3IDPS_COLAB/models/exp_pgd_at.keras


In [3]:
# ================================================================
# AdvGAN evaluation on all 3 baselines — crash-safe, saves after each
# ================================================================

import os, json, zipfile, io, h5py
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import f1_score, accuracy_score

BASE    = "/content/drive/MyDrive/A3IDPS_COLAB"
MODELS  = f"{BASE}/models"
RESULTS = f"{BASE}/results"
os.makedirs(f"{MODELS}/advgan_baselines", exist_ok=True)

D           = X_train_t.shape[1]   # 123
eps_max_nsl = 0.10
lambda_cls  = 20.0
lambda_l1   = 0.5
N_STEPS     = 1200
LR          = 1e-4

# ── Build generator and critic ───────────────────────────────────
def make_generator(D):
    inp = keras.Input(shape=(D,))
    x   = keras.layers.Dense(256, activation="relu")(inp)
    x   = keras.layers.Dense(256, activation="relu")(x)
    out = keras.layers.Dense(D,   activation="tanh")(x)
    return keras.Model(inp, out, name="advgan_gen")

def make_critic(D):
    inp = keras.Input(shape=(D,))
    x   = keras.layers.Dense(256)(inp)
    x   = keras.layers.LeakyReLU(0.2)(x)
    x   = keras.layers.Dense(256)(x)
    x   = keras.layers.LeakyReLU(0.2)(x)
    out = keras.layers.Dense(1)(x)
    return keras.Model(inp, out, name="advgan_critic")

loss_ce  = keras.losses.SparseCategoricalCrossentropy()
loss_bce = keras.losses.BinaryCrossentropy()

@tf.function
def train_step(G, C, ids_model, X_batch, y_batch, G_opt, C_opt):
    # Critic step
    with tf.GradientTape() as ct:
        delta   = G(X_batch, training=False) * eps_max_nsl
        X_adv   = tf.clip_by_value(X_batch + delta, 0.0, 1.0)
        real_s  = C(X_batch,   training=True)
        fake_s  = C(X_adv,     training=True)
        loss_c  = tf.reduce_mean(fake_s) - tf.reduce_mean(real_s)
    C_opt.apply_gradients(zip(ct.gradient(loss_c, C.trainable_variables),
                               C.trainable_variables))

    # Generator step
    with tf.GradientTape() as gt:
        delta   = G(X_batch, training=True) * eps_max_nsl
        X_adv   = tf.clip_by_value(X_batch + delta, 0.0, 1.0)
        fake_s  = C(X_adv, training=False)
        preds   = ids_model(X_adv, training=False)
        # Target: wrong class (shift labels by 1)
        y_adv   = tf.cast((y_batch + 1) % 5, tf.int32)
        loss_g  = (-tf.reduce_mean(fake_s)
                   + lambda_cls * loss_ce(y_adv, preds)
                   + lambda_l1  * tf.reduce_mean(tf.abs(delta)))
    G_opt.apply_gradients(zip(gt.gradient(loss_g, G.trainable_variables),
                               G.trainable_variables))
    return loss_g

def train_advgan_against(ids_model, X_train, label, save_path):
    """Train AdvGAN against a given IDS model and save generator."""
    G = make_generator(D)
    C = make_critic(D)
    G_opt = keras.optimizers.Adam(LR, beta_1=0.0, beta_2=0.9)
    C_opt = keras.optimizers.Adam(LR, beta_1=0.0, beta_2=0.9, clipnorm=1.0)

    dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train_t)) \
                .shuffle(50000).batch(512).repeat()

    print(f"\nTraining AdvGAN against {label}...")
    for step, (X_b, y_b) in enumerate(dataset):
        if step >= N_STEPS:
            break
        loss = train_step(G, C, ids_model, X_b, y_b, G_opt, C_opt)
        if step % 200 == 0:
            print(f"  Step {step}/{N_STEPS} | gen_loss: {loss:.4f}")

    # Save generator weights as numpy (avoids Lambda issues)
    weights = [l.get_weights() for l in G.layers
               if isinstance(l, keras.layers.Dense)]
    np.save(save_path, np.array(weights, dtype=object), allow_pickle=True)
    print(f"  ✅ Generator saved: {save_path}")
    return G

def evaluate_under_advgan(G, ids_model, X_test, y_test, label):
    """Generate adversarial samples and evaluate IDS model."""
    tanh_out = G.predict(X_test, batch_size=4096, verbose=0)
    delta    = tanh_out * eps_max_nsl
    X_adv    = np.clip(X_test + delta, 0.0, 1.0)
    preds    = ids_model.predict(X_adv, batch_size=4096, verbose=0)
    y_pred   = np.argmax(preds, axis=1)
    y_true   = y_test.numpy() if hasattr(y_test, 'numpy') else np.array(y_test)
    acc      = float(accuracy_score(y_true, y_pred))
    f1       = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    print(f"  {label:<25} Acc={acc*100:.2f}%  MacroF1={f1*100:.2f}%")
    return {"accuracy": acc, "macro_f1": f1}

# ── Load test data ────────────────────────────────────────────────
X_test_np  = X_test_t.numpy()
X_train_np = X_train_t.numpy()

# ── Run for each baseline ─────────────────────────────────────────
BASELINES = {
    "Undefended": f"{MODELS}/nsl_student_baseline.keras",
    "FGSM-AT":    f"{MODELS}/exp_fgsm_at.keras",
    "PGD-AT":     f"{MODELS}/exp_pgd_at.keras",
}

all_results = {}

for label, model_path in BASELINES.items():
    result_path = f"{RESULTS}/advgan_baseline_{label.lower().replace('-','_')}.json"

    # Skip if already done (crash recovery)
    if os.path.exists(result_path):
        print(f"⏭ Skipping {label} — already done")
        with open(result_path) as f:
            all_results[label] = json.load(f)
        continue

    if not os.path.exists(model_path):
        print(f"⚠ Model not found: {model_path}")
        continue

    # Load baseline model
    ids_model = keras.models.load_model(model_path, safe_mode=False)

    # Train AdvGAN against this model
    gen_save = f"{MODELS}/advgan_baselines/gen_{label.lower().replace('-','_')}.npy"
    G = train_advgan_against(ids_model, X_train_np, label, gen_save)

    # Evaluate
    print(f"  Evaluating {label} under its own AdvGAN...")
    result = evaluate_under_advgan(G, ids_model, X_test_np, y_test_t, label)
    all_results[label] = result

    # Save immediately after each model
    with open(result_path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"  ✅ Saved: {result_path}")

    del ids_model, G
    tf.keras.backend.clear_session()

# ── Final summary ─────────────────────────────────────────────────
print("\n" + "="*55)
print("TABLE 3 UPDATE — AdvGAN results for all models")
print("="*55)
print(f"{'Model':<25} {'AdvGAN Acc':>12} {'AdvGAN F1':>12}")
print("-"*52)
for label, r in all_results.items():
    if r:
        print(f"  {label:<23} {r['accuracy']*100:>11.2f}% {r['macro_f1']*100:>11.2f}%")
print(f"  {'A3IDPS (full)':<23} {'99.33%':>12} {'98.00%':>12}")
print("\nPaste this table here — I will update Table 3 in the paper.")


Training AdvGAN against Undefended...
  Step 0/1200 | gen_loss: 50.8200
  Step 200/1200 | gen_loss: 41.3464
  Step 400/1200 | gen_loss: 52.0695
  Step 600/1200 | gen_loss: 71.1275
  Step 800/1200 | gen_loss: 101.6553
  Step 1000/1200 | gen_loss: 145.2329
  ✅ Generator saved: /content/drive/MyDrive/A3IDPS_COLAB/models/advgan_baselines/gen_undefended.npy
  Evaluating Undefended under its own AdvGAN...
  Undefended                Acc=85.06%  MacroF1=66.86%
  ✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/advgan_baseline_undefended.json

Training AdvGAN against FGSM-AT...


ValueError: in user code:

    File "/tmp/ipykernel_8329/3694707939.py", line 52, in train_step  *
        C_opt.apply_gradients(zip(ct.gradient(loss_c, C.trainable_variables),
    File "/usr/local/lib/python3.12/dist-packages/keras/src/optimizers/base_optimizer.py", line 462, in apply_gradients  **
        self.apply(grads, trainable_variables)
    File "/usr/local/lib/python3.12/dist-packages/keras/src/optimizers/base_optimizer.py", line 501, in apply
        self.build(trainable_variables)
    File "/usr/local/lib/python3.12/dist-packages/keras/src/optimizers/adam.py", line 93, in build
        self._momentums, self._velocities = self.add_optimizer_variables(
    File "/usr/local/lib/python3.12/dist-packages/keras/src/optimizers/base_optimizer.py", line 389, in add_optimizer_variables
        self.add_variable_from_reference(
    File "/usr/local/lib/python3.12/dist-packages/keras/src/backend/tensorflow/optimizer.py", line 35, in add_variable_from_reference
        return super().ad

In [4]:
# ================================================================
# FIXED AdvGAN cell — run this, Undefended will be skipped
# ================================================================

import os, json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import f1_score, accuracy_score

BASE    = "/content/drive/MyDrive/A3IDPS_COLAB"
MODELS  = f"{BASE}/models"
RESULTS = f"{BASE}/results"
os.makedirs(f"{MODELS}/advgan_baselines", exist_ok=True)

D           = X_train_t.shape[1]
eps_max_nsl = 0.10
lambda_cls  = 20.0
lambda_l1   = 0.5
N_STEPS     = 1200
LR          = 1e-4

loss_ce = keras.losses.SparseCategoricalCrossentropy()

def make_generator(D):
    inp = keras.Input(shape=(D,))
    x   = keras.layers.Dense(256, activation="relu")(inp)
    x   = keras.layers.Dense(256, activation="relu")(x)
    out = keras.layers.Dense(D,   activation="tanh")(x)
    return keras.Model(inp, out)

def make_critic(D):
    inp = keras.Input(shape=(D,))
    x   = keras.layers.Dense(256)(inp)
    x   = keras.layers.LeakyReLU(0.2)(x)
    x   = keras.layers.Dense(256)(x)
    x   = keras.layers.LeakyReLU(0.2)(x)
    out = keras.layers.Dense(1)(x)
    return keras.Model(inp, out)

def train_advgan_against(ids_model, X_train_np, y_train_np, label, save_path):
    """Fresh G, C and optimizers every call — no @tf.function conflict."""
    G     = make_generator(D)
    C     = make_critic(D)
    G_opt = keras.optimizers.Adam(LR, beta_1=0.0, beta_2=0.9)
    C_opt = keras.optimizers.Adam(LR, beta_1=0.0, beta_2=0.9, clipnorm=1.0)

    dataset = (tf.data.Dataset
               .from_tensor_slices((X_train_np, y_train_np))
               .shuffle(50000).batch(512).repeat())

    print(f"\nTraining AdvGAN against {label}...")
    for step, (X_b, y_b) in enumerate(dataset):
        if step >= N_STEPS:
            break

        # Critic step — plain Python, no @tf.function
        with tf.GradientTape() as ct:
            delta  = G(X_b, training=False) * eps_max_nsl
            X_adv  = tf.clip_by_value(X_b + delta, 0.0, 1.0)
            loss_c = (tf.reduce_mean(C(X_adv, training=True))
                      - tf.reduce_mean(C(X_b,  training=True)))
        C_opt.apply_gradients(
            zip(ct.gradient(loss_c, C.trainable_variables),
                C.trainable_variables))

        # Generator step
        with tf.GradientTape() as gt:
            delta  = G(X_b, training=True) * eps_max_nsl
            X_adv  = tf.clip_by_value(X_b + delta, 0.0, 1.0)
            y_adv  = tf.cast((y_b + 1) % 5, tf.int32)
            preds  = ids_model(X_adv, training=False)
            loss_g = (-tf.reduce_mean(C(X_adv, training=False))
                      + lambda_cls * loss_ce(y_adv, preds)
                      + lambda_l1  * tf.reduce_mean(tf.abs(delta)))
        G_opt.apply_gradients(
            zip(gt.gradient(loss_g, G.trainable_variables),
                G.trainable_variables))

        if step % 200 == 0:
            print(f"  Step {step}/{N_STEPS} | gen_loss: {loss_g:.4f}")

    # Save weights as numpy — avoids Lambda serialization issue
    weights = [l.get_weights() for l in G.layers
               if isinstance(l, keras.layers.Dense)]
    np.save(save_path, np.array(weights, dtype=object), allow_pickle=True)
    print(f"  ✅ Generator saved: {save_path}")
    return G

def evaluate_under_advgan(G, ids_model, X_test_np, y_test_np, label):
    tanh_out = G.predict(X_test_np, batch_size=4096, verbose=0)
    X_adv    = np.clip(X_test_np + tanh_out * eps_max_nsl, 0.0, 1.0)
    preds    = ids_model.predict(X_adv, batch_size=4096, verbose=0)
    y_pred   = np.argmax(preds, axis=1)
    acc  = float(accuracy_score(y_test_np, y_pred))
    f1   = float(f1_score(y_test_np, y_pred, average="macro", zero_division=0))
    print(f"  {label:<25} Acc={acc*100:.2f}%  MacroF1={f1*100:.2f}%")
    return {"accuracy": acc, "macro_f1": f1}

# Prepare numpy arrays
X_train_np = X_train_t.numpy()
X_test_np  = X_test_t.numpy()
y_train_np = y_train_t.numpy()
y_test_np  = y_test_t.numpy()

BASELINES = {
    "Undefended": f"{MODELS}/nsl_student_baseline.keras",
    "FGSM-AT":    f"{MODELS}/exp_fgsm_at.keras",
    "PGD-AT":     f"{MODELS}/exp_pgd_at.keras",
}

all_results = {}

for label, model_path in BASELINES.items():
    result_path = (f"{RESULTS}/advgan_baseline_"
                   f"{label.lower().replace('-','_')}.json")

    # Skip if already completed in a previous run
    if os.path.exists(result_path):
        print(f"⏭  Skipping {label} — already saved")
        with open(result_path) as f:
            all_results[label] = json.load(f)
        continue

    if not os.path.exists(model_path):
        print(f"❌ Model not found: {model_path}")
        continue

    ids_model = keras.models.load_model(model_path, safe_mode=False)

    gen_save = (f"{MODELS}/advgan_baselines/"
                f"gen_{label.lower().replace('-','_')}.npy")
    G = train_advgan_against(ids_model, X_train_np, y_train_np,
                              label, gen_save)

    result = evaluate_under_advgan(G, ids_model, X_test_np, y_test_np, label)
    all_results[label] = result

    with open(result_path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"  ✅ Saved: {result_path}")

    del ids_model, G
    tf.keras.backend.clear_session()

# ── Final summary ─────────────────────────────────────────────────
print("\n" + "="*55)
print("TABLE 3 — AdvGAN results (all models)")
print("="*55)
print(f"{'Model':<25} {'AdvGAN Acc':>12} {'AdvGAN F1':>12}")
print("-"*52)
for label, r in all_results.items():
    if r:
        print(f"  {label:<23} "
              f"{r['accuracy']*100:>11.2f}%  "
              f"{r['macro_f1']*100:>11.2f}%")
print(f"  {'A3IDPS (full)':<23} {'99.33%':>12} {'98.00%':>12}")
print("\nPaste output here when done.")

⏭  Skipping Undefended — already saved

Training AdvGAN against FGSM-AT...
  Step 0/1200 | gen_loss: 85.3580
  Step 200/1200 | gen_loss: 75.7644
  Step 400/1200 | gen_loss: 82.4232
  Step 600/1200 | gen_loss: 100.7763
  Step 800/1200 | gen_loss: 128.1790
  Step 1000/1200 | gen_loss: 166.0226
  ✅ Generator saved: /content/drive/MyDrive/A3IDPS_COLAB/models/advgan_baselines/gen_fgsm_at.npy


  FGSM-AT                   Acc=82.86%  MacroF1=59.20%
  ✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/advgan_baseline_fgsm_at.json

Training AdvGAN against PGD-AT...
  Step 0/1200 | gen_loss: 83.1750
  Step 200/1200 | gen_loss: 75.4166
  Step 400/1200 | gen_loss: 78.7968
  Step 600/1200 | gen_loss: 93.3457
  Step 800/1200 | gen_loss: 114.8445
  Step 1000/1200 | gen_loss: 143.7634
  ✅ Generator saved: /content/drive/MyDrive/A3IDPS_COLAB/models/advgan_baselines/gen_pgd_at.npy
  PGD-AT                    Acc=81.42%  MacroF1=56.73%
  ✅ Saved: /content/drive/MyDrive/A3IDPS_COLAB/results/advgan_baseline_pgd_at.json

TABLE 3 — AdvGAN results (all models)
Model                       AdvGAN Acc    AdvGAN F1
----------------------------------------------------
  Undefended                    85.06%        66.86%
  FGSM-AT                       82.86%        59.20%
  PGD-AT                        81.42%        56.73%
  A3IDPS (full)                 99.33%       98.00%

Paste output here w

In [5]:
"""
A3IDPS Figure Generation Script
Run this in Colab after mounting Drive.
Generates Figures 1, 2, 3 as PDF and PNG.
Figure 4 (SHAP) is already saved on your Drive.
"""

from google.colab import drive
drive.mount('/content/drive')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patches as FancyArrowPatch
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np
import os

BASE   = "/content/drive/MyDrive/A3IDPS_COLAB"
FIGS   = f"{BASE}/figures"
os.makedirs(FIGS, exist_ok=True)

# Publication style settings
plt.rcParams.update({
    'font.family':      'serif',
    'font.size':        11,
    'axes.titlesize':   12,
    'axes.labelsize':   11,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  10,
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
    'savefig.pad_inches': 0.05,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

BLUE   = '#2C5F8A'
TEAL   = '#1D8A6E'
ORANGE = '#E07B30'
GRAY   = '#6B6B6B'
LGRAY  = '#E8E8E8'

# ================================================================
# FIGURE 2 — Training Progression (macro F1 across stages)
# ================================================================

stages = [
    'Baseline\nDNN',
    'Teacher\nDistillation',
    'GAN-A\n(no feedback)',
    'Feedback\nRound 1',
    'Feedback\nRound 2\n(A3IDPS)'
]
f1_values = [73.09, 83.67, 71.84, 97.93, 98.32]
colors     = [GRAY, BLUE, ORANGE, TEAL, TEAL]
alphas     = [0.85, 0.85, 0.85, 0.85, 1.0]

fig, ax = plt.subplots(figsize=(8, 4.5))

bars = ax.bar(stages, f1_values, color=colors, alpha=0.88,
              edgecolor='white', linewidth=0.8, width=0.6)

# Value labels on bars
for bar, val in zip(bars, f1_values):
    ypos = bar.get_height() + 0.8
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f'{val:.1f}%', ha='center', va='bottom',
            fontsize=9.5, fontweight='bold',
            color='#333333')

# Annotation arrow for drop at GAN-A
ax.annotate('', xy=(2, 71.84), xytext=(1, 83.67),
            arrowprops=dict(arrowstyle='->', color=ORANGE,
                            lw=1.5, connectionstyle='arc3,rad=0.2'))
ax.text(1.6, 76, 'boundary\ndisruption', ha='center', fontsize=8,
        color=ORANGE, style='italic')

# Dashed reference line at baseline
ax.axhline(73.09, color=GRAY, linestyle='--', linewidth=0.8, alpha=0.5)
ax.text(4.35, 74.2, 'baseline\n(73.1%)', fontsize=8, color=GRAY, va='bottom')

ax.set_ylabel('Macro F\u2081-Score (%)', labelpad=8)
ax.set_ylim(60, 105)
ax.set_yticks([60, 70, 80, 90, 100])
ax.set_yticklabels(['60%', '70%', '80%', '90%', '100%'])
ax.set_title('A3IDPS Training Progression on NSL-KDD',
             fontweight='bold', pad=12)
ax.set_xlabel('Training Stage', labelpad=8)
ax.grid(axis='y', linestyle='--', alpha=0.35, linewidth=0.7)

# Legend
legend_elements = [
    mpatches.Patch(color=GRAY,   label='Baseline'),
    mpatches.Patch(color=BLUE,   label='With distillation'),
    mpatches.Patch(color=ORANGE, label='GAN exposure only'),
    mpatches.Patch(color=TEAL,   label='Feedback rounds'),
]
ax.legend(handles=legend_elements, loc='upper left',
          framealpha=0.9, edgecolor=LGRAY, fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIGS}/figure2_training_progression.pdf')
plt.savefig(f'{FIGS}/figure2_training_progression.png')
plt.close()
print('✅ Figure 2 saved.')

# ================================================================
# FIGURE 3 — FGSM Robustness Comparison
# ================================================================

models    = ['Undefended\nbaseline', 'FGSM-AT', 'PGD-AT',
             'A3IDPS\nRound 1', 'A3IDPS\nRound 2']
eps_001   = [87.57, 82.30, 82.85, 99.17, 99.38]
eps_005   = [88.77, 81.09, 80.69, 98.42, 98.66]
eps_010   = [70.27, 75.62, 74.21, 95.87, 96.64]

x      = np.arange(len(models))
width  = 0.26

fig, ax = plt.subplots(figsize=(9, 5))

b1 = ax.bar(x - width,     eps_001, width, label='ε = 0.01',
            color=BLUE,   alpha=0.88, edgecolor='white', linewidth=0.6)
b2 = ax.bar(x,             eps_005, width, label='ε = 0.05',
            color=TEAL,   alpha=0.88, edgecolor='white', linewidth=0.6)
b3 = ax.bar(x + width,     eps_010, width, label='ε = 0.10',
            color=ORANGE, alpha=0.88, edgecolor='white', linewidth=0.6)

# Value labels — only on ε=0.10 bars to avoid clutter
for bar in b3:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.4,
            f'{h:.1f}', ha='center', va='bottom',
            fontsize=7.5, color='#444444')

# Vertical separator between baselines and A3IDPS
ax.axvline(2.5, color='#CCCCCC', linestyle='--', linewidth=1.0)
ax.text(2.55, 102, 'A3IDPS', fontsize=8.5, color=TEAL,
        fontweight='bold', va='top')
ax.text(0.0, 102, 'Baselines', fontsize=8.5, color=GRAY,
        fontweight='bold', va='top')

ax.set_ylabel('Adversarial Accuracy (%)', labelpad=8)
ax.set_xlabel('Model', labelpad=8)
ax.set_title('FGSM Adversarial Robustness on NSL-KDD',
             fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=9.5)
ax.set_ylim(60, 107)
ax.set_yticks([60, 70, 80, 90, 100])
ax.set_yticklabels(['60%', '70%', '80%', '90%', '100%'])
ax.legend(loc='lower right', framealpha=0.9,
          edgecolor=LGRAY, fontsize=9.5)
ax.grid(axis='y', linestyle='--', alpha=0.35, linewidth=0.7)

plt.tight_layout()
plt.savefig(f'{FIGS}/figure3_fgsm_robustness.pdf')
plt.savefig(f'{FIGS}/figure3_fgsm_robustness.png')
plt.close()
print('✅ Figure 3 saved.')

# ================================================================
# FIGURE 1 — A3IDPS Architecture Diagram
# ================================================================

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 5.5)
ax.axis('off')

def box(ax, x, y, w, h, label, sublabel=None,
        fc='#2C5F8A', tc='white', fs=10):
    rect = FancyBboxPatch((x, y), w, h,
                          boxstyle="round,pad=0.08",
                          facecolor=fc, edgecolor='#AAAAAA',
                          linewidth=0.8)
    ax.add_patch(rect)
    if sublabel:
        ax.text(x + w/2, y + h*0.62, label,
                ha='center', va='center',
                fontsize=fs, fontweight='bold', color=tc)
        ax.text(x + w/2, y + h*0.28, sublabel,
                ha='center', va='center',
                fontsize=fs - 1.5, color=tc, alpha=0.88,
                style='italic')
    else:
        ax.text(x + w/2, y + h/2, label,
                ha='center', va='center',
                fontsize=fs, fontweight='bold', color=tc)

def arrow(ax, x1, y1, x2, y2, color='#555555'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color,
                                lw=1.4, mutation_scale=14))

# ── Phase labels ──────────────────────────────────────────────────
for x, label, color in [
    (0.2,  'Phase 1\nBaseline Training', GRAY),
    (2.8,  'Phase 2\nGAN Training',      ORANGE),
    (5.5,  'Phase 3\nFeedback Loop',     TEAL),
    (9.5,  'Phase 4\nExplainability',    BLUE),
]:
    ax.text(x + 1.0, 5.2, label, ha='center', va='top',
            fontsize=8.5, color=color, fontweight='bold')

# ── Data box ──────────────────────────────────────────────────────
box(ax, 0.2, 2.2, 1.8, 1.0,
    'NSL-KDD', 'Network traffic data',
    fc='#4A4A4A', tc='white', fs=9)

arrow(ax, 2.0, 2.7, 2.4, 2.7)

# ── Teacher model ─────────────────────────────────────────────────
box(ax, 0.2, 0.6, 1.8, 1.0,
    'Teacher DNN', '[512→256→128→K]',
    fc='#888888', tc='white', fs=9)
ax.annotate('', xy=(1.2, 2.2), xytext=(1.2, 1.6),
            arrowprops=dict(arrowstyle='->', color='#888888',
                            lw=1.2, linestyle='dashed'))
ax.text(1.25, 1.9, 'distil', fontsize=7.5,
        color='#888888', style='italic')

# ── Baseline DNN ──────────────────────────────────────────────────
box(ax, 2.4, 2.0, 2.0, 1.4,
    'Baseline DNN', '[123→256→128→64→K]',
    fc=BLUE, tc='white', fs=9)
arrow(ax, 4.4, 2.7, 4.8, 2.7)

# ── AdvGAN ────────────────────────────────────────────────────────
box(ax, 4.8, 2.0, 2.0, 1.4,
    'AdvGAN Gen.', 'λ_cls=20, λ_l1=0.5\n1200 steps',
    fc=ORANGE, tc='white', fs=9)

# Arrow from AdvGAN down to feedback
ax.annotate('', xy=(5.8, 2.0), xytext=(5.8, 1.3),
            arrowprops=dict(arrowstyle='<->', color=ORANGE,
                            lw=1.3))
ax.text(6.05, 1.65, 'adversarial\nsamples', fontsize=7.5,
        color=ORANGE, style='italic')

# ── Feedback rounds ───────────────────────────────────────────────
arrow(ax, 6.8, 2.7, 7.2, 2.7)

box(ax, 7.2, 2.2, 1.5, 1.0,
    'Feedback\nRound 1', fc=TEAL, tc='white', fs=9)
arrow(ax, 8.7, 2.7, 9.0, 2.7)
box(ax, 9.0, 2.2, 1.5, 1.0,
    'Feedback\nRound 2', fc=TEAL, tc='white', fs=9)

# Feedback loop arrow
ax.annotate('', xy=(7.95, 2.2), xytext=(7.95, 1.0),
            arrowprops=dict(arrowstyle='->', color=TEAL,
                            lw=1.2, linestyle='dashed'))
ax.annotate('', xy=(5.8, 1.0), xytext=(7.95, 1.0),
            arrowprops=dict(arrowstyle='->', color=TEAL,
                            lw=1.2, linestyle='dashed'))
ax.text(6.87, 0.75, 'GAN re-trains\nagainst hardened model',
        ha='center', fontsize=7.5, color=TEAL, style='italic')

arrow(ax, 10.5, 2.7, 10.8, 2.7)

# ── SHAP ─────────────────────────────────────────────────────────
box(ax, 10.8, 2.2, 1.0, 1.0,
    'SHAP\nXAI', fc=BLUE, tc='white', fs=9)

# Final output
ax.text(11.3, 1.9, '↓', ha='center', fontsize=14, color='#555555')
box(ax, 10.7, 0.8, 1.2, 0.8,
    'Feature\nImportance', fc='#E8F4F1', tc='#1D8A6E', fs=8)

# ── Title ─────────────────────────────────────────────────────────
ax.text(6.0, 5.45, 'A3IDPS Pipeline',
        ha='center', va='top', fontsize=13,
        fontweight='bold', color='#222222')

plt.tight_layout()
plt.savefig(f'{FIGS}/figure1_pipeline.pdf')
plt.savefig(f'{FIGS}/figure1_pipeline.png')
plt.close()
print('✅ Figure 1 saved.')

# ================================================================
# FIGURE 4 — Copy SHAP plot path reminder
# ================================================================
shap_path = f"{BASE}/results/shap_cicids/cicids_shap_summary_bar.png"
if os.path.exists(shap_path):
    import shutil
    shutil.copy(shap_path, f"{FIGS}/figure4_shap_bar.png")
    print(f'✅ Figure 4 (SHAP) copied to figures folder.')
else:
    print(f'⚠ Figure 4 not found at: {shap_path}')
    print('  Check: results/shap_cicids/ on your Drive')

print(f'\n✅ All figures saved to: {FIGS}')
print('Files:')
for f in sorted(os.listdir(FIGS)):
    print(f'  {f}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Figure 2 saved.
✅ Figure 3 saved.
✅ Figure 1 saved.
✅ Figure 4 (SHAP) copied to figures folder.

✅ All figures saved to: /content/drive/MyDrive/A3IDPS_COLAB/figures
Files:
  figure1_pipeline.pdf
  figure1_pipeline.png
  figure2_training_progression.pdf
  figure2_training_progression.png
  figure3_fgsm_robustness.pdf
  figure3_fgsm_robustness.png
  figure4_shap_bar.png
